####Install / Imports

In [0]:
!pip install -q \
    "pydantic<2" spacy==3.7.4 \
    sentence-transformers==3.0.1 \
    numpy==1.26.4 \
    striprtf==0.0.22 \
    tqdm==4.66.5 \
    html2text \
    ftfy

In [0]:
!python -m spacy download pt_core_news_lg || python -m spacy download pt_core_news_sm

In [0]:
!pip install lxml

In [0]:
dbutils.library.restartPython()

In [0]:
# ==== Stdlib ====
import re, unicodedata, html as _html, json
from dataclasses import dataclass, asdict
from functools import lru_cache
from typing import List, Dict, Tuple, Any
from difflib import SequenceMatcher

# ==== Numérico / Dados ====
import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.5f' % x)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2000)

# ==== NLP / Embeddings ====
import spacy
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# ==== RTF -> Texto (OK no PyPI) ====
from striprtf.striprtf import rtf_to_text

# ==== HTML -> Texto ====
from lxml.html import fromstring
import html2text

# ==== Unicode/Mojibake ====
import ftfy

# ==== Progresso ====
from tqdm import tqdm


In [0]:
# === Configuração do Spacy ===
nlp = spacy.load("pt_core_news_lg", disable=["ner"])
nlp.add_pipe("sentencizer", config={"punct_chars": ["\n"]}, before="parser")
nlp.max_length = 1500000

####Variaveis de Ambiente

In [0]:
dbutils.widgets.text("id_projeto", "doencas_biliares", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("environment:", environment)

In [0]:
environment_tbl = "" if environment in ["hml", "prd"] else f"{environment}_"
print("environment_tbl:", environment_tbl)

In [0]:
import datetime

dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.datetime.now().strftime("%Y-%m-%d")

In [0]:
INPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_doencas_biliares"
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_doencas_biliares_saida"
SCHEMA = "ia"

In [0]:
print('Environment:    ', environment_tbl)
print('Tabela entrada: ', INPUT_TABLENAME)
print('Tabela destino: ', OUTPUT_TABLENAME)
print('schema:         ', SCHEMA)
print('Data Referencia:', data_execucao_modelo)

####Modelo (Target Organ = doencas_biliares)

In [0]:
###### o bloco acima foi substituido pelo bloco abaixo

# --- Utilitários ---
def norm(s: str) -> str:
    s = s.lower().strip()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"\s+", " ", s)
    return s

def unique(seq):
    seen=set(); out=[]
    for x in seq:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

#------- ALTERAÇÕES PARA HTML -------
_HTML_BRK_TAGS = re.compile(r'</?(p|div|br|li|tr|h[1-6])\b[^>]*>', re.IGNORECASE)
_HTML_TAGS     = re.compile(r'<[^>]+>')
_HTML_SCRIPT   = re.compile(r'<(script|style)\b[^>]>.?</\1\s*>', re.IGNORECASE|re.DOTALL)
_HTML_DATAIMG  = re.compile(r'data:image/[^;]+;base64,[A-Za-z0-9+/=\s]+', re.IGNORECASE)
_HTML_JUNK_ATTR= re.compile(r'\sdata-[\w\-]+="[^"]*"')
_HTML_JUNK_CLS = re.compile(r'\sclass="[^"]*"')
_HTML_WIDGET   = re.compile(r'\s(?:aria-[\w\-]+|role|tabindex|contenteditable|spellcheck|draggable|style|width|height)="[^"]*"', re.IGNORECASE)

def _looks_like_html(s: str) -> bool:
    head = s[:500]
    return bool(re.search(r'</?(html|head|body|div|p|span|br|img|script|style|table)\b', head, re.IGNORECASE))

def html_to_plain(s: str) -> str:
    if not s:
        return ""

    # normaliza EOL
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # Parse/limpeza estrutural com lxml (remove script/style/noscript)
    try:
        doc = fromstring(s)
        for bad in doc.xpath('//script|//style|//noscript'):
            bad.drop_tree()

        body = doc.find('body')
        node = body if body is not None else doc
        cleaned_html = lxml.html.tostring(node, encoding="unicode", method="html")
    except Exception:
        cleaned_html = s  # fallback: deixa o html2text tentar

    # HTML -> texto preservando parágrafos/quebras
    h = html2text.HTML2Text()
    h.body_width = 0                 # não “wrapa”
    h.single_line_break = True       # <br> vira \n
    h.ignore_links = True
    h.ignore_images = True
    h.ignore_emphasis = True
    h.unicode_snob = True

    try:
        s = h.handle(cleaned_html)
    except Exception:
        # fallback extremo: extrai texto puro do lxml
        try:
            s = fromstring(cleaned_html).text_content()
        except Exception:
            s = cleaned_html

    # decodifica entidades (&nbsp;, &ccedil;, etc.)
    s = _html.unescape(s)

    # limpa invisíveis e normaliza unicode
    s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
    s = unicodedata.normalize("NFKC", s)

    # colagens comuns do editor
    s = re.sub(r'[ \t]{2,}', ' ', s)
    s = re.sub(r' ?/ ?', ' / ', s)
    s = re.sub(r'\n[ \t]+', '\n', s)

    # linhas em branco excessivas
    s = re.sub(r'\n{3,}', '\n\n', s)

    # strip final por linha e geral
    lines = [ln.rstrip() for ln in s.splitlines()]
    s = "\n".join(lines).strip()

    # correções pontuais vistas no corpus Tasy/CKE
    s = s.replace("Hospita\u200bl", "Hospital")
    s = re.sub(r'\bColonosocopia\b', 'Colonoscopia', s, flags=re.IGNORECASE)
    s = re.sub(r'\bredundancia\b', 'redundância', s, flags=re.IGNORECASE)

    return s



# --- NOVO: limpeza leve de ruído pós-extração ---
def _cleanup_text_noise(text: str) -> str:
    if not text:
        return ""

    # invisíveis comuns + normalização
    text = (text.replace("\u200b", "")
                .replace("\u200c", "")
                .replace("\xa0", " "))
    text = unicodedata.normalize("NFKC", text)

    # 1) Normalizar espaçamento em barras quando há caractere dos dois lados
    text = re.sub(r'(?<=\w)\s*/\s*(?=\w)', '/', text, flags=re.UNICODE)

    # 2) Garantir espaço entre número e letra/unidade (não mexe nas barras)
    text = re.sub(r'(?<=\d)(?=[A-Za-zµμ°])', ' ', text)

    # 3) Compactar múltiplos espaços e aparar espaços antes/depois de \n
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\s*\n\s*', '\n', text)

    # 4) Limpar linhas em branco repetidas
    text = re.sub(r'\n{3,}', '\n\n', text)

    # 5) Trim por linha e global
    lines = [ln.rstrip() for ln in text.splitlines()]
    text = "\n".join(lines).strip()
    return text


def _rtf_to_text_fallback(rtf: str, debug: bool = False) -> str:
    if not rtf:
        return ""

    import re, unicodedata

    # ---- helpers locais ----
    def _undo_py_escapes(s: str) -> str:
        # reverte escapes que o Python injeta quando a string não é raw
        return (s
            .replace("\x0c", r"\f")   # form feed -> \f
            .replace("\t",  r"\tab")  # tab -> \tab (token RTF real)
        )

    def _strip_rtf_metadata_top(s: str) -> str:
        # Remove SOMENTE blocos padrão logo no topo (primeiros ~8 KB)
        head = s[:8192]
        tail = s[8192:]

        # cada bloco: remove no máximo 1 ocorrência por padrão
        patterns = [
            r"{\\\?\\fonttbl\b[^{}]}",
            r"{\\\?\\colortbl\b[^{}]}",
            r"{\\\?\\stylesheet\b[^{}]}",
            r"{\\\?\\listtable\b[^{}]}",
            r"{\\\?\\listoverridetable\b[^{}]}",
            r"{\\\?\\generator\b[^{}]}",
            r"{\\\?\\info\b[^{}]}",
        ]
        for pat in patterns:
            head = re.sub(pat, " ", head, flags=re.IGNORECASE)

        return head + tail

    def _drop_rtf_meta_lines(text: str) -> str:
        out = []
        only_words_semicolon_rx = re.compile(r'^[A-Za-z0-9 /+\-\\u00C0-\u017F]+;\s$')
        font_tokens = r"(unicode|opensymbol|wingdings|monospaced|serif|sans|arial|calibri|times|courier)"
        # aceita " * " OU "\* " antes do nome da fonte
        font_line_rx = re.compile(
            rf'^\s*[A-Za-z0-9 \-\u00C0-\u017F]+(?:\s\\?\\s)?(?:{font_tokens})(?:\s*[;:]?)\s*$', re.IGNORECASE)
        tiny_meta_rx = re.compile(r'^\s*(default|\jword2|\ generator|\* info)\s*[;:]?\s*$', re.IGNORECASE)

        for ln in text.splitlines():
            lns = ln.strip()
            if not lns:
                out.append(ln); continue
            if only_words_semicolon_rx.match(lns):
                continue
            if font_line_rx.match(lns):
                continue
            if tiny_meta_rx.match(lns):
                continue
            # fallback bem específico: linha curta com qualquer token de fonte + ';'
            if ';' in lns and re.search(font_tokens, lns, re.IGNORECASE) and len(lns) <= 80:
                continue
            out.append(ln)
        return "\n".join(out).strip()

    # -------------------------

    s = _undo_py_escapes(rtf)
    # normaliza EOL cedo
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # pega só se realmente parece RTF
    if not s.lstrip().startswith("{\\rtf"):
        # não é RTF: devolve como veio
        return s.strip()

    # tira metadados do topo de forma conservadora
    s = _strip_rtf_metadata_top(s)

    if debug:
        print("--- after strip_top ---")
        print(s[:1000])

    # 1) CP1252: \'hh
    def _hex_sub(m):
        try:
            return bytes([int(m.group(1), 16)]).decode('cp1252', errors='ignore')
        except Exception:
            return ""
    s = re.sub(r"\\'([0-9a-fA-F]{2})", _hex_sub, s)

    # 2) Unicode: \uNNNN?
    def _u_sub(m):
        try:
            code = int(m.group(1))
            if code < 0:
                code += 65536
            return chr(code)
        except Exception:
            return ""
    s = re.sub(r"\\u(-?\d+)\??", _u_sub, s)

    # 3) Quebras e tabs
    s = re.sub(r"\\par[d]?\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\line\b", "\n", s, flags=re.IGNORECASE)
    s = re.sub(r"\\tab\b", "\t", s, flags=re.IGNORECASE)

    # 4) Remove destinos conhecidos (restantes, pontuais)
    s = re.sub(r"{\\\?\\(fonttbl|colortbl|stylesheet|info|generator|listtable|listoverridetable)[^}]}",
               " ", s, flags=re.IGNORECASE)

    # 5) Remover comandos de controle (apenas os que começam com barra)
    #    Mais conservador: exige backslash + letras (com opcional número)
    #s = re.sub(r"\\[a-zA-Z]+-?\d*(?:\s|;)?", " ", s)
    s = re.sub(r"\\[a-zA-Z]+-?\d*\s?", " ", s)

    # 6) Escapes literais e chaves
    s = s.replace("\\{", "{").replace("\\}", "}").replace("\\\\", "\\")
    s = re.sub(r"[{}]", " ", s)  # neste ponto já decodificamos conteúdo útil

    # 7) Plano B: 'hh sem barra (alguns dumps)
    s = re.sub(r"(?<!\\)'([0-9a-fA-F]{2})", _hex_sub, s)

    # 8) Limpa tokens RTF residuais comuns (bem restritivo)
    s = re.sub(r"\b(?:rtf\d*|ansi|ansicpg\d+|deftab\d+|paper[wh]\d+|marg[tlrb]\d+|headery\d+|footery\d+|colsx?\d+|snext\d+|fs\d+|cf\d+|pard|plain|qc|ql|itap\d+|viewkind\d+|uc\d+|sa\d+|sl\d+|slmult\d+|lang\d+|kerning\d+|ulnone|b0|i0|f\d+|fs\d+|cf\d+|kerning\d+|deff\d+|colortbl|fonttbl|stylesheet|info|generator|listtable|listoverridetable)\b",
           " ", s, flags=re.IGNORECASE)

    # 9) Normalização
    s = unicodedata.normalize("NFKC", s)
    # remove invisíveis
    s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
    # espaços
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\s*\n\s*", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s).strip()

    # 10) Derruba linhas-meta finais
    s = _drop_rtf_meta_lines(s)

    # Garantia mínima: se sobrar vazio, devolve original “bruto” decodificado por CP1252
    if not s:
        s = bytes(rtf, "latin1", errors="ignore").decode("cp1252", errors="ignore").strip()

    return s


def remove_final_laudo(texto: str) -> str:
    if not texto:
        return ""

    # --- REMOÇÕES GLOBAIS INLINE (apareçam onde aparecerem) ---

    # 1) "Laudo Gerado/Lib[e]rado por Sistema Especialista."
    texto = re.sub(
        r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?',
        ' ',
        texto
    )

    # 2) "Para visualização do conteúdo do laudo, favor/… acessar/acesse … Viewer/opção imagem/imagem disponível"
    texto = re.sub(
        r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.?\b(?:favor\s+)?(?:acesse|acessar)\b.?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?',
        ' ',
        texto
    )

    # 3) "Laudo pode não estar completo na visualização em RTF/Imagem"
    texto = re.sub(
        r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?',
        ' ',
        texto
    )

    # 4) "Referências: - Endoscopic Classification Review Group..." (e variações)
    # Busca por "referencia(s)" e remove o restante do texto.
    texto = re.sub(
        r'(?i)\brefer[eê]ncia[s]?\s*[:\-\.]?.*$',
        ' ',
        texto,
        flags=re.DOTALL
    )

    # --- REMOÇÃO POR SENTENÇA (linhas inteiras de aviso/rodapé) ---
    pats_norm = [
        r"laudo\s+pode\s+nao\s+estar\s+completo.*?\b(rtf|imagem)\b",
        r"para\s+visualiza\w+.*?\b(rtf|imagem|sistema|webris|viewer)\b",
        r"acessar\s+a\s+op[cç][aã]o\s+imagem",
        r"sistema\s+especialista",
        r"sistema\s+da\s+radiologia\s+webris",
        # novas combinações explícitas:
        r"este\s+laudo\s+foi\s+liberado\s+por\s+um?\s+sistema\s+especialista",
        r"favor\s+acessar\s+a\s+op[cç][aã]o\s+imagem\s+dispon[ií]vel",
        r"visualiza\w+\s+do\s+conteudo\s+do\s+laudo",
        r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+rtf",
    ]

    rx = re.compile(
        r"^(?:\s*(?:{})(?:[.!?])?\s*)$".format("|".join(pats_norm)),
        re.IGNORECASE
    )

    # --- ALTERAÇÃO: preservar quebras de linha ---
    lines_in = texto.splitlines()
    lines_out = []

    for line in lines_in:
        if not line.strip():
            lines_out.append(line)
            continue

        # separa sentenças só dentro da linha (não quebra em \n)
        sents = re.split(r'(?<=[.!?])\s+', line.strip())
        kept = []
        for s in sents:
            s_norm = norm(s)
            if not rx.match(s_norm):
                kept.append(s)

        lines_out.append(" ".join(kept).strip())

    out = "\n".join(lines_out)

    # mantém os pós-processamentos originais
    out = re.sub(r'\s{2,}', ' ', out)
    out = re.sub(r'\s+([.!?,;:])', r'\1', out)
    return out.strip()



# --- Pós-processamento PT-BR para laudos (leve e seguro) ---
_UPPER_SPACED_RX = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)

def _despace_allcaps_header(line: str) -> str:
    # "C O N C L U S Ã O" -> "CONCLUSÃO"
    if _UPPER_SPACED_RX.match(line.strip()):
        return line.replace(" ", "")
    return line

def _titlecase_known_headers(line: str) -> str:
    # Padroniza cabeçalhos comuns
    l = line.strip()
    if l in {"CONCLUSAO", "CONCLUSÃO"}:
        return "Conclusão"
    return line

def _fix_common_ocr_pt(text: str) -> str:
    """
    Correções genéricas de OCR:
      - Colapsa CAPS espaçados (linha inteira e inline): "C O N C L U S A O" -> "CONCLUSAO"
      - Remove dígito de prefixo em tokens do tipo 1-2 dígitos + palavra (ex.: "6reto" -> "reto"),
        com salvaguardas para não mexer em unidades/dosagens (mg, ml, mm, cm, kg, bpm, mmHg, l/min etc.)
      - Corrige espaços intra-palavra (ex.: "seda ção" -> "sedação", "p ó s" -> "pós", "aux ílio" -> "auxílio")
      - Higiene de espaços/linhas ao final
    """
    text = unicodedata.normalize("NFKC", text)

    # --- (A) Colapsar CAPS espaçados (genérico) ---
    RX_SPACED_CAPS_LINE = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
    RX_SPACED_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)

    def _collapse_caps_spaces(s: str) -> str:
        if RX_SPACED_CAPS_LINE.match(s.strip()):
            return s.replace(" ", "")
        return RX_SPACED_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), s)

    lines = text.splitlines()
    lines = [_collapse_caps_spaces(ln) for ln in lines]
    text = "\n".join(lines)

    # --- (B) Remover dígito de prefixo OCR (genérico) ---
    UNITS = {
        "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
        "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
        "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
    }
    RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
    RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
    RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')

    def _strip_digit_prefix_token(m: re.Match) -> str:
        token = f"{m.group(1)}{m.group(2)}"
        if RX_DOSAGE.match(token) or RX_FLOW.match(token) or m.group(2).lower() in UNITS:
            return token
        return m.group(2)

    text = RX_OCR_PREFIX.sub(_strip_digit_prefix_token, text)

    # --- (C) Higiene básica de espaços e quebras ---
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\s*\n\s*", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    # --- (D) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
    # Iteramos até estabilizar para capturar padrões encadeados.
    ACCENTS = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
    for _ in range(5):
        before = text

        # D1) "á gua" -> "água", "ç ões" -> "ções", etc.
        text = re.sub(rf'(?i)\b([{ACCENTS}])\s+([A-Za-z]{{1,20}})\b', r'\1\2', text)

        # D2) "aux ílio" -> "auxílio" (letra(s) + espaço + letra acentuada + resto)
        text = re.sub(rf'(?i)\b([A-Za-z]{{1,30}})\s+([{ACCENTS}][A-Za-z]{{1,5}})\b', r'\1\2', text)

        # D3) "p ó s" -> "pós" e similares (trinca com acento no meio)
        text = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACCENTS}])\s+([A-Za-z])\b', r'\1\2\3', text)

        # D4) Sufixos muito comuns
        text = re.sub(r'(?i)çã\s+o\b',  'ção',  text)
        text = re.sub(r'(?i)çõ\s+es\b', 'ções', text)
        text = re.sub(r'(?i)ã\s+o\b',   'ão',   text)
        text = re.sub(r'(?i)õ\s+es\b',  'ões',  text)
        text = re.sub(r'(?i)ó\s+pico(s)?\b', r'ópico\1', text)
        text = re.sub(r'(?i)á\s+sico(s)?\b', r'ásico\1', text)

        # D5) Hífen com espaços indevidos: "p ó s - procedimento" -> "pós-procedimento"
        text = re.sub(rf'(?i)\b([A-Za-z{ACCENTS}]{{1,30}})\s*-\s*([A-Za-z{ACCENTS}]{{1,30}})\b', r'\1-\2', text)

        if text == before:
            break

    # --- (E) Higiene final ---
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\s+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text



# --- Heurísticas genéricas de limpeza pós-extração (valem para RTF/HTML/texto cru) ---

_FONT_WORDS = {
    "times", "times new roman", "arial", "calibri", "cambria", "courier", "courier new",
    "helvetica", "symbol", "wingdings", "opensymbol", "ms mincho", "simsun", "century",
    "cambria math"
}

def _drop_boilerplate_lines(text: str) -> str:
    out = []
    for ln in text.splitlines():
        # --- REMOÇÕES INLINE DE BOILERPLATE (não dependem de linha inteira) ---

        # 1) Variações de "Laudo Gerado/Lib[e]rado por Sistema Especialista"
        ln = re.sub(r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?', ' ', ln)

        # 2) Variações de "Para Visualizar/Visualização do Laudo Acesse/Acessar o Viewer/Opção Imagem"
        ln = re.sub(r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.?\b(?:acesse|acessar)\b.?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?', ' ', ln)

        # 3) Variações de "Laudo pode não estar completo na visualização em RTF/Imagem"
        ln = re.sub(r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?', ' ', ln)

        raw = ln
        s = ln.strip()
        sl = s.lower()

        if not s:
            out.append(raw); continue

        # 1) Linhas com muitos ';' (lista de fontes/estilos/cores)
        if s.count(';') >= 3:
            hits = sum(1 for w in _FONT_WORDS if w in sl)
            if hits >= 1 or re.search(r'\b(font|heading|colortbl|stylesheet|style)\b', sl):
                continue
            if re.fullmatch(r'(?:[^;]{1,40};\s*){5,}', s):
                continue

        # 2) Cabeçalhos estilo "heading 1; heading 2; ..."
        if re.fullmatch(r'(?:heading\s*\d+\s*;\s*){2,}heading\s*\d+\s*;?', sl):
            continue

        # 3) Geradores e metadados
        if re.search(r'(created by|generator|html\s*to\s*rtf|jword)', sl):
            continue

        # 4) Apenas pontuação/ruído
        if re.fullmatch(r'[;,\.\-–—\s]+', s):
            continue

        # 5) Avisos genéricos (texto normalizado, acento-insensível)
        ns = norm(s)
        if ("laudo" in ns and "visualiza" in ns and ("rtf" in ns or "imagem" in ns or "viewer" in ns or "sistema" in ns or "especialista" in ns)):
            continue
        if re.search(r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\b", ns):
            continue

        # 6) Linhas curtas com 2+ nomes de fontes + ';'
        if ';' in s and sum(1 for w in _FONT_WORDS if w in sl) >= 2 and len(s) <= 160:
            continue

        out.append(raw)

    return "\n".join(out).strip()


def _final_punct_cleanup(text: str) -> str:
    if not text:
        return ""
    # espaço antes de pontuação
    text = re.sub(r'\s+([.,;:])', r'\1', text)
    # espaço depois de abertura de parênteses/colchetes e antes de fechamento
    text = re.sub(r'([\(\[\{])\s+', r'\1', text)
    text = re.sub(r'\s+([\)\]\}])', r'\1', text)
    # múltiplas pontuações -> uma
    text = re.sub(r'([.,;:]){2,}', r'\1', text)
    # linhas em branco extras
    text = re.sub(r'\n{3,}', '\n\n', text)
    # trims por linha
    lines = [ln.rstrip() for ln in text.splitlines()]
    return "\n".join(lines).strip()



def to_plain(s: str) -> str:
    """
    Conversão robusta (RTF/HTML/texto) -> texto limpo:
      1) detecta RTF/HTML e usa parser dedicado (striprtf / lxml / html2text)
      2) ftfy para arrumar unicode/mojibake
      3) normalizações e correções de espaços intra-palavra/acento
      4) limpeza de boilerplate típico de RTF/HTML
      5) remoção de rodapés conhecidos e ajuste de pontuação
    """
    if not s:
        return ""

    raw = s.replace("\r\n", "\n").replace("\r", "\n")

    # --- NOVO: alguns dumps chegam com escapes literais "\r\n" e "\n" (texto, não EOL real) ---
    if "\\r\\n" in raw:
        raw = raw.replace("\\r\\n", "\n")

    # troca "\n" literal só quando é claramente quebra (antes de "{", "\" , espaço ou fim)
    if "\\n" in raw:
        raw = re.sub(r'\\n(?=[\s{\\}]|$)', "\n", raw)

    # --- 1) Detectar formato ---
    head = raw[:512].lstrip()
    is_rtf  = head.startswith("{\\rtf") or "\\rtf" in head.lower()[:128]
    is_html = bool(re.search(r"</?(html|head|body|div|p|span|br|table|tr|td|img)\b", raw[:2000], re.I))

    txt = raw
    try:
        if is_rtf:
            try:
                # tenta pandoc se existir (melhor preservação de estrutura em alguns RTF)
                try:
                    import pypandoc
                    txt = pypandoc.convert_text(raw, 'plain', format='rtf', extra_args=['--wrap=none'])
                except Exception:
                    txt = rtf_to_text(raw)
            except Exception:
                txt = _rtf_to_text_fallback(raw)
        elif is_html:
            txt = html_to_plain(raw)
        else:
            txt = raw
    except Exception:
        # fallback bruto caso algo quebre
        txt = raw

    # --- 2) Unicode/encodes (corrige mojibake, NBSP etc.) ---
    try:
        txt = ftfy.fix_text(txt)
    except Exception:
        pass

    # invisíveis e normalização
    txt = (txt.replace("\u200b", "")
              .replace("\u200c", "")
              .replace("\xa0", " "))
    txt = unicodedata.normalize("NFKC", txt)

    # --- 3) Colapsar cabeçalhos ALL CAPS espaçados (CON C LU S Ã O) ---
    RX_CAPS_LINE   = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
    RX_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)
    def _collapse_caps(line: str) -> str:
        if RX_CAPS_LINE.match(line.strip()):
            return line.replace(" ", "")
        return RX_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), line)
    txt = "\n".join(_collapse_caps(ln) for ln in txt.splitlines())

    # --- 4) Remover prefixo numérico de OCR grudado (ex.: "6reto" -> "reto"), sem afetar dosagens ---
    UNITS = {
        "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
        "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
        "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
    }
    RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
    RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
    RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')
    def _strip_digit_prefix(m: re.Match) -> str:
        tok = m.group(0)
        if RX_DOSAGE.match(tok) or RX_FLOW.match(tok) or m.group(2).lower() in UNITS:
            return tok
        return m.group(2)
    txt = RX_OCR_PREFIX.sub(_strip_digit_prefix, txt)

    # --- 5) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
    ACC = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
    for _ in range(4):
        before = txt

        # letras acentuadas separadas do resto
        txt = re.sub(rf'(?i)\b([{ACC}])\s+([A-Za-z]{{1,30}})\b', r'\1\2', txt)
        txt = re.sub(rf'(?i)\b([A-Za-z]{{1,30}})\s+([{ACC}][A-Za-z]{{1,5}})\b', r'\1\2', txt)

        # trinca com acento no meio: "p ó s" -> "pós"
        txt = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACC}])\s+([A-Za-z])\b', r'\1\2\3', txt)

        # sufixos comuns
        txt = re.sub(r'(?i)çã\s+o\b',  'ção',  txt)
        txt = re.sub(r'(?i)çõ\s+es\b', 'ções', txt)
        txt = re.sub(r'(?i)ã\s+o\b',   'ão',   txt)
        txt = re.sub(r'(?i)õ\s+es\b',  'ões',  txt)

        # hífen com espaços indevidos
        txt = re.sub(rf'(?i)\b([A-Za-z{ACC}]{{1,40}})\s*-\s*([A-Za-z{ACC}]{{1,40}})\b', r'\1-\2', txt)

        # casos de espaço antes de letras acentuadas dentro de palavra (cola sempre)
        txt = re.sub(rf'\b([A-Za-z]{{1,4}})\s+(?=[{ACC}])', r'\1', txt)
        txt = re.sub(rf'\b([{ACC}])\s+(?=[A-Za-z]{{1,3}}\b)', r'\1', txt)

        if txt == before:
            break

    # --- 6) Limpeza de boilerplate herdado de RTF/HTML ---
    lines = []
    for ln in txt.splitlines():
        s = ln.strip()
        sl = s.lower()
        if not s:
            lines.append(ln); continue
        # lixos clássicos de RTF/HTML
        if any(tok in sl for tok in ("fonttbl","colortbl","stylesheet","generator",
                                     "heading","listoverridetable","listtable")):
            continue
        if s.count(';') >= 2 and re.search(r'(arial|times|calibri|courier|symbol|wingdings)', sl):
            continue
        # linhas só de pontuação/traços
        if re.fullmatch(r'[;,\.\-–—\s]+', s):
            continue
        lines.append(ln)
    txt = "\n".join(lines)

    # --- 7) Espaços/pontuação finais e quebras ---
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\s*\n\s*', '\n', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    txt = re.sub(r'\s+([.,;:])', r'\1', txt)
    txt = re.sub(r'([\(\[\{])\s+', r'\1', txt)
    txt = re.sub(r'\s+([\)\]\}])', r'\1', txt)

    # --- 8) Remover rodapés/avisos de visualização incompleta e limpar pontas ---
    txt = remove_final_laudo(txt).strip()
    return txt

###### novo trecho recomendado para substituicao acima



#“Pretty name” do achado (tira underscores)
def _pretty_finding_name(f: str) -> str:
    return f.replace("_", " ").strip()

def _any_q_in_line(line_norm: str, queries_norm: set[str]) -> bool:
    # casa por token, não só por substring exata
    if not queries_norm:
        return False
    for q in queries_norm:
        if not q: 
            continue
        # aceita q inteiro ou seus tokens (ex.: "nodulo hipervascular" -> "nodulo" OU "hipervascular")
        toks = [t for t in q.split() if len(t) >= 4] or [q]
        if any(t in line_norm for t in toks):
            return True
    return False

def _shorten_clause(line_raw: str, query_norm: str, max_chars: int = 300) -> str: ## max_chars inicial era 160
    """
    Devolve o menor trecho útil dentro da linha que contém a query:
    - tenta cortar por pontuação (. ; : – ) ( )
    - se ainda ficar longo, usa uma janela em torno do match
    """
    ln = line_raw.strip()
    if len(ln) <= max_chars:
        return ln

    # 1) tenta quebrar por cláusulas
    parts = re.split(r'([.;:–\-\(\)])', ln)  # mantém separadores
    # remonta pares (texto+sep) para preservar sentenças curtas
    chunks = []
    cur = ""
    for p in parts:
        cur += p
        if p in ".;:–-)" and cur.strip():
            chunks.append(cur.strip())
            cur = ""
    if cur.strip():
        chunks.append(cur.strip())

    # procura a cláusula que contém a query (normalizada)
    q = norm(query_norm)
    for c in chunks:
        if q and norm(c).find(q) != -1:
            return c if len(c) <= max_chars else c[:max_chars].rsplit(" ", 1)[0] + "…"

    # 2) janela em torno do match (acentos-insensível)
    pat = _accent_rx(q) if q else None
    if pat:
        m = re.search(pat, ln, flags=re.IGNORECASE)
        if m:
            left = max(0, m.start() - max_chars//2)
            right = min(len(ln), m.end() + max_chars//2)
            snippet = ln[left:right].strip()
            # aparar nas bordas para próximo espaço
            if left > 0:
                snippet = "…" + snippet.split(" ", 1)[-1]
            if right < len(ln):
                snippet = snippet.rsplit(" ", 1)[0] + "…"
            return snippet

    # fallback
    return (ln[:max_chars].rsplit(" ", 1)[0] + "…") if len(ln) > max_chars else ln

#Compactador (escolhe 1 frase por achado, preferindo não-negado)
NEG_HINTS = (" sem ", " ausencia ", " nao ")
def compact_summary_from_hits(hits: list[dict], doc, organ_blocks: list[dict]) -> list[dict]:
    # 1) só positivos
    pos = [h for h in hits if not h.get("negated", False)]
    if not pos:
        return []

    # 2) Conclusão — RAW e NORM >> alterado pela funcao seguinte
    # concl_lines_raw = extract_section_lines(doc.text, "conclusao")
    # concl_lines_norm = [norm(ln) for ln in concl_lines_raw]
    #     # 2) Seção final (Conclusão / Impressão Diagnóstica) — RAW e NORM
    ## >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
    FINAL_HEADERS = [
        "impressao diagnostica",
        "impressão diagnóstica",
        "impressao",
        "impressao:",
        "impressão",
        "impressão:",
        "conclusao",
        "conclusão",
        "notamse",
        "notam-se",
        "notam se",
        "notam"
    ]

    final_lines_raw = []
    final_lines_norm = []

    for header in FINAL_HEADERS:
        lines = extract_section_lines(doc.text, header)
        if lines:
            final_lines_raw = lines
            final_lines_norm = [norm(ln) for ln in lines]
            break  # achou a primeira seção final disponível
    ## <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<


    # 3) Fallback por bloco do órgão — RAW e NORM
    organ_texts = {b["organ"]: doc.text[b["span"].start_char:b["span"].end_char] for b in organ_blocks}
    organ_lines_raw = {org: [ln.strip() for _,_,ln in _line_spans_text(txt)]
                       for org, txt in organ_texts.items()}
    organ_lines_norm = {org: [norm(ln) for ln in lines]
                        for org, lines in organ_lines_raw.items()}

    # 4) agrupa por achado
    by_f = {}
    for h in pos:
        by_f.setdefault(h["finding"], []).append(h)

    out = []
    for f, hs in by_f.items():
        # ordena por confiança desc e span menor
        hs.sort(key=lambda h: (-h.get("confidence", 0.0), h["end_char"] - h["start_char"]))

        # queries: surface + tokens do nome do achado + padrões auxiliares
        queries = set()
        for h in hs:
            s = norm(h.get("surface",""))
            if s:
                queries.add(s)
                # também unigrams da surface (pega "nodulo", "hipervascularizado")
                for t in s.split():
                    if len(t) >= 4:
                        queries.add(t)
        # nome canônico
        fname = _pretty_finding_name(f)               # ex.: "nodulo hipervascular"
        queries.add(norm(fname))
        for t in norm(fname).split():
            if len(t) >= 4:
                queries.add(t)

        # heurística: se é achado nodular, garanta termos gerais
        if "nodulo" in queries or "nodule" in queries:
            queries |= {"nodulo", "nodular", "lirads", "li-rads"}

        # substituido pela funcao a seguir
        # alterado pela funcao a seguir
        # best_line = None
        # best_q = ""
        # # # 4.1 Conclusão (preferência forte) — devolve RAW encurtado
        # for i, ln_norm in enumerate(concl_lines_norm):
        #     if _any_q_in_line(ln_norm, queries) and not any(h in ln_norm for h in NEG_HINTS):
        #         best_line = concl_lines_raw[i]
        #         # escolhe a query mais discriminativa para cortar
        #         # prioriza "lirads" ou "nodulo"
        #         if "lirads" in ln_norm:
        #             best_q = "lirads"
        #         elif "nodulo" in ln_norm:
        #             best_q = "nodulo"
        #         else:
        #             # pega a maior query que exista na linha (mais específica)
        #             best_q = max([q for q in queries if q in ln_norm], key=len, default="")
        #         best_line = _shorten_clause(best_line, best_q, max_chars=160)
        #         break
        ## >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
        best_line = None
        best_q = ""
        # 4.1 Seção final (Conclusão / Impressão) — devolve RAW encurtado
        for i, ln_norm in enumerate(final_lines_norm):
            if _any_q_in_line(ln_norm, queries) and not any(h in ln_norm for h in NEG_HINTS):
                best_line = final_lines_raw[i]

                # escolhe a query mais discriminativa para cortar
                if "crohn" in ln_norm:
                    best_q = "crohn"
                elif "retocolite" in ln_norm:
                    best_q = "retocolite"
                else:
                    present = [q for q in queries if q in ln_norm]
                    best_q = max(present, key=len, default="")

                best_line = _shorten_clause(best_line, best_q, max_chars=300) ## max_chars era 169
                break
        ## <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

        # 4.2 Fallback: bloco do órgão — RAW encurtado
        if best_line is None:
            organ = hs[0]["organ"]
            onorm = organ_lines_norm.get(organ, [])
            oraw  = organ_lines_raw.get(organ, [])
            for i, ln_norm in enumerate(onorm):
                if _any_q_in_line(ln_norm, queries):
                    # query mais específica presente na linha
                    present = [q for q in queries if q in ln_norm]
                    qbest = max(present, key=len) if present else ""
                    best_line = _shorten_clause(oraw[i], qbest, max_chars=300) ## max_chars era 169
                    break

        # 4.3 Último recurso: extrair contexto do documento usando a posição do hit
        if best_line is None:
            # Pegar o hit com maior confiança
            best_hit = hs[0]
            start_char = best_hit["start_char"]
            end_char = best_hit["end_char"]
            
            # Encontrar a sentença que contém o hit
            finding_sent = None
            for sent in doc.sents:
                if sent.start_char <= start_char < sent.end_char:
                    finding_sent = sent
                    break
            
            if finding_sent:
                # Usar a sentença completa
                sent_text = finding_sent.text.strip()
                
                # Se a sentença for muito longa, encurtar em torno do hit
                if len(sent_text) > 200:
                    # Calcular posição relativa do hit na sentença
                    hit_pos_in_sent = start_char - finding_sent.start_char
                    
                    # Janela de 100 chars antes e depois
                    window_start = max(0, hit_pos_in_sent - 100)
                    window_end = min(len(sent_text), hit_pos_in_sent + 100)
                    
                    snippet = sent_text[window_start:window_end].strip()
                    
                    # Adicionar reticências se cortou
                    if window_start > 0:
                        snippet = "…" + snippet.split(" ", 1)[-1] if " " in snippet else snippet
                    if window_end < len(sent_text):
                        snippet = snippet.rsplit(" ", 1)[0] + "…" if " " in snippet else snippet + "…"
                    
                    best_line = snippet
                else:
                    best_line = sent_text
            else:
                # Fallback: extrair janela em torno do hit
                window_start = max(0, start_char - 100)
                window_end = min(len(doc.text), end_char + 100)
                snippet = doc.text[window_start:window_end].strip()
                
                # Adicionar reticências
                if window_start > 0:
                    snippet = "…" + snippet
                if window_end < len(doc.text):
                    snippet = snippet + "…"
                
                best_line = snippet

        out.append({ _pretty_finding_name(f): best_line })
    return out


@lru_cache(maxsize=8192)
def _accent_rx(t: str) -> str:
    rep = {
        'a': '[aáàâãä]', 'e': '[eéèêë]', 'i': '[iíìîï]',
        'o': '[oóòôõö]', 'u': '[uúùûü]', 'c': '[cç]', 'n': '[nñ]'
    }
    out = []
    for ch in t.lower():
        out.append(rep.get(ch, re.escape(ch)))
    return ''.join(out)

#========================================================================

# termos genéricos de 1 palavra que geram falso positivo — ajuste conforme seu corpus
GENERIC_SINGLETON_BAN = {
    "livre","leito","difusa","difuso","vias","via","direito","esquerdo","normal",
    "discreto","discreta","helicoidal","multislice","aortoiliaca","contornos",
    "calibre","material","gasoso","fecal","porta","veia","arteria","hepatico",
    "hepatica","hilo","anastomose","fluxo","indice","resistencia","propria",
    "ramo","velocidade","pre","pos","maxima","mucosa","vascular","padrao",
    "padrao vascular","conteudo","homogeneo","integridade", "parietal",
    "questiona-se", "questionase", "questiona se", "retroperitoneal","retroperitoneais",
    "retroperitonio", "degenerativo", "degenerativos", "degenerativa", "degenerativas", 
    "mecanico", "mecanica", "mecanicas"
}  # como singleton; em compostos ok

def token_set(s: str) -> set[str]:
    return set([t for t in norm(s).split() if t and t not in STOP_WORDS_NORM])

def term_is_valid_for_seeds(term: str, seeds: list[str]) -> bool:
    toks = token_set(term)
    if not toks:
        return False

    # se houver seed multi-termo, exija candidato >=2 tokens
    if any(len(token_set(s)) >= 2 for s in seeds) and len(toks) < 2:
        return False

    # singletons banidos
    if len(toks) == 1 and next(iter(toks)) in GENERIC_SINGLETON_BAN:
        return False

    # sobreposição "literal"
    seed_tokens = set().union(*[token_set(s) for s in seeds])
    overlap_plain = len(toks & seed_tokens)
    if overlap_plain >= 1:
        return True

    # ---- raiz morfológica leve (agora unificada e um pouco mais robusta) ----
    term_root = {_rootize_token(t) for t in toks}
    seed_root = set().union(*[{_rootize_token(t) for t in token_set(s)} for s in seeds])

    overlap_root = len(term_root & seed_root)
    return overlap_root >= 1


#========================================================================

# --- CONFIG embutida (edite aqui) ---
CONFIG = {
    "negation": {
        "tokens": [
            "nao", "sem", "ausencia de", "negado", "nega", "descarta", "livre de", "sem evidencias de",
            "sem sinais de", "sem achados de", "sem alteraçoes", "sem alteracoes", "sem sinais", "sem achados",
            "sem alteracao", "sem alteraçao", "sem polipos", "sem diverticulos", "nao sugerindo",
            "não sugerindo", "questionase", "questiona-se", "questiona se"
        ],
        "window_tokens": 7  # alcance simples de negação
    },
    "organs": {
        # Cada órgão: seeds e (opcional) regex adicionais
    # --- Abdome superior ---
    "figado": {
        "seeds": ["figado", "fígado", "hepatico", "hepática"],
        "regex": [
            r"\bf[ií]gado\b",
            r"\bhep[aá]tic[ao]\b",
            r"\besteatose(\s+hep[aá]tica)?\b",
            r"\bhepatomegalia\b",
        ],
    },
#------------------------ seeds e regex usados no algoritmo de doencas biliares ----------------------------------------
    "doencas_biliares":{
        "seeds": [
            #vias biliares:
            "vias biliares", "vias biliares intra e extra-hepaticas", "vias biliares intra e extra-hepáticas",
            "ducto hepático comum","ducto hepatico comum", "ducto hepático direito", "ducto hepático esquerdo",
            "ducto hepatico direito", "ducto hepatico esquerdo",
            "ducto cistico", "ducto cístico", "coledoco", "colédoco",
            "ducto bilar comum"
            #vesicula biliar:
            "vesicula", "vesícula","vesicula biliar", "vesícula biliar"
        ],
        "regex": [
            #vias biliares:
            r"\bvias\s+biliares\b",
            r"\bcol[eé]doco\b",
            r"\bdilata[cç][aã]o\b.*\bvias\s+biliares\b",
            #vesiula biliar:
            r"\bves[ií]cula(\s+biliar)?\b",
            r"bc[aá]lculos?\s+na\s+ves[ií]cula\b",
        ],
    },
#---------------------------------------------------------------------------------------------------------------
     "pancreas": {
        "seeds": ["pancreas", "pâncreas"],
        "regex": [
              r"\bp[aâ]ncreas\b|\bpancreas\b",
            r"\bpancreatite\b",
            r"\bducto\b.*\bpancre[aá]tic[ao]\b|\bwirsung\b",
        ],
     },
    "baco": {
        "seeds": ["baco", "baço"],
        "regex": [
            r"\bba[cç]o\b|\bbaco\b",
            r"\besplenomegalia\b",
            r"\bespl[eê]nic[ao]\b",
        ],
    },
    "rins": {
        "seeds": ["rim", "rins", "renal"],
        "regex": [
            r"\brins?\b",
            r"\bnefrolit[ií]ase\b",
            r"\bhidronefrose\b",
            r"\bcistos?\s+renais?\b|\bcisto\s+renal\b",
        ],
    },
    "ureteres": {
        "seeds": ["ureter", "ureteres"],
        "regex": [
            r"\buret[eé]res?\b|\bureter\b"
        ],
    },
    "bexiga": {
        "seeds": ["bexiga"],
        "regex": [
            r"\bbexiga\b"
        ],
    },
    "adrenais": {
        "seeds": ["adrenal", "adrenais", "suprarrenal", "suprarrenais", "supra-renais"],
        "regex": [
            r"\badrenais?\b|\badrenal\b",
            r"\bsuprarrenais?\b|\bsuprarrenal\b|\bsupra-?renais\b",
            r"\badenoma\b.*\badrenal\b|\bhiperplasia\b.*\badrenal\b",
        ],
    },
    "utero": {
        "seeds": ["utero", "útero","mioma", "leiomioma","endometrio", "endométrio"],
        "regex": [
            r"\b[uú]tero\b|\butero\b",
            r"\bmioma(s)?\b|\bleiomioma(s)?\b",
            r"\bendom[eé]trio\b",
            r"\bdispositivo\s+intrauterino\b|\bdiu\b",
        ],
    },
    "ovarios": {
        "seeds": ["ovario", "ovário"],
        "regex": [
            r"\bov[aá]ri[oa]s?\b|\bovario\b|\bovário\b",
            r"\banexos\b",
            r"\bcisto\b.*\bov[aá]ri[oa]\b|\bmassa\b.*\banexial\b",
        ],
    },

    "prostata": {
        "seeds": ["prostata", "próstata"],
        "regex": [
            r"\bpr[oó]stata\b|\bprostata\b",
            r"\bhiperplasia\b.*\bprost[aá]tic[ao]\b",
        ],
    },
    "vasos": {
        "seeds": ["vasos","aorta", "aorta abdominal","veia cava", "veia cava inferior"],
        "regex": [
            r"\baorta\b",
            r"\bveia\s+cava\b",
            r"\bveia\s+porta\b|\bporta\b",
            r"\baneurisma\b|\btrombose\b",
        ],
    },

    "peritonio": {
        "seeds": ["peritonio", "peritônio"],
        "regex": [
            r"\bperit[oô]nio\b|\bperitonio\b",
            r"\bl[ií]quido\s+livre\b|\bascite\b",
            r"\bpneumoperit[oô]nio\b|\bpneumoperitonio\b",
        ],
    },

    "retroperitonio": {
        "seeds": ["retroperitonio", "retroperitônio"],
        "regex": [
            r"\bretroperit[oô]nio\b|\bretroperitonio\b",
            r"\blinfonodomegalia(s)?\b|\blinfonodos?\b",
        ],
    },

    # --- Pulmão base / MSK ---
    "pulmao": {
        "seeds": ["bases pulmonares","pulmao", "pulmão", "pulmoes", "pulmões", "toracoabdominal"],
        "regex": [
            r"\bbases?\s+pulmonares\b",
            r"\bpulm[oõ]es?\b|\bpulmao\b|\bpulmão\b",
            r"\batelectasia\b|\bderrame\s+pleural\b",
        ],
    },

    "musculoesqueletico": {
        "seeds": ["partes moles","estruturas osseas", "estruturas ósseas"],
        "regex": [
            r"\bpartes\s+moles\b",
            r"\bestruturas?\s+os[só]eas\b",
            r"\bespondilose\b|\bartrose\b|\bfratura\b",
        ],
    },

    # --- Final (não precisa findings) ---
    "conclusao": {
        "seeds": ["conclusao", "conclusão","impressao", "impressão","impressao diagnostica", "impressão diagnóstica"],
        "regex": [
            r"\bconclus[aã]o\b",
            r"\bimpress[aã]o\b",
            r"\bimpress[aã]o\s+diagn[oó]stica\b",
        ],
    },
    "colon_reto": {
        "seeds": [
            # Termos gerais
            "colon", "cólon",
            # Segmentos do cólon
            "colon ascendente", "colon ascendente", "colon transverso", "colon transverso",
            "colon descendente", "colon descendente", "colon sigmoide", "colon sigmoide", 
            "colon direito", "colon esquerdo",
            # Ceco e apêndice
            "ceco", "apendice", "apêndice", "apendice cecal", "apêndice cecal",
            "apendice retrocecal", "apêndice retrocecal", "apendice vermiforme", "apêndice vermiforme",
            # Sigmoide
            "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
            # Reto
            "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
            # Transição
            "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide", "transição reto-sigmoide",
            "juncao retossigmoide", "junção retossigmoide",
            # Alças e estruturas
            "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon", "alças do cólon",
            "parede colonica", "parede colônica", "intestino grosso"
        ],
        "regex": [
            r"\bc[oó]l[oô]n\b",  # cólon ou colon (SEM "oscopia")
            r"\breto(s)?\b",  # reto ou retos
            r"\bsigm[oó]ide\b",  # sigmoide ou sigmóide
            r"\b(c[oó]l[oô]n|colon)\s+(ascendente|transverso|descendente)\b",
            r"\b(ascendente|transverso|descendente)\s+do\s+(c[oó]l[oô]n|colon)\b",  # segmentos
            r"\bceco\b",  # ceco
            r"\bap[eê]ndice( cecal| retrocecal| vermiforme)?\b",  # apêndice e variações
            r"\bv[aá]lvula (i|í)leo ?-?cecal\b",  # válvula ileocecal
            r"\btransi[cç][aã]o reto-?sigm[oó]ide\b",  # transição retossigmoide
            r"\bjun[cç][aã]o retossigm[oó]ide\b",  # junção retossigmoide
            r"\bal[cç]as col[oô]nicas\b",  # alças colônicas
            r"\bparede col[oô]nica\b",  # parede colônica
            r"\bdivert[ií]culos?( col[oô]nicos?| c[oó]licos?)?\b",  # divertículos e variações
            r"\bdiverticulose( col[oô]nica)?\b", # diverticulose
        ]  
    },
    "reumatologia": {
        "seeds": [
            # Termos gerais da coluna
            "coluna", "coluna vertebral", "coluna espinal", "espinha dorsal",
            "vertebra", "vértebra", "vertebras", "vértebras",
            # Regiões específicas
            "cervical", "torácica", "toracica", "lombar", "sacral",
            "coluna cervical", "coluna toracica", "coluna torácica", "coluna lombar", "coluna sacral",
            "região cervical", "região toracica", "região torácica", "região lombar", "região sacral",
            # Estruturas vertebrais
            "corpo vertebral", "corpos vertebrais", "disco intervertebral", "discos intervertebrais",
            "faceta articular", "facetas articulares", "processo espinhoso", "processos espinhosos",
            "apófise espinhosa", "apófises espinhosas",
            # Termos de movimento/posicionamento
            "espondilo", "espondilite", "espondilartrite",
            # Ligamentos e estruturas adjacentes
            "ligamento longitudinal", "ligamento amarelo", "ligamento nucal",
            # Termos gerais
            "articulação", "articulacoes", "articulações", "junta", "juntas",
            "artrite", "artrites", "artralgias", "artralgia",
            # Articulações específicas
            "articulação sacroilíaca", "articulacao sacroiliaca", "articulação sacroilíaca",
            "sacroilíaca", "sacroiliaca", "sacroilíacas", "sacroiliacas",
            "articulação do quadril", "articulacao do quadril", "quadril", "quadris",
            "articulação do joelho", "articulacao do joelho", "joelho", "joelhos",
            "articulação do tornozelo", "articulacao do tornozelo", "tornozelo", "tornozelos",
            "articulação do ombro", "articulacao do ombro", "ombro", "ombros",
            "articulação do punho", "articulacao do punho", "punho", "punhos",
            "articulações das mãos", "articulacoes das maos", "articulações das mãos",
            "articulações dos pés", "articulacoes dos pes", "articulações dos pés",
            "articulações interfalangeanas", "articulacoes interfalangeanas",
            "articulações metacarpofalangeanas", "articulacoes metacarpofalangeanas",
            # Estruturas articulares
            "sinovial", "sinóvia", "cartilagem articular", "cartilagem hialina",
            "membrana sinovial", "membrana sinovial", "cápsula articular", "capsula articular",
            "espaço articular", "espaco articular", "espaço intra-articular", "espaco intra-articular"
        ],
        "regex": [
            # >>> regex SEM acento (porque o texto de match está sem acento)
            r"\bcoluna( vertebral| espinal)?\b",
            r"\bvertebr(a|as)\b",
            r"\b(cervical|toracica|lombar|sacral)\b",
            r"\bespondil\w*\b",
            r"\bdisco(s)? intervertebral(is)?\b",
            r"\bcorpo(s)? vertebral(is)?\b",
            r"\bfaceta(s)? articular(es)?\b",
            r"\bapofise(s)? espinhosa(s)?\b",          # <<< sem acento
            r"\barticulacao(oes)?\b",                  # <<< sem acento + vírgula OK
            r"\barticulacoes?\b",
            r"\bjunta(s)?\b",
            r"\bartrite(s)?\b",
            r"\bartralgia(s)?\b",
            r"\bsacroiliac(a|as)\b",                   # sacroiliaca(s)
            r"\b(quadril|joelho|tornozelo|ombro|punho)\b",
            r"\b(sinovial|sinovia|cartilagem articular)\b",
            r"\bcapsula articular\b",
            r"\bmembrana sinovial\b",
            r"\bespaco(s)? (intra-)?articular(es)?\b",
        ]
    }
    },
    "findings": {
        # Achados por órgão (canônicos + seeds). A expansão semântica cria sinônimos dinâmicos no runtime
        "colon_reto": {
        "lesao": [
            "lesao", "lesoes", "lesao vegetante", "lesao infiltrativa", "lesao exofitica", "lesao deprimida",
            "estenose por lesao", "lesao suspeita", "lesao ulcerada",  "lesão estenosante"
        ],
        "polipo": [
            "polipo", "polipos", "polipo pediculado", "polipo sessil","polipo sésil","polipo sesseil",
            "lesao plana", "adenoma colorretal", "lesao sessil","lesao séssil", "lesao pediculada",
            "paris 0-ii", "paris 0-iia", "paris 0-iib", "paris 0-iic", "paris 0-is",
            "polipo suspeito", "polipos suspeitos"
        ],
        "ulceracao":[
            "ulcera", "ulcero", "ulcerativa", "ulcero infitrativa", "ulcero-infiltrativa", "ulceracao", "ulceracoes", "erosao", "erosoes",
            "erosiva", 
        ],
        "tumor_ou_massa": [
            "massa", "tumor", "tumoracao", "tumoracoes", "tumoracao exofitica", "laterally spreading tumor", "lst", "lst-g", "lst-ng", "neoplasia", "adenocarcinoma", "carcinoma", "neoplasica", 
            "neoplasico", "metastase"
        ]
        },
        "reumatologia": {
            "espondilite": [
                "espondilite anquilosante",
                "sindesmofito marginal",
                "anquilose vertebral",
                "romanus"
            ],
            "sacroileite": [
                "sacroileite",
                "sacroilite",
                "sacroiliite",
                "sacroileite bilateral",
                "erosoes sacroiliacas",
                "edema osseo sacroiliaco",
                "anquilose sacroiliaca",
                "sinovite sacroiliaca"
            ],
            "artrite_reumatoide": [
                "artrite reumatoide",
                "reumatoide",
                "artrite reumatoide ativa",
                "pannus sinovial inflamatorio",
                "sinovite proliferativa",
                "erosoes osseas marginais"
            ],
            "artrite_psoriasica": [
                "artrite psoriasica",
                "artropatia psoriasica",
                "psoriasica",
                "dactilite",
                "entesite inflamatoria",
                "periostite inflamatoria",
                "sindesmofito nao marginal",
                "acroosteolise"
                ],
            "nodulo_reumatoide": [
                "nodulo reumatoide",
                "nodulos reumatoides",
                "lesao nodular reumatoide"
            ]
        },
        "doencas_biliares": {
            "diagnosticos": [
                "colecistite cronica",
                 "coledocolitiase", 
                 "pancreatite biliar",
                 "ileo biliar",
                 "hidropsia de vesicula", 
                 "carcinoma de vesicula",
                 "vesicula hidropica",
                 "vesicula em porcelana"
            ],
            "colelitiase":[
                "calculo",
               # "focos hiperecogenicos com sombra acustica",
               # "imagens arredondadas hiperecogenicas com sombra acustica", 
                "sinal da parede-Eco-sombra", 
                "sinal WES"
            ],
            "neoplasias":[
                "polipo sugestivo de malignidade",
                "polipo sessil",
                "polipo com realce intenso",
               # "polipo maior que 1 cm",
                "massa sugestiva de neoplasia",
                "massa heterogenea",
                "massa heterogenea com distensao vascular"
                "paredes irregulares",
                "invasao de parede" 
            ],
            "colecistite cronica":[
                "calculo no ducto cistico",
                "calculo no infundibulo",
                "calculo no colo",
                "fibrose da vesicula",
                "retração da vesicula",
                "vesicula contrida", 
                "paredes espessadas",
                "paredes fibroticas",
                "calcificação das paredes",
                "infiltrado inflamatorio cronico",
                "seios de Rokitansky-Aschoff",
                "sindrome de Mirizzi"
            ],
            "coledocolitiase":[
                "dilacao",
                "calculo no coledoco",
                "coledoco dilatado"
            ],
            "sinais inflamatorios":[
                "espessamento de parede",
                "colecao pericolecistica",
                "distensao da vesicula",
                "vesicula aumentada",
                "calculo impactado",
                "sinal de Murphy ultrassonografico",
                "empiema de vesicula"
            ]
        }
    },
    # Limiar de similaridade para incluir novos termos do texto no léxico semântico
    "semantic": {
        "min_sim_seed_vocab": 0.65,  # quão parecido um termo do vocabulário do laudo precisa ser com os seeds
        "top_k_per_seed": 24,
        # Geramos candidatos do vocabulário via noun_chunks + n-grams leves
        "min_token_len": 3
    }
}

# ===
# --- Seleção de órgão alvo (string ou lista). Deixe None para todos (não recomendado).
# TARGET_ORGAN = "colon_reto"   # ex.: "figado" ou ["figado","rim"] ou None
TARGET_ORGAN = ["doencas_biliares"]
FORCE_FULL_DOC_FOR = set()

# ===
# --- Stopwords com preservação de operadores de negação ---
# base spaCy
SPACY_STOP = set(w.lower() for w in nlp.Defaults.stop_words)

# operadores de negação que NÃO devem ser removidos
OPERATORS = {"não", "nao", "sem", "há", "ha"}  # pode incluir mais

# também considere os tokens de negação que já listamos no CONFIG
OPERATORS |= set(CONFIG["negation"]["tokens"])

# normaliza tudo (mesmo critério da função norm)
def _norm_plain(s: str) -> str:
    import unicodedata, re
    s = s.lower().strip()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"\s+", " ", s)
    return s

STOP_WORDS_NORM = { _norm_plain(w) for w in SPACY_STOP }
OPERATORS_NORM  = { _norm_plain(w) for w in OPERATORS }

# remove operadores do conjunto de stopwords
STOP_WORDS_NORM = STOP_WORDS_NORM - OPERATORS_NORM

#=======================================================

# Cabeçalho: começa no início da linha, pode ter bullet, NOME:, e pode TER CONTEÚDO depois do ":"
HEADER_RX = re.compile(r'^\s*[-•]?\s*([A-ZÁ-Úa-zá-ú/ ]{2,})\s*:\s*', re.UNICODE)

def _line_spans_text(text: str):
    spans, off = [], 0
    for line in text.splitlines(True):
        spans.append((off, off+len(line), line))
        off += len(line)
    return spans

def _norm_header_name(s: str) -> str:
    return norm(s.replace(":", "").replace("-", " ").strip())

def extract_section_lines(text: str, header_name: str) -> list[str]:
    """Linhas (brutas) da seção `header_name` até o próximo cabeçalho."""
    lines = _line_spans_text(text)
    headers = []
    for (s,e,line) in lines:
        m = HEADER_RX.match(line)
        if m:
            headers.append((s, _norm_header_name(m.group(1))))
    if not headers:
        return []
    target = _norm_header_name(header_name)
    starts = [s for (s,name) in headers if name.startswith(target)]
    if not starts:
        return []
    s0 = starts[0]
    nexts = [S for (S,_) in headers if S > s0]
    s1 = nexts[0] if nexts else len(text)
    sub = text[s0:s1]
    return [ln.strip() for _,_,ln in _line_spans_text(sub)]

# Sinônimos básicos para casar cabeçalho textual com órgão de CONFIG
HEADER_ALIASES = {
    "figado": {"figado", "fígado"},
    "vias_biliares": {"vias biliares", "colédoco", "coledoco"},
    "vesicula_biliar": {"vesicula", "vesícula", "vesicula biliar", "vesícula biliar"},
    "pancreas": {"pancreas", "pâncreas"},
    "baco": {"baco", "baço"},
    "rins": {"rim", "rins"},
    "ureteres": {"ureter", "ureteres"},
    "bexiga": {"bexiga"},
    "adrenais": {"adrenais", "supra-renais", "suprarrenais", "suprarrenal"},
    "utero": {"utero", "útero"},
    "ovarios": {"ovario", "ovário", "anexos"},
    "prostata": {"prostata", "próstata"},
    "vasos": {"vasos", "aorta", "veia cava", "porta", "veias hepáticas", "veia porta"},
    "peritonio": {"peritonio", "peritônio"},
    "retroperitonio": {"retroperitonio", "retroperitônio"},
    "pulmao": {"bases pulmonares", "pulmoes", "pulmões", "toracoabdominal"},
    "musculoesqueletico": {"partes moles", "estruturas osseas", "estruturas ósseas"},
    "conclusao": {"conclusao", "conclusão", "impressao", "impressão", "impressao diagnostica", "impressão diagnóstica"},
    "colon_reto": {
        # Termos gerais
        "colon", "cólon",
        # Segmentos do cólon
        "colon ascendente", "colon ascendente", "colon transverso", "colon transverso",
        "colon descendente", "colon descendente", "colon sigmoide", "colon sigmoide", 
        "colon direito", "colon esquerdo",
        # Ceco e apêndice
        "ceco", "apendice", "apêndice", "apendice cecal", "apêndice cecal",
        "apendice retrocecal", "apêndice retrocecal", "apendice vermiforme", "apêndice vermiforme",
        # Sigmoide
        "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
        # Reto
        "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
        # Transição
        "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide", "transição reto-sigmoide",
        "juncao retossigmoide", "junção retossigmoide",
        # Alças e estruturas
        "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon", "alças do cólon",
        "parede colonica", "parede colônica",
    }
}

def segment_by_headers(doc, target_organs: list[str]) -> list[dict]:
    """
    Delimita blocos pelo NOME: do cabeçalho.
    Fecha o bloco no PRÓXIMO cabeçalho (qualquer nome), mesmo que haja texto na mesma linha do cabeçalho.
    """
    text = doc.text
    lines = _line_spans_text(text)

    # 1) capture TODOS os cabeçalhos (nome e posição de início)
    all_headers = []  # [(start_char, header_name_norm)]
    for (s, e, line) in lines:
        m = HEADER_RX.match(line)
        if m:
            name = _norm_header_name(m.group(1))
            all_headers.append((s, name))

    if not all_headers:
        return []

    # 2) quais cabeçalhos são do(s) órgão(s) alvo?
    wanted = set()
    for o in target_organs:
        wanted |= HEADER_ALIASES.get(o, {o})

    organ_headers = [(s, name) for (s, name) in all_headers if any(name.startswith(_norm_header_name(w)) for w in wanted)]

    if not organ_headers:
        return []

    # 3) blocos: [header alvo i, próximo header global)
    all_headers_sorted = sorted(all_headers, key=lambda x: x[0])
    blocks = []
    for s, name in sorted(organ_headers, key=lambda x: x[0]):
        # próximo cabeçalho global após 's'
        next_starts = [S for (S, _) in all_headers_sorted if S > s]
        end = next_starts[0] if next_starts else len(text)
        span = doc.char_span(s, end, alignment_mode="expand")
        if span is not None and (span.end - span.start) > 0:
            # mapeia de volta para o órgão alvo (pega o primeiro que bate)
            mapped = None
            for o in target_organs:
                if any(name.startswith(_norm_header_name(w)) for w in HEADER_ALIASES.get(o, {o})):
                    mapped = o
                    break
            blocks.append({"organ": mapped or target_organs[0], "span": span})
    return blocks

#=======================================================

def fuzzy_sim(a: str, b: str) -> float:
    # similaridade 0..1 (robusto a typos simples)
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

def best_fuzzy_in_vocab(term: str, vocab: list[str], min_ratio: float = 0.80) -> str | None:
    # retorna o termo do vocabulário mais parecido acima do limiar
    best, best_s = None, 0.0
    for v in vocab:
        s = fuzzy_sim(term, v)
        if s > best_s:
            best, best_s = v, s
    return best if best_s >= min_ratio else None

def _resolve_target_organs(target, organs_cfg):
    if target is None:
        return list(organs_cfg.keys())
    if isinstance(target, str):
        target = [target]
    out = []
    for o in target:
        k = o.strip().lower()
        if k not in organs_cfg:
            raise ValueError(f"Órgão desconhecido: {o}. Disponíveis: {list(organs_cfg.keys())}")
        out.append(k)
    return out

# --- Extração de vocabulário candidato do laudo (noun chunks + n-grams leves) ---
def build_doc_candidates(doc, min_len=3) -> list[str]:
    cand = []

    # 1) noun_chunks (como você já fazia)
    for nc in doc.noun_chunks:
        t = norm(nc.text)
        if len(t) >= min_len and not t.isnumeric():
            cand.append(t)

    # 2) n-grams por sentença (1–3), SEM cruzar sentença
    for sent in doc.sents:
        toks = [norm(t.text) for t in sent if t.is_alpha and len(norm(t.text)) >= min_len]
        toks = [t for t in toks if t not in STOP_WORDS_NORM]

        for n in (1, 2, 3):
            for i in range(len(toks) - n + 1):
                cand.append(" ".join(toks[i:i+n]))

    return unique([c for c in cand if len(c) >= min_len])


def _rootize_token(t: str) -> str:
    """
    Rootize leve e genérico (PT-BR) para aumentar recall sem virar stemmer.
    Faz só normalizações de inflexão muito comuns em laudos.

    Regras:
      - remove '-', '_' e '/' dentro do token
      - plurais: oes/ões -> ao ; ais -> al ; eis -> el ; s final
      - (opcional) remove gênero final a/o com guarda
    """
    x = norm(t)
    if not x:
        return ""

    # cola separadores frequentes em laudos (intra-articular, gleno-umeral, subacromial/subdeltoidea)
    x = x.replace("-", "").replace("_", "").replace("/", "")

    # não mexe em tokens curtos
    if len(x) <= 3:
        return x

    # "lesoes"/"alteracoes" -> "lesao"/"alteracao"
    if x.endswith("oes") and len(x) > 4:
        x = x[:-3] + "ao"
    elif x.endswith("ões") and len(x) > 4:  # redundante se norm já tira acento, mas deixa explícito
        x = x[:-3] + "ao"

    # "sinoviais" -> "sinovial", "cervicais" -> "cervical"
    if x.endswith("ais") and len(x) > 4:
        x = x[:-3] + "al"

    # "tendineis" -> "tendinel" (rarinho, mas genérico). Se não quiser, pode tirar.
    if x.endswith("eis") and len(x) > 4:
        x = x[:-3] + "el"

    # plural simples: remove 's'
    if x.endswith("s") and len(x) > 4:
        x = x[:-1]

    # gênero final (leve): inflamatorio/inflamatoria, sacroiliaco/sacroiliaca
    if len(x) > 5 and x.endswith(("a", "o")):
        x = x[:-1]

    return x



# --- Expansão semântica de seeds usando vocabulário do próprio laudo ---
def semantic_expand(seeds: List[str], vocab: List[str],
                    min_sim: float, top_k: int) -> Dict[str, List[Tuple[str, float]]]:
    """
    Expansão semântica baseada no vocabulário do próprio texto.

    Anti-vazamento (MUDANÇA-CHAVE):
      - Exige evidência lexical SEMPRE (overlap por token/root).
      - Remove bypass antigo: "embedding >= X aceita sem overlap".

    Nota:
      - Para seed multi-termo (>=2 tokens), exige overlap >=2 (mais seguro).
      - Permite overlap por raiz leve (plural/hífen/gênero/sufixos comuns) para não matar recall.
    """
    if not seeds or not vocab:
        return {s: [] for s in seeds}

    # seeds normalizadas + tokens
    seed_norm = [norm(s) for s in seeds]
    seed_toks = [token_set(s) for s in seeds]
    seed_roots = [{_rootize_token(t) for t in toks} for toks in seed_toks]

    # pré-filtra vocabulário: mantém só candidato que sobrepõe alguma seed (token OU root)
    kept_vocab: List[str] = []
    kept_norm: List[str] = []
    seed_to_cands: List[List[int]] = [[] for _ in seeds]

    for v in vocab:
        v_n = norm(v)
        if not v_n:
            continue

        v_toks = token_set(v_n)
        if not v_toks:
            continue
        v_roots = {_rootize_token(t) for t in v_toks}

        matched_seeds = []
        for i, s_toks in enumerate(seed_toks):
            if not s_toks:
                continue

            # overlap mínimo depende do tamanho da seed (mais forte p/ multi-termo)
            min_overlap = 2 if len(s_toks) >= 2 else 1

            overlap_tok  = len(v_toks & s_toks)
            overlap_root = len(v_roots & seed_roots[i])

            if max(overlap_tok, overlap_root) >= min_overlap:
                matched_seeds.append(i)

        if not matched_seeds:
            continue

        idx = len(kept_vocab)
        kept_vocab.append(v)
        kept_norm.append(v_n)
        for i in matched_seeds:
            seed_to_cands[i].append(idx)

    if not kept_vocab:
        return {s: [] for s in seeds}

    # embeddings só do vocabulário filtrado
    seed_emb  = embedder.encode(seed_norm,  convert_to_numpy=True, normalize_embeddings=True)
    vocab_emb = embedder.encode(kept_norm, convert_to_numpy=True, normalize_embeddings=True)

    out: Dict[str, List[Tuple[str, float]]] = {s: [] for s in seeds}
    for i, s in enumerate(seeds):
        pairs: List[Tuple[str, float]] = []
        cand_idx_list = seed_to_cands[i]
        if not cand_idx_list:
            continue

        for j in cand_idx_list:
            vtxt = kept_vocab[j]
            emb_sim = float(np.dot(seed_emb[i], vocab_emb[j]))
            fz_sim  = fuzzy_sim(s, vtxt)
            score   = 0.75 * emb_sim + 0.25 * fz_sim
            if score < min_sim:
                continue

            # overlap já foi garantido no pré-filtro; não existe mais bypass por embedding.
            pairs.append((vtxt, score))

        pairs.sort(key=lambda x: x[1], reverse=True)
        out[s] = pairs[:top_k]

    return out


def merge_semantic_sets(seed_list: List[str], sem_expanded: Dict[str, List[Tuple[str,float]]]) -> List[str]:
    terms = list(seed_list)
    for s, pairs in sem_expanded.items():
        for term, sim in pairs:
            if term not in terms:
                terms.append(term)
    return unique(terms)

# --- Segmentação por órgão (ancora por match semântico+regex+seed) ---
def organ_anchors(doc: "spacy.tokens.Doc",
                  organ_name: str,
                  organ_cfg: Dict[str, Any],
                  organ_lexicon: List[str]) -> List[Tuple[int, int]]:
    """
    Âncoras por evidência explícita:
      - seeds do órgão
      - regex do órgão
      - organ_lexicon lexical-only (typos/fuzzy do próprio texto)
    """
    import re
    spans = []

    organ_lexicon = organ_lexicon or []
    organ_terms = set([norm(organ_name)] +
                      [norm(x) for x in organ_cfg.get("seeds", [])] +
                      [norm(x) for x in organ_lexicon])

    for sent in doc.sents:
        s = norm(sent.text)
        hit = False

        for t in organ_terms:
            if not t:
                continue
            pattern = r'\b' + re.escape(t) + r'\b'
            if re.search(pattern, s):
                hit = True
                break

        if not hit and organ_cfg.get("regex"):
            for rx in organ_cfg["regex"]:
                if re.search(rx, s):
                    hit = True
                    break

        if hit:
            spans.append((sent.start, sent.end))

    return spans


def segment_by_organs(doc: "spacy.tokens.Doc",
                      organs_cfg: Dict[str, Any],
                      organ_lexicons: Dict[str, List[str]]) -> List[Dict[str, Any]]:
    """
    Fallback quando não há cabeçalhos:
    cria blocos por sentença-âncora do órgão (seeds/regex + lexicon lexical-only).
    """
    blocks = []
    organ_lexicons = organ_lexicons or {}

    for organ, cfg in organs_cfg.items():
        anchors = organ_anchors(doc, organ, cfg, organ_lexicons.get(organ, []))
        for (a, b) in anchors:
            span = doc[a:b]
            blocks.append({"organ": organ, "span": span})

    return blocks


# --- Não casar termos quando começar com negador ---
NEG_SET = {norm(x) for x in CONFIG["negation"]["tokens"]}

def _starts_with_negator(term: str) -> bool:
    toks = norm(term).split()
    return len(toks) > 0 and toks[0] in NEG_SET

# --- Negação simples baseada em janela ---
def is_negated_in_sentence(doc,
                           start_char: int, end_char: int,
                           neg_list: list[str],
                           window_tokens: int = 12) -> bool:
    """
    True se houver negador NA MESMA SENTENÇA, numa janela:
      - à esquerda do termo
      - à direita do termo

    FIX: não usa substring (ex.: 'sem' NÃO bate em 'semelhante').
    Casa negadores por token e frases multi-termo por n-gram contíguo.
    """
    hit = doc.char_span(start_char, end_char, alignment_mode="expand")
    if hit is None:
        return False

    sent = hit.sent
    sent_tokens = list(sent)

    # --- tokeniza janelas esquerda/direita (por idx) ---
    left = [norm(t.text) for t in sent_tokens if t.idx < hit.start_char]
    right = [norm(t.text) for t in sent_tokens if t.idx >= hit.end_char]

    if window_tokens and window_tokens > 0:
        left = left[-window_tokens:]
        right = right[:window_tokens]

    # --- prepara negadores (1 palavra vs multi-termo) ---
    neg_unigrams: set[str] = set()
    neg_phrases: list[tuple[str, ...]] = []

    for n in (neg_list or []):
        n_norm = norm(n)
        if not n_norm:
            continue
        toks = tuple(t for t in n_norm.split() if t)
        if not toks:
            continue
        if len(toks) == 1:
            neg_unigrams.add(toks[0])
        else:
            neg_phrases.append(toks)

    def _has_phrase(tokens: list[str], phrase: tuple[str, ...]) -> bool:
        k = len(phrase)
        if k == 0 or len(tokens) < k:
            return False
        for i in range(len(tokens) - k + 1):
            if tuple(tokens[i:i+k]) == phrase:
                return True
        return False

    # --- se o próprio hit começar por negador (raro, mas possível) ---
    hit_toks = [norm(t.text) for t in sent_tokens
                if (t.idx >= hit.start_char and t.idx < hit.end_char)]
    if hit_toks:
        if hit_toks[0] in neg_unigrams:
            return True
        for ph in neg_phrases:
            if len(hit_toks) >= len(ph) and tuple(hit_toks[:len(ph)]) == ph:
                return True

    # --- esquerda ---
    if any(t in neg_unigrams for t in left):
        return True
    for ph in neg_phrases:
        if _has_phrase(left, ph):
            return True

    # --- direita ---
    if any(t in neg_unigrams for t in right):
        return True
    for ph in neg_phrases:
        if _has_phrase(right, ph):
            return True

    return False

# --- Matching de achados (léxico expandido) dentro de cada bloco ---
@dataclass
class FindingHit:
    organ: str
    finding: str
    surface: str
    start_char: int
    end_char: int
    negated: bool
    confidence: float

def _sentence_mentions_organ(sent_text: str,
                             organ_name: str,
                             organ_cfg: Dict[str, Any],
                             organ_lexicon: List[str] = None) -> bool:
    """
    True se a sentença menciona o órgão por evidência explícita (seeds/regex)
    + lexicon lexical-only (typos/fuzzy), para manter consistência com o anchor.
    """
    import re
    s = norm(sent_text)

    organ_lexicon = organ_lexicon or []
    organ_terms = set([norm(organ_name)] +
                      [norm(x) for x in organ_cfg.get("seeds", [])] +
                      [norm(x) for x in organ_lexicon])

    for t in organ_terms:
        if not t:
            continue
        pattern = r'\b' + re.escape(t) + r'\b'
        if re.search(pattern, s):
            return True

    if organ_cfg.get("regex"):
        for rx in organ_cfg.get("regex"):
            if re.search(rx, s):
                return True

    return False



def _norm_keep_ws(s: str) -> str:
    """
    Normalização para MATCH que preserva tamanho/índices:
    - lower
    - remove acentos (NFD + remove Mn)
    - NÃO dá strip
    - NÃO colapsa whitespace
    """
    if s is None:
        return ""
    s = s.lower()
    s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
    return s


def _find_organ_mentions_in_text(text: str,
                                organ_name: str,
                                organ_cfg: Dict[str, Any],
                                organ_lexicon: List[str] = None) -> List[Tuple[int, int]]:
    """
    Encontra menções ao órgão em cima de texto normalizado com _norm_keep_ws(),
    para retornar índices compatíveis com o regex usado em match_findings_in_block.
    """
    import re
    mentions: List[Tuple[int, int]] = []

    organ_lexicon = organ_lexicon or []
    text_match = _norm_keep_ws(text)

    organ_terms = set([norm(organ_name)] +
                      [norm(x) for x in organ_cfg.get("seeds", [])] +
                      [norm(x) for x in organ_lexicon])

    # seeds/lexicon
    for term in organ_terms:
        if not term:
            continue
        term_pat = _accent_rx(term).replace(r"\ ", r"\s+")
        pattern = r"(?i)(?<!\w)" + term_pat + r"(?!\w)"
        for m in re.finditer(pattern, text_match):
            mentions.append((m.start(), m.end()))

    # regex do órgão (em cima do texto normalizado)
    if organ_cfg.get("regex"):
        for rx_pattern in organ_cfg.get("regex"):
            for m in re.finditer(rx_pattern, text_match, flags=re.IGNORECASE):
                mentions.append((m.start(), m.end()))

    return mentions



def _finding_is_near_organ_mention(finding_pos: int, organ_mentions: List[Tuple[int, int]], max_distance: int = 200) -> bool:
    """
    Verifica se o finding está próximo (dentro de max_distance caracteres) de alguma menção ao órgão.
    """
    if not organ_mentions:
        return False
    
    for mention_start, mention_end in organ_mentions:
        dist_to_start = abs(finding_pos - mention_start)
        dist_to_end = abs(finding_pos - mention_end)
        min_dist = min(dist_to_start, dist_to_end)
        
        if min_dist <= max_distance:
            return True
    
    return False


def _extract_snippet_around_finding(doc_text: str, start: int, end: int, window: int = 100) -> str:
    """
    Extrai trecho ao redor do finding
    """
    snippet_start = max(0, start - window)
    snippet_end = min(len(doc_text), end + window)
    return doc_text[snippet_start:snippet_end]


def _snippet_mentions_other_organs(snippet: str,
                                  target_organ: str,
                                  all_organs_config: Dict,
                                  finding_start: int,
                                  finding_end: int,
                                  margin_chars: int = 12) -> bool:
    """
    DESAMBIGUAÇÃO REAL (nearest-organ wins), mas mantive o nome pra patch mínimo.

    snippet: texto onde vamos desempatar (recomendado: sp.sent.text)
    finding_start/end: offsets DO FINDING dentro do snippet (0..len(snippet))
    margin_chars: quanto outro órgão precisa ser mais perto que o target pra descartar

    Retorna True se deve DESCARTAR por estar mais próximo de outro órgão.
    """
    import re

    if not snippet:
        return False

    # centro do finding
    f_center = (finding_start + finding_end) // 2

    # guarda a menor distância encontrada para cada órgão
    best_dist_by_organ: Dict[str, int] = {}

    def _add_mention(organ_key: str, m_start: int, m_end: int):
        m_center = (m_start + m_end) // 2
        d = abs(f_center - m_center)
        cur = best_dist_by_organ.get(organ_key)
        if cur is None or d < cur:
            best_dist_by_organ[organ_key] = d

    # 1) varre órgãos e captura menções (seeds + regex)
    for organ_key, organ_cfg in all_organs_config.items():
        # seeds + nome do órgão (chave) não ajuda muito, mas seeds sim
        terms = [organ_key] + list(organ_cfg.get("seeds", []) or [])

        # --- seeds (accent-insensitive + whitespace flexível) ---
        for t in terms:
            t_norm = norm(t)
            if not t_norm:
                continue
            # espaço vira \s+ (pega quebra de linha etc.)
            pat = r'(?i)(?<!\w)' + _accent_rx(t_norm).replace(r"\ ", r"\s+") + r'(?!\w)'
            for m in re.finditer(pat, snippet):
                _add_mention(organ_key, m.start(), m.end())

        # --- regex do órgão ---
        for rx in (organ_cfg.get("regex") or []):
            for m in re.finditer(rx, snippet, flags=re.IGNORECASE):
                _add_mention(organ_key, m.start(), m.end())

    # 2) decisão: se não tem target na sentença, não corta aqui (evita FN)
    d_target = best_dist_by_organ.get(target_organ)
    if d_target is None:
        return False

    # 3) pega o órgão mais próximo (global)
    nearest_organ = min(best_dist_by_organ.items(), key=lambda kv: kv[1])[0]
    d_nearest = best_dist_by_organ[nearest_organ]

    # se o mais perto não é o target, e é *bem* mais perto => descarta
    if nearest_organ != target_organ and (d_nearest + margin_chars) < d_target:
        return True

    return False

def build_organ_lexicon_lexical(seeds: list[str],
                                doc_vocab: list[str],
                                min_ratio: float = 0.84,
                                top_k_per_seed: int = 4) -> list[str]:
    """
    Léxico do órgão SEM embedding:
      - seeds normalizadas (sempre entram)
      - variações/typos detectadas por fuzzy dentro do doc_vocab (top_k por seed)

    Filtra candidatos com:
      - term_is_valid_for_seeds (overlap lexical / raiz leve)
      - não começa com negador
      - evita singletons genéricos via term_is_valid_for_seeds/GENERIC_SINGLETON_BAN
    """
    base_seeds = unique([norm(s) for s in seeds if s and norm(s)])
    if not base_seeds or not doc_vocab:
        return base_seeds

    fuzzy_terms: set[str] = set()

    vocab_norm = [norm(v) for v in doc_vocab if v and norm(v)]

    for seed in seeds:
        seed_n = norm(seed)
        if not seed_n:
            continue

        scored = []
        for v_raw, v_n in zip(doc_vocab, vocab_norm):
            s = SequenceMatcher(None, seed_n, v_n).ratio()
            if s >= min_ratio:
                scored.append((v_raw, s))

        if not scored:
            continue

        scored.sort(key=lambda x: x[1], reverse=True)
        for v_raw, _ in scored[:top_k_per_seed]:
            t = norm(v_raw)
            if (t
                and t not in base_seeds
                and term_is_valid_for_seeds(t, base_seeds)
                and not _starts_with_negator(t)):
                fuzzy_terms.add(t)

    return unique(base_seeds + sorted(fuzzy_terms))


def match_findings_in_block(block: Dict[str,Any],
                            finding_cfg: Dict[str, List[str]],
                            neg_tokens: List[str], neg_window: int,
                            min_sim_seed_vocab: float,
                            organ_cfg: Dict[str, Any] = None,
                            organ_lexicon: List[str] = None,
                            strict_organ_filters: bool = True) -> List[FindingHit]:
    """
    Mantém sua estratégia atual:
      - match em texto normalizado preservando whitespace (_norm_keep_ws)
      - termo com espaço vira \\s+ no regex
      - validações (quando strict_organ_filters=True):
          (A) finding perto de menção explícita do órgão no BLOCO
          (B) sentença do finding menciona o órgão-alvo
          (C) desempate: órgão mais próximo na sentença (corta vazamento)
      - negação por sentença

    Ajuste:
      - organ_mentions é calculado por seeds/regex.
      - Se NÃO houver nenhuma menção explícita (seeds/regex) no bloco,
        usa organ_lexicon (lexical-only) como fallback.
      - Se strict_organ_filters=False (ex.: force_full_doc), não aplica A/B/C.
    """
    span = block["span"]
    text_match = _norm_keep_ws(span.text)  # base de match (preserva WS)
    hits: List[FindingHit] = []
    organ_name = block["organ"]

    local_vocab = build_doc_candidates(span, min_len=CONFIG["semantic"]["min_token_len"])
    neg_list_norm = [norm(x) for x in neg_tokens]

    # ========= menções do órgão no bloco =========
    organ_mentions: List[Tuple[int, int]] = []
    organ_lexicon = organ_lexicon or []

    if organ_cfg is not None:
        # Seeds do órgão + nome canônico (key)
        organ_terms = set([norm(organ_name)] + [norm(x) for x in (organ_cfg.get("seeds", []) or [])])

        # match seeds no MESMO text_match (mesma coordenada do m.start())
        for t in organ_terms:
            if not t:
                continue
            t_pat = _accent_rx(t).replace(r"\ ", r"\s+")
            rx_t = re.compile(r"(?i)(?<!\w)" + t_pat + r"(?!\w)")
            for mm in rx_t.finditer(text_match):
                organ_mentions.append((mm.start(), mm.end()))

        # match regex do órgão (se existir) também no text_match
        for rx_org in (organ_cfg.get("regex", []) or []):
            try:
                for mm in re.finditer(rx_org, text_match, flags=re.IGNORECASE):
                    organ_mentions.append((mm.start(), mm.end()))
            except re.error:
                pass

        # --- PATCH: fallback lexical-only só se NÃO houve menção explícita ---
        organ_mentions_explicit = organ_mentions[:]  # snapshot (seeds+regex)

        if (not organ_mentions_explicit) and organ_lexicon:
            for t in set(norm(x) for x in organ_lexicon if x):
                if not t:
                    continue
                t_pat = _accent_rx(t).replace(r"\ ", r"\s+")
                rx_t = re.compile(r"(?i)(?<!\w)" + t_pat + r"(?!\w)")
                for mm in rx_t.finditer(text_match):
                    organ_mentions.append((mm.start(), mm.end()))
    # ============================================

    for f_can, seeds in finding_cfg.items():
        base_seeds = unique([norm(s) for s in seeds if s and norm(s)])

        sem = semantic_expand(seeds, local_vocab, min_sim_seed_vocab, CONFIG["semantic"]["top_k_per_seed"])

        sem_terms = []
        for _s, pairs in sem.items():
            for term, _sc in pairs:
                t = norm(term)
                if (t
                    and t not in sem_terms
                    and t not in base_seeds
                    and term_is_valid_for_seeds(t, base_seeds)
                    and not _starts_with_negator(t)):
                    sem_terms.append(t)

        fuzzy_terms = set()
        for seed in seeds:
            hit = best_fuzzy_in_vocab(seed, local_vocab, min_ratio=0.86)
            if hit:
                fuzzy_terms.add(norm(hit))

        filtered_fuzzy = [
            t for t in fuzzy_terms
            if t not in base_seeds
            and term_is_valid_for_seeds(t, base_seeds)
            and not _starts_with_negator(t)
        ]

        lexicon = unique(base_seeds + sem_terms + list(filtered_fuzzy))

        for term in lexicon:
            term_pat = _accent_rx(term).replace(r"\ ", r"\s+")
            pat = r"(?i)(?<!\w)" + term_pat + r"(?!\w)"
            rx = re.compile(pat)

            for m in rx.finditer(text_match):
                abs_start = int(span.start_char + m.start())
                abs_end   = int(span.start_char + m.end())

                sp = span.doc.char_span(abs_start, abs_end, alignment_mode="expand")
                if sp is None or sp.sent is None:
                    continue

                # >>>>>>>>>>>>>>> NOVO TRECHO (A/B/C condicional) <<<<<<<<<<<<<<<
                if strict_organ_filters and (organ_cfg is not None):
                    # --- (A) Perto de menção ao órgão no bloco ---
                    finding_pos_in_block = m.start()
                    if not _finding_is_near_organ_mention(finding_pos_in_block, organ_mentions, max_distance=200):
                        continue

                    # --- (B) Sentença do finding menciona o órgão-alvo ---
                    if not _sentence_mentions_organ(sp.sent.text, organ_name, organ_cfg, organ_lexicon):
                        continue

                    # --- (C) Desempate: órgão mais próximo na sentença ---
                    sent = sp.sent
                    rel_start = abs_start - sent.start_char
                    rel_end   = abs_end   - sent.start_char
                    if _snippet_mentions_other_organs(
                        snippet=sent.text,
                        target_organ=organ_name,
                        all_organs_config=CONFIG["organs"],
                        finding_start=int(rel_start),
                        finding_end=int(rel_end),
                        margin_chars=25
                    ):
                        continue
                # <<<<<<<<<<<<<< FIM DO NOVO TRECHO <<<<<<<<<<<<<<<

                # --- negação ---
                neg = is_negated_in_sentence(
                    span.doc,
                    start_char=abs_start,
                    end_char=abs_end,
                    neg_list=neg_list_norm,
                    window_tokens=neg_window
                )

                surface = span.doc.text[abs_start:abs_end]

                hits.append(FindingHit(
                    organ=block["organ"],
                    finding=f_can,
                    surface=surface,
                    start_char=abs_start,
                    end_char=abs_end,
                    negated=neg,
                    confidence=0.9 if not neg else 0.35
                ))

    return hits


#def _force_sentence_breaks(text: str) -> str:
#    """
#    Força quebras de sentença em pontos finais seguidos de maiúscula.
#    """
#    import re
#    # Adiciona quebra de linha após ponto + espaço + maiúscula
#    text = re.sub(r'\.(\s+)([A-Z])', r'.\n\2', text)
#    return text

def _force_sentence_breaks(text: str) -> str:
    # quebra em ". - " (muito comum em laudo)
    text = re.sub(r'\.\s*-\s*', '.\n- ', text)

    # garante quebra antes de bullets em nova linha
    text = re.sub(r'\n\s*-\s*', '\n- ', text)

    # opcional: quebra quando começa outro bloco " - Algo:"
    text = re.sub(r'(\n-\s*[A-Za-zÁ-Úa-zá-ú/ ]{2,}:\s*)', r'\n\1', text)
    return text

# --- Pipeline principal ---
def process_report(text: str, target_organ=None) -> Dict[str, Any]:
    """
    target_organ: None | 'figado' | ['figado','rim',...]
    Executa a análise apenas para o(s) órgão(s) selecionado(s).

    NOVO (lexical-only p/ órgão):
      - NÃO usa semantic_expand / embedding para construir organ_lexicons
      - organ_lexicons = seeds + variações por fuzzy dentro do próprio doc_vocab
    """
    text = to_plain(text)  # já inclui cleanup + remove_final_laudo
    text = _force_sentence_breaks(text)
    doc = nlp(text)

    # 1) resolve órgãos alvo
    target_organs = _resolve_target_organs(
        target_organ if target_organ is not None else TARGET_ORGAN,
        CONFIG["organs"]
    )

    # 2) vocabulário global do documento (para fuzzy lexical-only)
    doc_vocab = build_doc_candidates(doc, min_len=CONFIG["semantic"]["min_token_len"])

    # 3) organ_lexicons lexical-only (seeds + fuzzy do doc_vocab)
    organ_lexicons: Dict[str, List[str]] = {}
    for organ in target_organs:
        ocfg = CONFIG["organs"][organ]
        organ_lexicons[organ] = build_organ_lexicon_lexical(
            seeds=ocfg.get("seeds", []),
            doc_vocab=doc_vocab,
            min_ratio=0.86,
            top_k_per_seed=2
        )

    # 4) segmentação
    organs_subset_cfg = {k: CONFIG["organs"][k] for k in target_organs}

    # NORMALIZAÇÃO do FORCE_FULL_DOC_FOR (aceita string ou set/list)
    _force_set = set()
    if isinstance(FORCE_FULL_DOC_FOR, str) and FORCE_FULL_DOC_FOR.strip():
        _force_set = {norm(FORCE_FULL_DOC_FOR)}
    else:
        try:
            _force_set = {norm(x) for x in (FORCE_FULL_DOC_FOR or set())}
        except TypeError:
            _force_set = set()

    # Força somente se o órgão-alvo estiver no FORCE_FULL_DOC_FOR
    force_full = any(norm(o) in _force_set for o in target_organs)

    if force_full:
        main_organ = next((o for o in target_organs if norm(o) in _force_set), target_organs[0])
        blocks = [{"organ": main_organ, "span": doc[:]}]
    else:
        header_blocks = segment_by_headers(doc, target_organs)
        if header_blocks:
            blocks = header_blocks
        else:
            blocks = segment_by_organs(doc, organs_subset_cfg, organ_lexicons)

    # 5) matching de achados apenas para os órgãos alvo
    all_hits: List[FindingHit] = []
    for b in blocks:
        organ = b["organ"]
        finding_cfg = CONFIG["findings"].get(organ, {})
        if not finding_cfg:
            continue

        hits = match_findings_in_block(
            b, finding_cfg,
            neg_tokens=[norm(x) for x in CONFIG["negation"]["tokens"]],
            neg_window=CONFIG["negation"]["window_tokens"],
            min_sim_seed_vocab=CONFIG["semantic"]["min_sim_seed_vocab"],
            organ_cfg=CONFIG["organs"].get(organ, {}),
            organ_lexicon=organ_lexicons.get(organ, []),
            strict_organ_filters=not force_full
        )
        all_hits.extend(hits)

    findings_dicts = [asdict(h) for h in all_hits]
    result = {
        "organs_requested": target_organs,
        "organs_detected": unique([blk["organ"] for blk in blocks]),
        "findings": findings_dicts,
        "relevant": any(not h.negated for h in all_hits),
    }

    # 6) resumo compacto
    sc = compact_summary_from_hits(findings_dicts, doc, blocks)

    def _is_useful(v):
        if v is None:
            return False
        if isinstance(v, str):
            x = v.strip().lower()
            return x not in {"", "null", "none"}
        return True

    sc_clean = []
    for d in sc:
        clean = {k: v for k, v in d.items() if _is_useful(v)}
        if clean:
            sc_clean.append(clean)

    result["summary_compact"] = sc_clean
    return result


def process_reports_batch(texts: list[str], target_organ=None, desc: str = "Processando laudos"):
    try:
        from tqdm import tqdm
        it = tqdm(texts, desc=desc)
    except Exception:
        it = texts
    return [process_report(txt, target_organ=target_organ) for txt in it]


def print_result(res, mode="full"):
    if mode == "summary":
        import json
        print(json.dumps(res.get("summary_compact", []), ensure_ascii=False, indent=2))
        return

    print(f"organs_requested: {res.get('organs_requested')}")
    print(f"organs_detected:  {res.get('organs_detected')}")
    print(f"relevant:         {res.get('relevant')}\n")
    print("findings:")
    for h in res.get("findings", []):
        print(f"- organ: {h['organ']} | finding: {h['finding']} | surface: \"{h['surface']}\" "
              f"| start: {h['start_char']:>3} | end: {h['end_char']:>3} | negated: {h['negated']} | conf: {h['confidence']}")
        
def extrair_nao_nulos(resumo):
    """
    resumo: pode ser list[dict], dict, None ou lixo.
    Retorna list[dict] só com {chave: texto} onde texto é não-nulo.
    """
    out = []

    def _valido(v):
        if v is None:
            return False
        if isinstance(v, float) and pd.isna(v):
            return False
        s = str(v).strip()
        if s == "" or s.lower() == "null":
            return False
        return True

    if isinstance(resumo, list):
        for item in resumo:
            if isinstance(item, dict):
                for k, v in item.items():
                    if _valido(v):
                        out.append({k: v})
    elif isinstance(resumo, dict):
        for k, v in resumo.items():
            if _valido(v):
                out.append({k: v})
    # qualquer outra coisa -> lista vazia
    return out

def extract_keys_unique(dict_list):
    seen = set(); out = []
    for d in dict_list:
        if isinstance(d, dict) and d:
            k = next(iter(d))
            if k not in seen:
                seen.add(k); out.append(k)
    return out

In [0]:
# # --- Utilitários ---
# def norm(s: str) -> str:
#     s = s.lower().strip()
#     s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
#     s = re.sub(r"\s+", " ", s)
#     return s

# def unique(seq):
#     seen=set(); out=[]
#     for x in seq:
#         if x not in seen:
#             out.append(x); seen.add(x)
#     return out

# #------- ALTERAÇÕES PARA HTML -------
# _HTML_BRK_TAGS = re.compile(r'</?(p|div|br|li|tr|h[1-6])\b[^>]*>', re.IGNORECASE)
# _HTML_TAGS     = re.compile(r'<[^>]+>')
# _HTML_SCRIPT   = re.compile(r'<(script|style)\b[^>]*>.*?</\1\s*>', re.IGNORECASE|re.DOTALL)
# _HTML_DATAIMG  = re.compile(r'data:image/[^;]+;base64,[A-Za-z0-9+/=\s]+', re.IGNORECASE)
# _HTML_JUNK_ATTR= re.compile(r'\sdata-[\w\-]+="[^"]*"')
# _HTML_JUNK_CLS = re.compile(r'\sclass="[^"]*"')
# _HTML_WIDGET   = re.compile(r'\s(?:aria-[\w\-]+|role|tabindex|contenteditable|spellcheck|draggable|style|width|height)="[^"]*"', re.IGNORECASE)

# def _looks_like_html(s: str) -> bool:
#     head = s[:500]
#     return bool(re.search(r'</?(html|head|body|div|p|span|br|img|script|style|table)\b', head, re.IGNORECASE))

# def html_to_plain(s: str) -> str:
#     if not s: 
#         return ""
#     # normaliza EOL
#     s = s.replace("\r\n","\n").replace("\r","\n")

#     # remove blocos <script>/<style>
#     s = _HTML_SCRIPT.sub('\n', s)

#     # tira atributos inúteis pesados antes de remover tags (menos agressivo que strip global)
#     s = _HTML_JUNK_ATTR.sub('', s)
#     s = _HTML_WIDGET.sub('', s)
#     s = _HTML_JUNK_CLS.sub('', s)

#     # remove data URIs
#     s = _HTML_DATAIMG.sub('', s)

#     # converte <br>, </p>, </div>, </li>, </tr>, <h*> em quebras
#     s = _HTML_BRK_TAGS.sub('\n', s)

#     # remove o restante das tags
#     s = _HTML_TAGS.sub('', s)

#     # decodifica entidades (&nbsp;, &ccedil;, etc.)
#     s = _html.unescape(s)

#     # limpa invisíveis e normaliza unicode
#     s = s.replace("\u200b","").replace("\u200c","").replace("\xa0"," ")
#     s = unicodedata.normalize("NFKC", s)

#     # colagens comuns do editor
#     # ex: “ /  ” múltiplos espaços & separadores
#     s = re.sub(r'[ \t]{2,}', ' ', s)
#     s = re.sub(r' ?/ ?', ' / ', s)
#     s = re.sub(r'\n[ \t]+', '\n', s)

#     # linhas em branco excessivas
#     s = re.sub(r'\n{3,}', '\n\n', s)

#     # strip final por linha e geral
#     lines = [ln.rstrip() for ln in s.splitlines()]
#     s = "\n".join(lines).strip()

#     # correções pontuais vistas no corpus Tasy/CKE
#     # “Hospita​l” (ZWSP), “Colonosocopia”, “redundancia” -> acentos
#     s = s.replace("Hospita\u200bl", "Hospital")
#     s = re.sub(r'\bColonosocopia\b', 'Colonoscopia', s, flags=re.IGNORECASE)
#     s = re.sub(r'\bredundancia\b', 'redundância', s, flags=re.IGNORECASE)

#     return s


# # --- NOVO: limpeza leve de ruído pós-extração ---
# def _cleanup_text_noise(text: str) -> str:
#     if not text:
#         return ""

#     # invisíveis comuns + normalização
#     text = (text.replace("\u200b", "")
#                 .replace("\u200c", "")
#                 .replace("\xa0", " "))
#     text = unicodedata.normalize("NFKC", text)

#     # 1) Normalizar espaçamento em barras quando há caractere dos dois lados
#     text = re.sub(r'(?<=\w)\s*/\s*(?=\w)', '/', text, flags=re.UNICODE)

#     # 2) Garantir espaço entre número e letra/unidade (não mexe nas barras)
#     text = re.sub(r'(?<=\d)(?=[A-Za-zµμ°])', ' ', text)

#     # 3) Compactar múltiplos espaços e aparar espaços antes/depois de \n
#     text = re.sub(r'[ \t]+', ' ', text)
#     text = re.sub(r'\s*\n\s*', '\n', text)

#     # 4) Limpar linhas em branco repetidas
#     text = re.sub(r'\n{3,}', '\n\n', text)

#     # 5) Trim por linha e global
#     lines = [ln.rstrip() for ln in text.splitlines()]
#     text = "\n".join(lines).strip()
#     return text


# def _rtf_to_text_fallback(rtf: str, debug: bool = False) -> str:
#     if not rtf:
#         return ""

#     import re, unicodedata

#     # ---- helpers locais ----
#     def _undo_py_escapes(s: str) -> str:
#         # reverte escapes que o Python injeta quando a string não é raw
#         return (s
#             .replace("\x0c", r"\f")   # form feed -> \f
#             .replace("\t",  r"\tab")  # tab -> \tab (token RTF real)
#         )

#     def _strip_rtf_metadata_top(s: str) -> str:
#         # Remove SOMENTE blocos padrão logo no topo (primeiros ~8 KB)
#         head = s[:8192]
#         tail = s[8192:]

#         # cada bloco: remove no máximo 1 ocorrência por padrão
#         patterns = [
#             r"{\\\*?\\fonttbl\b[^{}]*}",
#             r"{\\\*?\\colortbl\b[^{}]*}",
#             r"{\\\*?\\stylesheet\b[^{}]*}",
#             r"{\\\*?\\listtable\b[^{}]*}",
#             r"{\\\*?\\listoverridetable\b[^{}]*}",
#             r"{\\\*?\\generator\b[^{}]*}",
#             r"{\\\*?\\info\b[^{}]*}",
#         ]
#         for pat in patterns:
#             head = re.sub(pat, " ", head, flags=re.IGNORECASE)

#         return head + tail

#     def _drop_rtf_meta_lines(text: str) -> str:
#         out = []
#         only_words_semicolon_rx = re.compile(r'^[A-Za-z0-9 /+\-\*\u00C0-\u017F]+;\s*$')
#         font_tokens = r"(unicode|opensymbol|wingdings|monospaced|serif|sans|arial|calibri|times|courier)"
#         # aceita " * " OU "\* " antes do nome da fonte
#         font_line_rx = re.compile(
#             rf'^\s*[A-Za-z0-9 \-\u00C0-\u017F]+(?:\s\\?\*\s*)?(?:{font_tokens})(?:\s*[;:]?)\s*$', re.IGNORECASE)
#         tiny_meta_rx = re.compile(r'^\s*(default|\*jword2|\* generator|\* info)\s*[;:]?\s*$', re.IGNORECASE)

#         for ln in text.splitlines():
#             lns = ln.strip()
#             if not lns:
#                 out.append(ln); continue
#             if only_words_semicolon_rx.match(lns):
#                 continue
#             if font_line_rx.match(lns):
#                 continue
#             if tiny_meta_rx.match(lns):
#                 continue
#             # fallback bem específico: linha curta com qualquer token de fonte + ';'
#             if ';' in lns and re.search(font_tokens, lns, re.IGNORECASE) and len(lns) <= 80:
#                 continue
#             out.append(ln)
#         return "\n".join(out).strip()

#     # -------------------------

#     s = _undo_py_escapes(rtf)
#     # normaliza EOL cedo
#     s = s.replace("\r\n", "\n").replace("\r", "\n")

#     # pega só se realmente parece RTF
#     if not s.lstrip().startswith("{\\rtf"):
#         # não é RTF: devolve como veio
#         return s.strip()

#     # tira metadados do topo de forma conservadora
#     s = _strip_rtf_metadata_top(s)

#     if debug:
#         print("--- after strip_top ---")
#         print(s[:1000])

#     # 1) CP1252: \'hh
#     def _hex_sub(m):
#         try:
#             return bytes([int(m.group(1), 16)]).decode('cp1252', errors='ignore')
#         except Exception:
#             return ""
#     s = re.sub(r"\\'([0-9a-fA-F]{2})", _hex_sub, s)

#     # 2) Unicode: \uNNNN?
#     def _u_sub(m):
#         try:
#             code = int(m.group(1))
#             if code < 0:
#                 code += 65536
#             return chr(code)
#         except Exception:
#             return ""
#     s = re.sub(r"\\u(-?\d+)\??", _u_sub, s)

#     # 3) Quebras e tabs
#     s = re.sub(r"\\par[d]?\b", "\n", s, flags=re.IGNORECASE)
#     s = re.sub(r"\\line\b", "\n", s, flags=re.IGNORECASE)
#     s = re.sub(r"\\tab\b", "\t", s, flags=re.IGNORECASE)

#     # 4) Remove destinos conhecidos (restantes, pontuais)
#     s = re.sub(r"{\\\*?\\(fonttbl|colortbl|stylesheet|info|generator|listtable|listoverridetable)[^}]*}",
#                " ", s, flags=re.IGNORECASE)

#     # 5) Remover comandos de controle (apenas os que começam com barra)
#     #    Mais conservador: exige backslash + letras (com opcional número)
#     #s = re.sub(r"\\[a-zA-Z]+-?\d*(?:\s|;)?", " ", s)
#     s = re.sub(r"\\[a-zA-Z]+-?\d*\s?", " ", s)

#     # 6) Escapes literais e chaves
#     s = s.replace("\\{", "{").replace("\\}", "}").replace("\\\\", "\\")
#     s = re.sub(r"[{}]", " ", s)  # neste ponto já decodificamos conteúdo útil

#     # 7) Plano B: 'hh sem barra (alguns dumps)
#     s = re.sub(r"(?<!\\)'([0-9a-fA-F]{2})", _hex_sub, s)

#     # 8) Limpa tokens RTF residuais comuns (bem restritivo)
#     s = re.sub(r"\b(?:rtf\d*|ansi|ansicpg\d+|deftab\d+|paper[wh]\d+|marg[tlrb]\d+|headery\d+|footery\d+|colsx?\d+|snext\d+|fs\d+|cf\d+|pard|plain|qc|ql|itap\d+|viewkind\d+|uc\d+|sa\d+|sl\d+|slmult\d+|lang\d+|kerning\d+|ulnone|b0|i0|f\d+|fs\d+|cf\d+|kerning\d+|deff\d+|colortbl|fonttbl|stylesheet|info|generator|listtable|listoverridetable)\b",
#            " ", s, flags=re.IGNORECASE)

#     # 9) Normalização
#     s = unicodedata.normalize("NFKC", s)
#     # remove invisíveis
#     s = s.replace("\u200b", "").replace("\u200c", "").replace("\xa0", " ")
#     # espaços
#     s = re.sub(r"[ \t]+", " ", s)
#     s = re.sub(r"\s*\n\s*", "\n", s)
#     s = re.sub(r"\n{3,}", "\n\n", s).strip()

#     # 10) Derruba linhas-meta finais
#     s = _drop_rtf_meta_lines(s)

#     # Garantia mínima: se sobrar vazio, devolve original “bruto” decodificado por CP1252
#     if not s:
#         s = bytes(rtf, "latin1", errors="ignore").decode("cp1252", errors="ignore").strip()

#     return s


# def remove_final_laudo(texto: str) -> str:
#     if not texto:
#         return ""

#     # --- REMOÇÕES GLOBAIS INLINE (apareçam onde aparecerem) ---

#     # 1) "Laudo Gerado/Lib[e]rado por Sistema Especialista."
#     texto = re.sub(r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?', ' ', texto)

#     # 2) "Para visualização do conteúdo do laudo, favor/… acessar/acesse … Viewer/opção imagem/imagem disponível"
#     texto = re.sub(r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.*?\b(?:favor\s+)?(?:acesse|acessar)\b.*?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?', ' ', texto)

#     # 3) "Laudo pode não estar completo na visualização em RTF/Imagem"
#     texto = re.sub(r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?', ' ', texto)

#     # 4) "Referências: - Endoscopic Classification Review Group..." (e variações)
#     # Busca por "referencia(s)" e remove o restante do texto.
#     texto = re.sub(r'(?i)\brefer[eê]ncia[s]?\s*[:\-\.]?.*$', ' ', texto, flags=re.DOTALL)

#     # --- REMOÇÃO POR SENTENÇA (linhas inteiras de aviso/rodapé) ---
#     pats_norm = [
#         r"laudo\s+pode\s+nao\s+estar\s+completo.*?\b(rtf|imagem)\b",
#         r"para\s+visualiza\w+.*?\b(rtf|imagem|sistema|webris|viewer)\b",
#         r"acessar\s+a\s+op[cç][aã]o\s+imagem",
#         r"sistema\s+especialista",
#         r"sistema\s+da\s+radiologia\s+webris",
#         # novas combinações explícitas:
#         r"este\s+laudo\s+foi\s+liberado\s+por\s+um?\s+sistema\s+especialista",
#         r"favor\s+acessar\s+a\s+op[cç][aã]o\s+imagem\s+dispon[ií]vel",
#         r"visualiza\w+\s+do\s+conteudo\s+do\s+laudo",
#         r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+rtf",
#     ]
#     rx = re.compile(r"^(?:\s*(?:{})(?:[.!?])?\s*)$".format("|".join(pats_norm)), re.IGNORECASE)

#     sents = re.split(r'(?<=[.!?])\s+|\n+', texto.strip())
#     kept = []
#     for s in sents:
#         s_norm = norm(s)
#         if not rx.match(s_norm):
#             kept.append(s)

#     out = " ".join(kept)
#     out = re.sub(r'\s{2,}', ' ', out)
#     out = re.sub(r'\s+([.!?,;:])', r'\1', out)
#     return out.strip()



# # --- Pós-processamento PT-BR para laudos (leve e seguro) ---
# _UPPER_SPACED_RX = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)

# def _despace_allcaps_header(line: str) -> str:
#     # "C O N C L U S Ã O" -> "CONCLUSÃO"
#     if _UPPER_SPACED_RX.match(line.strip()):
#         return line.replace(" ", "")
#     return line

# def _titlecase_known_headers(line: str) -> str:
#     # Padroniza cabeçalhos comuns
#     l = line.strip()
#     if l in {"CONCLUSAO", "CONCLUSÃO"}:
#         return "Conclusão"
#     return line

# def _fix_common_ocr_pt(text: str) -> str:
#     """
#     Correções genéricas de OCR:
#       - Colapsa CAPS espaçados (linha inteira e inline): "C O N C L U S A O" -> "CONCLUSAO"
#       - Remove dígito de prefixo em tokens do tipo 1-2 dígitos + palavra (ex.: "6reto" -> "reto"),
#         com salvaguardas para não mexer em unidades/dosagens (mg, ml, mm, cm, kg, bpm, mmHg, l/min etc.)
#       - Corrige espaços intra-palavra (ex.: "seda ção" -> "sedação", "p ó s" -> "pós", "aux ílio" -> "auxílio")
#       - Higiene de espaços/linhas ao final
#     """
#     text = unicodedata.normalize("NFKC", text)

#     # --- (A) Colapsar CAPS espaçados (genérico) ---
#     RX_SPACED_CAPS_LINE = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
#     RX_SPACED_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)

#     def _collapse_caps_spaces(s: str) -> str:
#         if RX_SPACED_CAPS_LINE.match(s.strip()):
#             return s.replace(" ", "")
#         return RX_SPACED_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), s)

#     lines = text.splitlines()
#     lines = [_collapse_caps_spaces(ln) for ln in lines]
#     text = "\n".join(lines)

#     # --- (B) Remover dígito de prefixo OCR (genérico) ---
#     UNITS = {
#         "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
#         "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
#         "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
#     }
#     RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
#     RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
#     RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')

#     def _strip_digit_prefix_token(m: re.Match) -> str:
#         token = f"{m.group(1)}{m.group(2)}"
#         if RX_DOSAGE.match(token) or RX_FLOW.match(token) or m.group(2).lower() in UNITS:
#             return token
#         return m.group(2)

#     text = RX_OCR_PREFIX.sub(_strip_digit_prefix_token, text)

#     # --- (C) Higiene básica de espaços e quebras ---
#     text = re.sub(r"[ \t]{2,}", " ", text)
#     text = re.sub(r"\s*\n\s*", "\n", text)
#     text = re.sub(r"\n{3,}", "\n\n", text).strip()

#     # --- (D) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
#     # Iteramos até estabilizar para capturar padrões encadeados.
#     ACCENTS = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
#     for _ in range(5):
#         before = text

#         # D1) "á gua" -> "água", "ç ões" -> "ções", etc.
#         text = re.sub(rf'(?i)\b([{ACCENTS}])\s+([A-Za-z]{{1,20}})\b', r'\1\2', text)

#         # D2) "aux ílio" -> "auxílio" (letra(s) + espaço + letra acentuada + resto)
#         text = re.sub(rf'(?i)\b([A-Za-z]{{1,20}})\s+([{ACCENTS}][A-Za-z]{{1,20}})\b', r'\1\2', text)

#         # D3) "p ó s" -> "pós" e similares (trinca com acento no meio)
#         text = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACCENTS}])\s+([A-Za-z])\b', r'\1\2\3', text)

#         # D4) Sufixos muito comuns
#         text = re.sub(r'(?i)çã\s+o\b',  'ção',  text)
#         text = re.sub(r'(?i)çõ\s+es\b', 'ções', text)
#         text = re.sub(r'(?i)ã\s+o\b',   'ão',   text)
#         text = re.sub(r'(?i)õ\s+es\b',  'ões',  text)
#         text = re.sub(r'(?i)ó\s+pico(s)?\b', r'ópico\1', text)
#         text = re.sub(r'(?i)á\s+sico(s)?\b', r'ásico\1', text)

#         # D5) Hífen com espaços indevidos: "p ó s - procedimento" -> "pós-procedimento"
#         text = re.sub(rf'(?i)\b([A-Za-z{ACCENTS}]{{1,30}})\s*-\s*([A-Za-z{ACCENTS}]{{1,30}})\b', r'\1-\2', text)

#         if text == before:
#             break

#     # --- (E) Higiene final ---
#     text = re.sub(r"[ \t]{2,}", " ", text)
#     text = re.sub(r"\s+\n", "\n", text)
#     text = re.sub(r"\n{3,}", "\n\n", text).strip()
#     return text



# # --- Heurísticas genéricas de limpeza pós-extração (valem para RTF/HTML/texto cru) ---

# _FONT_WORDS = {
#     "times", "times new roman", "arial", "calibri", "cambria", "courier", "courier new",
#     "helvetica", "symbol", "wingdings", "opensymbol", "ms mincho", "simsun", "century",
#     "cambria math"
# }

# def _drop_boilerplate_lines(text: str) -> str:
#     out = []
#     for ln in text.splitlines():
#         # --- REMOÇÕES INLINE DE BOILERPLATE (não dependem de linha inteira) ---

#         # 1) Variações de "Laudo Gerado/Lib[e]rado por Sistema Especialista"
#         ln = re.sub(r'(?i)\blaudo\s+(?:gerado|liberado)\s+por\s+um?\s*sistema\s+especialista\b\.?', ' ', ln)

#         # 2) Variações de "Para Visualizar/Visualização do Laudo Acesse/Acessar o Viewer/Opção Imagem"
#         ln = re.sub(r'(?i)\bpara\s+visualiza\w*\s+(?:o\s+conte[uú]do\s+do\s+)?laudo\b.*?\b(?:acesse|acessar)\b.*?\b(?:viewer|op[cç][aã]o\s+imagem|imagem\s+dispon[ií]vel)\b\.?', ' ', ln)

#         # 3) Variações de "Laudo pode não estar completo na visualização em RTF/Imagem"
#         ln = re.sub(r'(?i)\blaudo\s+pode\s+na[oã]\s+estar\s+completo\s+na\s+visualiza\w+\s+em\s+(?:rtf|imagem)\b\.?', ' ', ln)

#         raw = ln
#         s = ln.strip()
#         sl = s.lower()

#         if not s:
#             out.append(raw); continue

#         # 1) Linhas com muitos ';' (lista de fontes/estilos/cores)
#         if s.count(';') >= 3:
#             hits = sum(1 for w in _FONT_WORDS if w in sl)
#             if hits >= 1 or re.search(r'\b(font|heading|colortbl|stylesheet|style)\b', sl):
#                 continue
#             if re.fullmatch(r'(?:[^;]{1,40};\s*){5,}', s):
#                 continue

#         # 2) Cabeçalhos estilo "heading 1; heading 2; ..."
#         if re.fullmatch(r'(?:heading\s*\d+\s*;\s*){2,}heading\s*\d+\s*;?', sl):
#             continue

#         # 3) Geradores e metadados
#         if re.search(r'(created by|generator|html\s*to\s*rtf|jword)', sl):
#             continue

#         # 4) Apenas pontuação/ruído
#         if re.fullmatch(r'[;,\.\-–—\s]+', s):
#             continue

#         # 5) Avisos genéricos (texto normalizado, acento-insensível)
#         ns = norm(s)
#         if ("laudo" in ns and "visualiza" in ns and ("rtf" in ns or "imagem" in ns or "viewer" in ns or "sistema" in ns or "especialista" in ns)):
#             continue
#         if re.search(r"laudo\s+pode\s+nao\s+estar\s+completo\s+na\s+visualiza\w+\s+em\b", ns):
#             continue

#         # 6) Linhas curtas com 2+ nomes de fontes + ';'
#         if ';' in s and sum(1 for w in _FONT_WORDS if w in sl) >= 2 and len(s) <= 160:
#             continue

#         out.append(raw)

#     return "\n".join(out).strip()


# def _final_punct_cleanup(text: str) -> str:
#     if not text:
#         return ""
#     # espaço antes de pontuação
#     text = re.sub(r'\s+([.,;:])', r'\1', text)
#     # espaço depois de abertura de parênteses/colchetes e antes de fechamento
#     text = re.sub(r'([\(\[\{])\s+', r'\1', text)
#     text = re.sub(r'\s+([\)\]\}])', r'\1', text)
#     # múltiplas pontuações -> uma
#     text = re.sub(r'([.,;:]){2,}', r'\1', text)
#     # linhas em branco extras
#     text = re.sub(r'\n{3,}', '\n\n', text)
#     # trims por linha
#     lines = [ln.rstrip() for ln in text.splitlines()]
#     return "\n".join(lines).strip()



# def to_plain(s: str) -> str:
#     """
#     Conversão robusta (RTF/HTML/texto) -> texto limpo:
#       1) detecta RTF/HTML e usa parser dedicado (striprtf / lxml / html2text)
#       2) ftfy para arrumar unicode/mojibake
#       3) normalizações e correções de espaços intra-palavra/acento
#       4) limpeza de boilerplate típico de RTF/HTML
#       5) remoção de rodapés conhecidos e ajuste de pontuação
#     """
#     if not s:
#         return ""

#     raw = s.replace("\r\n", "\n").replace("\r", "\n")

#     # --- 1) Detectar formato ---
#     head = raw[:512].lstrip()
#     is_rtf  = head.startswith("{\\rtf") or "\\rtf" in head.lower()[:128]
#     is_html = bool(re.search(r"</?(html|head|body|div|p|span|br|table|tr|td|img)\b", raw[:2000], re.I))

#     txt = raw
#     try:
#         if is_rtf:
#             try:
#                 # Tenta usar a biblioteca externa (rtf_to_text)
#                 txt = rtf_to_text(raw)
#             except Exception:
#                 # Fallback para a função interna se a biblioteca falhar
#                 txt = _rtf_to_text_fallback(raw)
#         elif is_html:
#             # ... (lógica HTML)
#             pass
#         else:
#             txt = raw
#     except Exception:
#         # fallback bruto caso algo quebre
#         txt = raw

#     # --- 2) Unicode/encodes (corrige mojibake, NBSP etc.) ---
#     try:
#         txt = ftfy.fix_text(txt)
#     except Exception:
#         pass

#     # invisíveis e normalização
#     txt = (txt.replace("\u200b", "")
#               .replace("\u200c", "")
#               .replace("\xa0", " "))
#     txt = unicodedata.normalize("NFKC", txt)

#     # --- 3) Colapsar cabeçalhos ALL CAPS espaçados (CON C LU S Ã O) ---
#     RX_CAPS_LINE   = re.compile(r'^(?:[A-ZÁ-Ú]\s+){2,}[A-ZÁ-Ú]$', re.UNICODE)
#     RX_CAPS_INLINE = re.compile(r'(?<!\w)([A-ZÁ-Ú](?:\s+[A-ZÁ-Ú]){2,})(?!\w)', re.UNICODE)
#     def _collapse_caps(line: str) -> str:
#         if RX_CAPS_LINE.match(line.strip()):
#             return line.replace(" ", "")
#         return RX_CAPS_INLINE.sub(lambda m: m.group(1).replace(" ", ""), line)
#     txt = "\n".join(_collapse_caps(ln) for ln in txt.splitlines())

#     # --- 4) Remover prefixo numérico de OCR grudado (ex.: "6reto" -> "reto"), sem afetar dosagens ---
#     UNITS = {
#         "mg","g","kg","mcg","ug","μg","ml","l","dl","cl","mm","cm","m","km",
#         "bpm","mmhg","pa","ua","u/l","ui","iu","u","meq","mol","mmol","nmol",
#         "na","k","cl","co2","o2","sat","spo2","fio2","cr","vdrl","ph"
#     }
#     RX_DOSAGE = re.compile(r'(?i)^\d{1,3}\s*(?:' + r'|'.join(sorted(UNITS, key=len, reverse=True)) + r')\b')
#     RX_FLOW   = re.compile(r'(?i)^\d{1,3}\s*[lL]\s*/\s*min\b')
#     RX_OCR_PREFIX = re.compile(r'(?i)\b(\d{1,2})([A-Za-zÁ-Úá-ú]{3,})\b')
#     def _strip_digit_prefix(m: re.Match) -> str:
#         tok = m.group(0)
#         if RX_DOSAGE.match(tok) or RX_FLOW.match(tok) or m.group(2).lower() in UNITS:
#             return tok
#         return m.group(2)
#     txt = RX_OCR_PREFIX.sub(_strip_digit_prefix, txt)

#     # --- 5) Corrigir espaços intra-palavra (acentos/ç quebrados) ---
#     ACC = "ÁÉÍÓÚÂÊÎÔÛÃÕÄËÏÖÜáéíóúâêîôûãõäëïöüÇç"
#     for _ in range(4):
#         before = txt

#         # letras acentuadas separadas do resto
#         txt = re.sub(rf'(?i)\b([{ACC}])\s+([A-Za-z]{{1,30}})\b', r'\1\2', txt)
#         txt = re.sub(rf'(?i)\b([A-Za-z]{{1,30}})\s+([{ACC}][A-Za-z]{{0,30}})\b', r'\1\2', txt)

#         # trinca com acento no meio: "p ó s" -> "pós"
#         txt = re.sub(rf'(?i)\b([A-Za-z])\s+([{ACC}])\s+([A-Za-z])\b', r'\1\2\3', txt)

#         # sufixos comuns
#         txt = re.sub(r'(?i)çã\s+o\b',  'ção',  txt)
#         txt = re.sub(r'(?i)çõ\s+es\b', 'ções', txt)
#         txt = re.sub(r'(?i)ã\s+o\b',   'ão',   txt)
#         txt = re.sub(r'(?i)õ\s+es\b',  'ões',  txt)

#         # hífen com espaços indevidos
#         txt = re.sub(rf'(?i)\b([A-Za-z{ACC}]{{1,40}})\s*-\s*([A-Za-z{ACC}]{{1,40}})\b', r'\1-\2', txt)

#         # casos de espaço antes de letras acentuadas dentro de palavra (cola sempre)
#         txt = re.sub(rf'([A-Za-z])\s+(?=[{ACC}])', r'\1', txt)
#         txt = re.sub(rf'([{ACC}])\s+(?=[A-Za-z])', r'\1', txt)

#         if txt == before:
#             break

#     # --- 6) Limpeza de boilerplate herdado de RTF/HTML ---
#     lines = []
#     for ln in txt.splitlines():
#         s = ln.strip()
#         sl = s.lower()
#         if not s:
#             lines.append(ln); continue
#         # lixos clássicos de RTF/HTML
#         if any(tok in sl for tok in ("fonttbl","colortbl","stylesheet","generator",
#                                      "heading","listoverridetable","listtable")):
#             continue
#         if s.count(';') >= 2 and re.search(r'(arial|times|calibri|courier|symbol|wingdings)', sl):
#             continue
#         # linhas só de pontuação/traços
#         if re.fullmatch(r'[;,\.\-–—\s]+', s):
#             continue
#         lines.append(ln)
#     txt = "\n".join(lines)

#     # --- 7) Espaços/pontuação finais e quebras ---
#     txt = re.sub(r'[ \t]+', ' ', txt)
#     txt = re.sub(r'\s*\n\s*', '\n', txt)
#     txt = re.sub(r'\n{3,}', '\n\n', txt)
#     txt = re.sub(r'\s+([.,;:])', r'\1', txt)
#     txt = re.sub(r'([\(\[\{])\s+', r'\1', txt)
#     txt = re.sub(r'\s+([\)\]\}])', r'\1', txt)

#     # --- 8) Remover rodapés/avisos de visualização incompleta e limpar pontas ---
#     txt = remove_final_laudo(txt).strip()
#     return txt



# #“Pretty name” do achado (tira underscores)
# def _pretty_finding_name(f: str) -> str:
#     return f.replace("_", " ").strip()

# def _any_q_in_line(line_norm: str, queries_norm: set[str]) -> bool:
#     # casa por token, não só por substring exata
#     if not queries_norm:
#         return False
#     for q in queries_norm:
#         if not q: 
#             continue
#         # aceita q inteiro ou seus tokens (ex.: "nodulo hipervascular" -> "nodulo" OU "hipervascular")
#         toks = [t for t in q.split() if len(t) >= 4] or [q]
#         if any(t in line_norm for t in toks):
#             return True
#     return False

# def _shorten_clause(line_raw: str, query_norm: str, max_chars: int = 160) -> str:
#     """
#     Devolve o menor trecho útil dentro da linha que contém a query:
#     - tenta cortar por pontuação (. ; : – ) ( )
#     - se ainda ficar longo, usa uma janela em torno do match
#     """
#     ln = line_raw.strip()
#     if len(ln) <= max_chars:
#         return ln

#     # 1) tenta quebrar por cláusulas
#     parts = re.split(r'([.;:–\-\(\)])', ln)  # mantém separadores
#     # remonta pares (texto+sep) para preservar sentenças curtas
#     chunks = []
#     cur = ""
#     for p in parts:
#         cur += p
#         if p in ".;:–-)" and cur.strip():
#             chunks.append(cur.strip())
#             cur = ""
#     if cur.strip():
#         chunks.append(cur.strip())

#     # procura a cláusula que contém a query (normalizada)
#     q = norm(query_norm)
#     for c in chunks:
#         if q and norm(c).find(q) != -1:
#             return c if len(c) <= max_chars else c[:max_chars].rsplit(" ", 1)[0] + "…"

#     # 2) janela em torno do match (acentos-insensível)
#     pat = _accent_rx(q) if q else None
#     if pat:
#         m = re.search(pat, ln, flags=re.IGNORECASE)
#         if m:
#             left = max(0, m.start() - max_chars//2)
#             right = min(len(ln), m.end() + max_chars//2)
#             snippet = ln[left:right].strip()
#             # aparar nas bordas para próximo espaço
#             if left > 0:
#                 snippet = "…" + snippet.split(" ", 1)[-1]
#             if right < len(ln):
#                 snippet = snippet.rsplit(" ", 1)[0] + "…"
#             return snippet

#     # fallback
#     return (ln[:max_chars].rsplit(" ", 1)[0] + "…") if len(ln) > max_chars else ln

# #Compactador (escolhe 1 frase por achado, preferindo não-negado)
# NEG_HINTS = (" sem ", " ausencia ", " nao ")
# def compact_summary_from_hits(hits: list[dict], doc, organ_blocks: list[dict]) -> list[dict]:
#     # 1) só positivos
#     pos = [h for h in hits if not h.get("negated", False)]
#     if not pos:
#         return []

#     # 2) Conclusão — RAW e NORM
#     concl_lines_raw = extract_section_lines(doc.text, "conclusao")
#     concl_lines_norm = [norm(ln) for ln in concl_lines_raw]

#     # 3) Fallback por bloco do órgão — RAW e NORM
#     organ_texts = {b["organ"]: doc.text[b["span"].start_char:b["span"].end_char] for b in organ_blocks}
#     organ_lines_raw = {org: [ln.strip() for _,_,ln in _line_spans_text(txt)]
#                        for org, txt in organ_texts.items()}
#     organ_lines_norm = {org: [norm(ln) for ln in lines]
#                         for org, lines in organ_lines_raw.items()}

#     # 4) agrupa por achado
#     by_f = {}
#     for h in pos:
#         by_f.setdefault(h["finding"], []).append(h)

#     out = []
#     for f, hs in by_f.items():
#         # ordena por confiança desc e span menor
#         hs.sort(key=lambda h: (-h.get("confidence", 0.0), h["end_char"] - h["start_char"]))

#         # queries: surface + tokens do nome do achado + padrões auxiliares
#         queries = set()
#         for h in hs:
#             s = norm(h.get("surface",""))
#             if s:
#                 queries.add(s)
#                 # também unigrams da surface (pega "nodulo", "hipervascularizado")
#                 for t in s.split():
#                     if len(t) >= 4:
#                         queries.add(t)
#         # nome canônico
#         fname = _pretty_finding_name(f)               # ex.: "nodulo hipervascular"
#         queries.add(norm(fname))
#         for t in norm(fname).split():
#             if len(t) >= 4:
#                 queries.add(t)

#         # heurística: se é achado nodular, garanta termos gerais
#         if "nodulo" in queries or "nodule" in queries:
#             queries |= {"nodulo", "nodular", "lirads", "li-rads"}

#         best_line = None
#         best_q = ""

#         # 4.1 Conclusão (preferência forte) — devolve RAW encurtado
#         for i, ln_norm in enumerate(concl_lines_norm):
#             if _any_q_in_line(ln_norm, queries) and not any(h in ln_norm for h in NEG_HINTS):
#                 best_line = concl_lines_raw[i]
#                 # escolhe a query mais discriminativa para cortar
#                 # prioriza "lirads" ou "nodulo"
#                 if "lirads" in ln_norm:
#                     best_q = "lirads"
#                 elif "nodulo" in ln_norm:
#                     best_q = "nodulo"
#                 else:
#                     # pega a maior query que exista na linha (mais específica)
#                     best_q = max([q for q in queries if q in ln_norm], key=len, default="")
#                 best_line = _shorten_clause(best_line, best_q, max_chars=160)
#                 break

#         # 4.2 Fallback: bloco do órgão — RAW encurtado
#         if best_line is None:
#             organ = hs[0]["organ"]
#             onorm = organ_lines_norm.get(organ, [])
#             oraw  = organ_lines_raw.get(organ, [])
#             for i, ln_norm in enumerate(onorm):
#                 if _any_q_in_line(ln_norm, queries):
#                     # query mais específica presente na linha
#                     present = [q for q in queries if q in ln_norm]
#                     qbest = max(present, key=len) if present else ""
#                     best_line = _shorten_clause(oraw[i], qbest, max_chars=160)
#                     break

#         # 4.3 Último recurso: extrair contexto do documento usando a posição do hit
#         if best_line is None:
#             # Pegar o hit com maior confiança
#             best_hit = hs[0]
#             start_char = best_hit["start_char"]
#             end_char = best_hit["end_char"]
            
#             # Encontrar a sentença que contém o hit
#             finding_sent = None
#             for sent in doc.sents:
#                 if sent.start_char <= start_char < sent.end_char:
#                     finding_sent = sent
#                     break
            
#             if finding_sent:
#                 # Usar a sentença completa
#                 sent_text = finding_sent.text.strip()
                
#                 # Se a sentença for muito longa, encurtar em torno do hit
#                 if len(sent_text) > 200:
#                     # Calcular posição relativa do hit na sentença
#                     hit_pos_in_sent = start_char - finding_sent.start_char
                    
#                     # Janela de 100 chars antes e depois
#                     window_start = max(0, hit_pos_in_sent - 100)
#                     window_end = min(len(sent_text), hit_pos_in_sent + 100)
                    
#                     snippet = sent_text[window_start:window_end].strip()
                    
#                     # Adicionar reticências se cortou
#                     if window_start > 0:
#                         snippet = "…" + snippet.split(" ", 1)[-1] if " " in snippet else snippet
#                     if window_end < len(sent_text):
#                         snippet = snippet.rsplit(" ", 1)[0] + "…" if " " in snippet else snippet + "…"
                    
#                     best_line = snippet
#                 else:
#                     best_line = sent_text
#             else:
#                 # Fallback: extrair janela em torno do hit
#                 window_start = max(0, start_char - 100)
#                 window_end = min(len(doc.text), end_char + 100)
#                 snippet = doc.text[window_start:window_end].strip()
                
#                 # Adicionar reticências
#                 if window_start > 0:
#                     snippet = "…" + snippet
#                 if window_end < len(doc.text):
#                     snippet = snippet + "…"
                
#                 best_line = snippet

#         out.append({ _pretty_finding_name(f): best_line })
#     return out


# @lru_cache(maxsize=8192)
# def _accent_rx(t: str) -> str:
#     rep = {
#         'a': '[aáàâãä]', 'e': '[eéèêë]', 'i': '[iíìîï]',
#         'o': '[oóòôõö]', 'u': '[uúùûü]', 'c': '[cç]', 'n': '[nñ]'
#     }
#     out = []
#     for ch in t.lower():
#         out.append(rep.get(ch, re.escape(ch)))
#     return ''.join(out)

# #========================================================================

# # termos genéricos de 1 palavra que geram falso positivo — ajuste conforme seu corpus
# GENERIC_SINGLETON_BAN = {
#     "livre","leito","difusa","difuso","vias","via","direito","esquerdo","normal",
#     "discreto","discreta","helicoidal","multislice","aortoiliaca","contornos",
#     "calibre","material","gasoso","fecal","porta","veia","arteria","hepatico",
#     "hepatica","hilo","anastomose","fluxo","indice","resistencia","propria",
#     "ramo","velocidade","pre","pos","maxima","mucosa","vascular","padrao",
#     "padrao vascular","conteudo","homogeneo","integridade", "parietal"
# }  # como singleton; em compostos ok

# def token_set(s: str) -> set[str]:
#     return set([t for t in norm(s).split() if t and t not in STOP_WORDS_NORM])

# def term_is_valid_for_seeds(term: str, seeds: list[str]) -> bool:
#     toks = token_set(term)
#     if not toks:
#         return False

#     # se houver seed multi-termo, exija candidato >=2 tokens
#     if any(len(token_set(s)) >= 2 for s in seeds) and len(toks) < 2:
#         return False

#     # singletons banidos
#     if len(toks) == 1 and next(iter(toks)) in GENERIC_SINGLETON_BAN:
#         return False

#     # sobreposição "literal"
#     seed_tokens = set().union(*[token_set(s) for s in seeds])
#     overlap_plain = len(toks & seed_tokens)

#     if overlap_plain >= 1:
#         return True

#     # ---- raiz morfológica leve (inline, sem função nova) ----
#     def _rootize_token(t: str) -> str:
#         x = norm(t)
#         # plurais/variações simples
#         x = re.sub(r'(oes|ões|s)$', '', x, flags=re.IGNORECASE)
#         x = re.sub(r'(ario|ária|ario(s)?|ária(s)?|ário|ária|ários|árias)$', '', x, flags=re.IGNORECASE)
#         return x

#     term_root = {_rootize_token(t) for t in toks}
#     seed_root = set().union(*[{_rootize_token(t) for t in token_set(s)} for s in seeds])

#     overlap_root = len(term_root & seed_root)
#     return overlap_root >= 1


# #========================================================================

# # --- CONFIG embutida (edite aqui) ---
# CONFIG = {
#     "negation": {
#         "tokens": [
#             "nao", "sem", "ausencia de", "negado", "nega", "descarta", "livre de", "sem evidencias de",
#             "sem sinais de", "sem achados de", "sem alteraçoes", "sem alteracoes", "sem sinais", "sem achados",
#             "sem alteracao", "sem alteraçao", "sem polipos", "sem diverticulos"
#         ],
#         "window_tokens": 6  # alcance simples de negação
#     },
#     "organs": {
#         # Cada órgão: seeds e (opcional) regex adicionais
#         "figado": {
#             "seeds": ["figado", "hepatico", "hepatica", "lobos hepaticos", "parenquima hepatico", "hepatopatia"],
#             "regex": [r"\bhepat\w*"]
#         },
#         "rim": {
#             "seeds": ["rim", "renal", "parênquima renal", "pielocalicial", "cortex renal"],
#             "regex": [r"\bren(al|ais)\b", r"\brim(es)?\b"]
#         },
#         "vesicula biliar": {
#             "seeds": ["vesicula", "vesicula biliar", "colecisto", "colecistica"],
#             "regex": [r"\bcolec\w*"]
#         },
#         "colon_reto": {
#             "seeds": [
#                 # Termos gerais
#                 "colon", "cólon",
#                 # Segmentos do cólon (encontrados nos laudos)
#                 "colon ascendente", "cólon ascendente", "colon transverso", "cólon transverso",
#                 "colon descendente", "cólon descendente", "colon sigmoide", "cólon sigmoide",
#                 # Ceco e apêndice (MUITO FREQUENTES - 72x apêndice, 70x apêndice cecal)
#                 "ceco", "apendice", "apêndice", "apendice cecal", "apêndice cecal",
#                 "apendice retrocecal", "apêndice retrocecal", "apendice vermiforme", "apêndice vermiforme",
#                 # Válvula ileocecal
#                 "valvula ileocecal", "válvula ileocecal", "valvula ileo cecal", "válvula íleo cecal",
#                 "valvula ileo-cecal", "válvula íleo-cecal",
#                 # Sigmoide
#                 "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
#                 # Reto
#                 "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
#                 # Transição
#                 "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide", "transição reto-sigmoide",
#                 "juncao retossigmoide", "junção retossigmoide",
#                 # Alças e estruturas (encontrados nos laudos)
#                 "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon", "alças do cólon",
#                 "parede colonica", "parede colônica",
#                 # Divertículos (FREQUENTES - 20x, termo comum em tomografias)
#                 "diverticulos", "divertículos", "diverticulos colonicos", "divertículos colônicos",
#                 "diverticulos colicos", "divertículos cólicos", "diverticulose", "diverticulose colonica", "diverticulose colônica"
#             ],
#             "regex": [
#                 r"\bc[oó]l[oô]n\b",  # cólon ou colon (SEM "oscopia")
#                 r"\breto(s)?\b",  # reto ou retos
#                 r"\bsigm[oó]ide\b",  # sigmoide ou sigmóide
#                 r"\b(ascendente|transverso|descendente)\b",  # segmentos
#                 r"\bceco\b",  # ceco
#                 r"\bap[eê]ndice( cecal| retrocecal| vermiforme)?\b",  # apêndice e variações
#                 r"\bv[aá]lvula (i|í)leo ?-?cecal\b",  # válvula ileocecal
#                 r"\btransi[cç][aã]o reto-?sigm[oó]ide\b",  # transição retossigmoide
#                 r"\bjun[cç][aã]o retossigm[oó]ide\b",  # junção retossigmoide
#                 r"\bal[cç]as col[oô]nicas\b",  # alças colônicas
#                 r"\bparede col[oô]nica\b",  # parede colônica
#                 r"\bdivert[ií]culos?( col[oô]nicos?| c[oó]licos?)?\b",  # divertículos e variações
#                 r"\bdiverticulose( col[oô]nica)?\b", # diverticulose
#             ]  
#         },
#         "dii": {
#             "seeds": [
#                 # --- REGIÕES DO TRATO INTESTINAL ASSOCIADAS À DII ---
#                 "intestino delgado",
#                 "intestino grosso",
#                 "ileo", "íleo",
#                 "ileo distal", "íleo distal",
#                 "ileo terminal", "íleo terminal",
                
#                 "jejuno",
#                 "segmento jejunal", "segmento jejunoileal",
#                 "colon", "cólon",
#                 "colon ascendente", "cólon ascendente",
#                 "colon transverso", "cólon transverso",
#                 "colon descendente", "cólon descendente",
#                 "colon sigmoide", "cólon sigmoide",
#                 "sigmoide", "sigmóide",
#                 "reto", "reto distal", "reto medio", "reto proximal",
#                 "regiao perianal", "perianal",
#                 "valvula ileocecal", "válvula ileocecal",
#                 "mesorreto",
#                 "alcas intestinais", "alças intestinais",
#                 "alcas do intestino", "alca intestinal",
#                 "trato gastrointestinal",
#                 "colon",
#                 "cólon",
#                 "colon ascendente",
#                 "cólon ascendente",
#                 "colon transverso",
#                 "cólon transverso",
#                 "colon descendente",
#                 "cólon descendente",
#                 "colon sigmoide",
#                 "cólon sigmoide",
#                 "sigmoide",
#                 "sigmóide",
#                 "reto",
#                 "reto distal",
#                 "reto proximal",
#                 "apendice cecal",
#                 "apêndice cecal",
#                 "valvula ileocecal",
#                 "válvula ileocecal",


#                 # --- ALTERAÇÕES ESTRUTURAIS QUE SÃO FORTES “ÂNCORAS” DE DII ---
#                 "espessamento parietal intestinal",
#                 "espessamento parietal difuso",
#                 "espessamento do colon",
#                 "engrossamento parietal",
#                 "densificacao mesenterica", 
#                 "densificação mesenterica",
#                 "infiltracao inflamatória", "infiltração inflamatória",
#                 "realce mucoso anormal",
#                 "edema da parede intestinal",
#                 "espessamento parietal",
#                 "espessamento parietal intestinal",
#                 "espessamento do intestino",
#                 "distensao liquida",
#                 "distensão líquida",
#                 "densificacao dos planos adiposos",
#                 "densificação dos planos adiposos",
#                 "ingurgitamento dos vasos retos",
#                 "vasos retos ingurgitados",
#                 "planejamento adiposo densificado",
#                 "inflamacao intestinal",
#                 "inflamação intestinal",
#                 "processo inflamatorio intestinal",
#                 "processo inflamatório intestinal",
#                 "enterite",   # sem assumir causa
#                 "enterite inflamatoria",
#                 "enterite inflamatória",

#                 # --- OUTROS ---
#                 "ingurgitamento de vasos retos",
#                 "densificação da gordura adjacente",


#                 # --- SEEDS DE DOENÇA (DEIXAMOS TAMBÉM, MAS O ACHADO VEM EM FINDINGS) ---
#                 "dii",
#                 "doenca de crohn",
#                 "retocolite ulcerativa",
#                 "colite ulcerativa",
#                 "compatível com doenca de crohn",
#                 "sugestivo de doenca de crohn",
#                 "aspecto compatível com retocolite ulcerativa"
#             ],

#             "regex": [
#                 # REGIÕES
#                 r"\b(i|í)leo( terminal)?\b",
#                 r"\bjejuno\b",
#                 r"\bintestino(s)? (delgado|grosso)\b",
#                 r"\bmesorreto\b",
#                 r"\bperianal\b",
#                 r"\bvalvul(a|á) (i|í)leo ?-?cecal\b",

#                 # ALTERAÇÕES SUSPEITAS
#                 r"\bespessamento parietal\b",
#                 r"\bdensifica[cç][aã]o mesent[eé]rica\b",
#                 r"\binfiltra[cç][aã]o inflamat[oó]ria\b",
#                 r"\bedema da parede\b",

#                 # DOENÇAS
#                 r"\bdoen[cç]a de crohn\b",
#                 r"\bretocolite ulcerativa\b",
#                 r"\bcolite ulcerativa\b"
#             ]
#         }
#         # "dii": {
#         #     "seeds": [
#         #         "dii",
#         #         "doenca de crohn",
#         #         "compatível com doenca de crohn",
#         #         "sugestivo de doenca de crohn",
#         #         "retocolite ulcerativa",
#         #         "aspecto compatível com retocolite ulcerativa"
#         #     ],
#         #     "regex": [
#         #         r"\bdoenca de crohn\b",
#         #         r"\bretocolite ulcerativa\b"
#         #     ]
#         # }
#         # Adicione novos órgãos aqui
#     },
#     "findings": {
#         # Achados por órgão (canônicos + seeds). A expansão semântica cria sinônimos dinâmicos no runtime
#         "figado": {
#             "esteatose": ["esteatose", "infiltracao gordurosa", "gordura hepatica", "esteatose", "doenca hepatica gordurosa", "gordura hepatica", "ecogenicidade aumentada", "hiperecogenicidade difusa", "ecotextura aumentada", "ecogenicidade parenquimatosa aumentada", "ecotextura hepatica aumentada", "ecotextura parenquimatosa aumentada", "ecogenicidade difusamente aumentada", "hipoatenuacao difusa", "atenuacao reduzida difusa"],
#             "infiltracao_gordurosa": ["infiltracao gordurosa"],
#             "fibrose": ["fibrose", "fibrose hepatica", "fibrose avancada", "fibrose cirrosica", "fibrose cirrosica avancada"],
#             "nodulo": ["nodulo", "nódulo", "nódulos", "nódulos hepáticos", "nodularidade", "nodular", "nodulares"],
#             "cirrose": ["cirrose", "cirrose hepatica", "cirrosica", "cirrose avancada"],
#             "hepatomegalia": ["hepatomegalia", "figado aumentado"],
#             "trombose_veia_porta": ["trombose da veia porta", "trombose portal", "trombose da veia portal"],
#             "baco_aumentado": ["baco aumentado"],
#             "cavernoma_porta": ["cavernoma porta"],
#             "circulacao_colateral": ["circulacao colateral"],
#             "congestao_passiva_cronica_figado": ["congestao passiva cronica figado", "congestao passiva cronica"],
#             "contorno_irregular": ["contorno irregular"],
#             "degeneracao_gordurosa_figado": ["degeneracao gordurosa figado", "degeneracao gordurosa"],
#             "dilatacao_vias_biliares": ["dilatacao vias-biliares"],
#             "doenca_alcoolica_figado": ["doenca alcoolica figado", "doenca alcoolica"],
#             "doenca_hepatica": ["doenca hepatica", "contorno lobulado", "contorno levemente lobulado"],
#             "encefalopatia_hepatica": ["encefalopatia hepatica"],
#             "esclerose_hepatica": ["esclerose hepatica"],
#             "fibrose_esclerose_alcoolicas": ["fibrose esclerose alcoolicas"],
#             "figado_gorduroso_alcoolico": ["figado gorduroso alcoolico"],
#             "hepatite": ["hepatite"],
#             "hipertensao_portal": ["hipertensao portal"],
#             "hipertrofia_lobo_caudado": ["hipertrofia lobo caudado"],
#             "infarto_figado": ["infarto figado"],
#             "insuficiencia_hepatica_alcoolica": ["insuficiencia hepatica alcoolica"],
#             "insuficiencia_hepatica": ["insuficiencia hepatica"],
#             "lirads_4": ["lirads 4", "categoria 4", "categ 4", "rads 4"],
#             "lirads_5": ["lirads 5", "categoria 5", "categ 5", "rads 5"],
#             "lobulado": ["lobulado", "lobuloso", "lobulos"],
#             "necrose_hemorragica": ["necrose hemorragica"],
#             "nodulo_hipervascular": ["nodulo hipervascular", "nodular hipervascularizado"],
#             "reduzido": ["reduzido", "diminuido"],
#             "sindrome_obstrucao_sinusoidal_hepatica": ["sindrome obstrucao sinusoidal hepatica"],
#             "varizes_esofagianas": ["varizes esofagianas"],
#             "varizes_gastricas": ["varizes gastricas"],
#             "veia_porta_dilatada": ["veia porta dilatada"],
#             "hepatopatia_cronica": ["hepatopatia cronica"],
#             "microlitiase": ["microlitiase"],
#             "multiplos_calculos": ["multiplos calculos"]
#         },
#         "rim": {
#             "hidronefrose": ["hidronefrose", "dilatacao pielocalicial", "sistema coletor dilatado"],
#             "litiases": ["calculo renal", "litiases renais", "nefrolitiase"]
#         },
#         "vesicula biliar": {
#             "colelitíase": ["calculo em vesicula", "colelitíase", "calculose vesicular"],
#             "colecistite": ["colecistite", "parede espessada", "sinal de murphy ultrassonografico"]
#         },
#         "colon_reto": {
#         "lesao": [
#             "lesao", "lesoes", "lesao vegetante", "lesao infiltrativa", "lesao exofitica", "lesao deprimida",
#             "estenose por lesao", "lesao suspeita", "lesao ulcerada", "areas de necrose", "tecido suspeito",
#             "angiectasia", "angiodisplasia"
#         ],
#         "polipo": [
#             "polipo", "polipos", "polipo pediculado", "polipo sessil","polipo sésil","polipo sesseil",
#             "lesao plana", "adenoma", "lesao sessil","lesao séssil", "lesao pediculada",
#             "paris 0-ii", "paris 0-iia", "paris 0-iib", "paris 0-iic", "paris 0-is",
#             "polipo suspeito", "polipos suspeitos"
#         ],
#         "ulceracao":[
#             "ulcera", "ulcero", "ulcerativa", "ulcero infitrativa", "ulcero-infiltrativa", "ulceracao", "ulceracoes", "erosao", "erosoes",
#             "erosiva", "afoide", "aftoide", "aftoides"
#         ],
#         "tumor_ou_massa": [
#             "massa", "tumor", "tumoracao", "tumoracoes", "tumoracao exofitica", "nodulo", "nodular", "nodulacao", "nodulacoes",
#             "laterally spreading tumor", "lst", "lst-g", "lst-ng", "neoplasia", "adeoncarcinoma", "carcinoma", "neoplasica", 
#             "neoplasico", "nodulares", "metastase"
#         ],
#         "espessamento":[
#             "rigidez", "espessamento"
#         ],
#         "sangramento":[
#             "friabilidade", "sangramento"
#         ],
#         "contorno_ou_margem":[
#             "alteracao irregular", "margem irregulares", "margens irregulares", "parede irregular", "alteracao focal", "alteracao de contorno", "bordas irregulares",
#             "crescimento heterogeneo", "opacidade heterogenea", "densidade heterogenea", "escavacoes", "perda de margens claras", "perda do padrao",
#             "lateralmente extensa", "margem mal definida", "margem indistinta", "contorno irregular"
#         ],
#         #"neoplasia_confirmada": [
#         #    "adenocarcinoma", "carcinoma", "carcinoma in situ", "neoplasia maligna", "neoplasia invasiva"
#         #],
#         #"displasia_alto_grau": [
#         #    "displasia de alto grau", "alto grau de displasia", "hgd"
#         #],
#         #"hiperplasia_linfoide": [
#         #    "hiperplasia linfoide", "hiperplasia linfoide reacional"
#         #],
#         #"diverticulose": [
#         #    "diverticulos", "diverticulose", "ostios diverticulares"
#         #],
#         #"hemorroidas": [
#         #    "hemorroida", "hemorroidas", "botoes hemorroidarios", "botões hemorroidários"
#         #],
#     #     },
#     # },
#         # ---------------------------
#         # 🔎 DETECÇÃO DE DII (NOVO)
#         # ---------------------------
 
#         },
#         "dii": {
#             "dii": [
#                 "dii",
#                 "doenca inflamatória intestinal",
#                 "doenca inflamatoria intestinal",

#                 # Doença de Crohn
#                 "doenca de crohn",
#                 "doença de crohn",
#                 "sugestivo de doenca de crohn",
#                 "sugestivo de doença de crohn",
#                 "compatível com doenca de crohn",
#                 "compatível com doença de crohn",
#                 "padrão compatível com crohn",
#                 "sugere crohn",
#                 "atividade da doença de crohn",
#                 "atividade da doenca de crohn",

#                 # Retocolite ulcerativa (RCU)
#                 "retocolite ulcerativa",
#                 "colite ulcerativa",
#                 "colite ulcerativa cronica",
#                 "colite ulcerativa crônica",
#                 "compatível com retocolite ulcerativa",
#                 "aspecto compatível com retocolite ulcerativa",

#                 # Termos clínicos que sugerem DII
#                 "doença de base crohn",
#                 "doenca de base crohn",
#                 "doenca de base inflamatória intestinal",
#                 "doença de base inflamatória intestinal",

#                 # Frases muito fortes clinicamente
#                 "enterite cronica",
#                 "enterite crônica",
#                 "processo inflamatorio cronico intestinal",
#                 "processo inflamatório crônico intestinal",
#                 "inflamacao transmural",
#                 "inflamação transmural",
#                 "padrao de crohn"
#             ]
#         },
#             # ACHADOS ESPECÍFICOS PARA O "ÓRGÃO" DII
#         # "dii": {
#         #     "dii": [
#         #         "dii",
#         #         "doenca de crohn",
#         #         "compatível com doenca de crohn",
#         #         "sugestivo de doenca de crohn",
#         #         "retocolite ulcerativa",
#         #         "aspecto compatível com retocolite ulcerativa"
#         #     ]
#         # },
#     },
#     # Limiar de similaridade para incluir novos termos do texto no léxico semântico
#     "semantic": {
#         "min_sim_seed_vocab": 0.85,  # quão parecido um termo do vocabulário do laudo precisa ser com os seeds
#         "top_k_per_seed": 8,
#         # Geramos candidatos do vocabulário via noun_chunks + n-grams leves
#         "min_token_len": 3
#     }
# }

# # ===
# # --- Seleção de órgão alvo (string ou lista). Deixe None para todos (não recomendado).
# # TARGET_ORGAN = "colon_reto"   # ex.: "figado" ou ["figado","rim"] ou None
# TARGET_ORGAN = "dii"
# #Troque o órgão se quiser que o algoritmo leia todo o laudo, sem filtrar por trechos.
# FORCE_FULL_DOC_FOR = set()
# # FORCE_FULL_DOC_FOR = {"dii"}
# #Use quando nao quiser forçar leitura completa do laudo
# #FORCE_FULL_DOC_FOR = set()

# # ===
# # --- Stopwords com preservação de operadores de negação ---
# # base spaCy
# SPACY_STOP = set(w.lower() for w in nlp.Defaults.stop_words)

# # operadores de negação que NÃO devem ser removidos
# OPERATORS = {"não", "nao", "sem", "há", "ha"}  # pode incluir mais

# # também considere os tokens de negação que já listamos no CONFIG
# OPERATORS |= set(CONFIG["negation"]["tokens"])

# # normaliza tudo (mesmo critério da função norm)
# def _norm_plain(s: str) -> str:
#     import unicodedata, re
#     s = s.lower().strip()
#     s = ''.join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != 'Mn')
#     s = re.sub(r"\s+", " ", s)
#     return s

# STOP_WORDS_NORM = { _norm_plain(w) for w in SPACY_STOP }
# OPERATORS_NORM  = { _norm_plain(w) for w in OPERATORS }

# # remove operadores do conjunto de stopwords
# STOP_WORDS_NORM = STOP_WORDS_NORM - OPERATORS_NORM

# #=======================================================

# # Cabeçalho: começa no início da linha, pode ter bullet, NOME:, e pode TER CONTEÚDO depois do ":"
# HEADER_RX = re.compile(r'^\s*[-•]?\s*([A-ZÁ-Úa-zá-ú/ ]{2,})\s*:\s*', re.UNICODE)

# def _line_spans_text(text: str):
#     spans, off = [], 0
#     for line in text.splitlines(True):
#         spans.append((off, off+len(line), line))
#         off += len(line)
#     return spans

# def _norm_header_name(s: str) -> str:
#     return norm(s.replace(":", "").replace("-", " ").strip())

# def extract_section_lines(text: str, header_name: str) -> list[str]:
#     """Linhas (brutas) da seção `header_name` até o próximo cabeçalho."""
#     lines = _line_spans_text(text)
#     headers = []
#     for (s,e,line) in lines:
#         m = HEADER_RX.match(line)
#         if m:
#             headers.append((s, _norm_header_name(m.group(1))))
#     if not headers:
#         return []
#     target = _norm_header_name(header_name)
#     starts = [s for (s,name) in headers if name.startswith(target)]
#     if not starts:
#         return []
#     s0 = starts[0]
#     nexts = [S for (S,_) in headers if S > s0]
#     s1 = nexts[0] if nexts else len(text)
#     sub = text[s0:s1]
#     return [ln.strip() for _,_,ln in _line_spans_text(sub)]

# # Sinônimos básicos para casar cabeçalho textual com órgão de CONFIG
# HEADER_ALIASES = {
#     "figado": {"figado", "fígado", "fígado ", "figado "},
#     "rim": {"rim", "rins"},
#     "vesicula biliar": {"vesicula", "vesícula", "vesicula biliar", "vesícula biliar"},
#     # adicione aliases para outros cabeçalhos que existam no laudo (ex.: vias biliares, pancreas, adrenais, etc.)
#     "vias biliares": {"vias biliares", "vias biliares intra e extra-hepaticas", "vias biliares intra e extra-hepáticas"},
#     "adrenais": {"adrenais", "supra-renais", "suprarrenais"},
#     "pancreas": {"pancreas", "pâncreas"},
#     "bases pulmonares": {"bases pulmonares", "pulmoes", "pulmões"},
#     "conclusao": {"conclusao", "conclusão"},
#     "colon_reto": {
#         # Termos gerais
#         "colon", "cólon",
#         # Segmentos do cólon
#         "colon ascendente", "cólon ascendente", "colon transverso", "cólon transverso",
#         "colon descendente", "cólon descendente", "colon sigmoide", "cólon sigmoide",
#         # Ceco e apêndice (apendice removido)
#         "ceco",
#         # Válvula ileocecal
#         "valvula ileocecal", "válvula ileocecal", "valvula ileo cecal", "válvula íleo cecal",
#         "valvula ileo-cecal", "válvula íleo-cecal",
#         # Sigmoide
#         "sigmoide", "sigmóide", "sigmoide redundante", "sigmóide redundante",
#         # Reto
#         "reto", "reto distal", "reto medio", "reto médio", "reto proximal",
#         # Transição
#         "transicao retossigmoide", "transição retossigmoide", "transicao reto-sigmoide",
#         "transição reto-sigmoide", "juncao retossigmoide", "junção retossigmoide",
#         # Alças e estruturas
#         "alças colônicas", "alças colônicas", "alcas colonicas", "alças do colon",
#         "alças do cólon", "parede colonica", "parede colônica",
#         # Divertículos
#         "diverticulos", "divertículos", "diverticulos colonicos", "divertículos colônicos",
#         "diverticulos colicos", "divertículos cólicos", "diverticulose",
#         "diverticulose colonica", "diverticulose colônica"
#     }
# }

# def segment_by_headers(doc, target_organs: list[str]) -> list[dict]:
#     """
#     Delimita blocos pelo NOME: do cabeçalho.
#     Fecha o bloco no PRÓXIMO cabeçalho (qualquer nome), mesmo que haja texto na mesma linha do cabeçalho.
#     """
#     text = doc.text
#     lines = _line_spans_text(text)

#     # 1) capture TODOS os cabeçalhos (nome e posição de início)
#     all_headers = []  # [(start_char, header_name_norm)]
#     for (s, e, line) in lines:
#         m = HEADER_RX.match(line)
#         if m:
#             name = _norm_header_name(m.group(1))
#             all_headers.append((s, name))

#     if not all_headers:
#         return []

#     # 2) quais cabeçalhos são do(s) órgão(s) alvo?
#     wanted = set()
#     for o in target_organs:
#         wanted |= HEADER_ALIASES.get(o, {o})

#     organ_headers = [(s, name) for (s, name) in all_headers if any(name.startswith(_norm_header_name(w)) for w in wanted)]

#     if not organ_headers:
#         return []

#     # 3) blocos: [header alvo i, próximo header global)
#     all_headers_sorted = sorted(all_headers, key=lambda x: x[0])
#     blocks = []
#     for s, name in sorted(organ_headers, key=lambda x: x[0]):
#         # próximo cabeçalho global após 's'
#         next_starts = [S for (S, _) in all_headers_sorted if S > s]
#         end = next_starts[0] if next_starts else len(text)
#         span = doc.char_span(s, end, alignment_mode="expand")
#         if span is not None and (span.end - span.start) > 0:
#             # mapeia de volta para o órgão alvo (pega o primeiro que bate)
#             mapped = None
#             for o in target_organs:
#                 if any(name.startswith(_norm_header_name(w)) for w in HEADER_ALIASES.get(o, {o})):
#                     mapped = o
#                     break
#             blocks.append({"organ": mapped or target_organs[0], "span": span})
#     return blocks

# #=======================================================

# def fuzzy_sim(a: str, b: str) -> float:
#     # similaridade 0..1 (robusto a typos simples)
#     return SequenceMatcher(None, norm(a), norm(b)).ratio()

# def best_fuzzy_in_vocab(term: str, vocab: list[str], min_ratio: float = 0.86) -> str | None:
#     # retorna o termo do vocabulário mais parecido acima do limiar
#     best, best_s = None, 0.0
#     for v in vocab:
#         s = fuzzy_sim(term, v)
#         if s > best_s:
#             best, best_s = v, s
#     return best if best_s >= min_ratio else None

# def _resolve_target_organs(target, organs_cfg):
#     if target is None:
#         return list(organs_cfg.keys())
#     if isinstance(target, str):
#         target = [target]
#     out = []
#     for o in target:
#         k = o.strip().lower()
#         if k not in organs_cfg:
#             raise ValueError(f"Órgão desconhecido: {o}. Disponíveis: {list(organs_cfg.keys())}")
#         out.append(k)
#     return out

# # --- Extração de vocabulário candidato do laudo (noun chunks + n-grams leves) ---
# def build_doc_candidates(doc, min_len=3) -> list[str]:
#     cand = []

#     # 1) noun_chunks: mantemos inteiros (úteis como termos compostos reais do laudo)
#     for nc in doc.noun_chunks:
#         t = norm(nc.text)
#         if len(t) >= min_len and not t.isnumeric():
#             cand.append(t)

#     # 2) n-grams: antes de montar, filtramos stopwords (com negadores preservados)
#     toks = [norm(t.text) for t in doc if t.is_alpha and len(norm(t.text)) >= min_len]
#     # STOP_WORDS_NORM deve ter sido criado antes, removendo OPERATORS (negadores)
#     toks = [t for t in toks if t not in STOP_WORDS_NORM]

#     # 3) gerar n-grams (1–3) a partir dos tokens filtrados
#     for n in (1, 2, 3):
#         for i in range(len(toks) - n + 1):
#             cand.append(" ".join(toks[i:i+n]))

#     # 4) dedup e retorno
#     return unique([c for c in cand if len(c) >= min_len])

# # --- Expansão semântica de seeds usando vocabulário do próprio laudo ---
# def semantic_expand(seeds: List[str], vocab: List[str],
#                     min_sim: float, top_k: int) -> Dict[str, List[Tuple[str, float]]]:
#     if not seeds or not vocab:
#         return {s: [] for s in seeds}

#     seed_norm  = [norm(s) for s in seeds]
#     seed_emb   = embedder.encode(seed_norm, convert_to_numpy=True, normalize_embeddings=True)
#     vocab_norm = [norm(v) for v in vocab]
#     vocab_emb  = embedder.encode(vocab_norm, convert_to_numpy=True, normalize_embeddings=True)

#     out = {s: [] for s in seeds}
#     for i, s in enumerate(seeds):
#         s_tokens = token_set(s)  # tokens normalizados da seed
#         pairs = []

#         for j, v in enumerate(vocab):
#             emb_sim = float(np.dot(seed_emb[i], vocab_emb[j]))
#             fz_sim  = fuzzy_sim(s, v)
#             score   = 0.75 * emb_sim + 0.25 * fz_sim
#             if score < min_sim:
#                 continue

#             v_tokens = token_set(v)
#             overlap = len(v_tokens & s_tokens)

#             # Regra FLEXÍVEL de interseção:
#             # - Se a similaridade vetorial é ALTA (>= 0.70), aceita mesmo sem interseção explícita
#             # - Se é BAIXA/MÉDIA, exige interseção (>=1 token). Para seeds multi-termo, exige >=2.
#             if emb_sim < 0.70:
#                 if len(s_tokens) >= 2:
#                     if overlap < 2:
#                         continue
#                 else:
#                     if overlap < 1:
#                         continue

#             pairs.append((v, score))

#         pairs.sort(key=lambda x: x[1], reverse=True)
#         out[s] = pairs[:top_k]
#     return out

# def merge_semantic_sets(seed_list: List[str], sem_expanded: Dict[str, List[Tuple[str,float]]]) -> List[str]:
#     terms = list(seed_list)
#     for s, pairs in sem_expanded.items():
#         for term, sim in pairs:
#             if term not in terms:
#                 terms.append(term)
#     return unique(terms)

# # --- Segmentação por órgão (ancora por match semântico+regex+seed) ---
# def organ_anchors(doc: "spacy.tokens.Doc", organ_name: str, organ_cfg: Dict[str, Any],
#                   organ_lexicon: List[str]) -> List[Tuple[int,int]]:
#     # Retorna spans (start_token, end_token) que ancoram menções ao órgão
#     import re
#     spans = []
#     organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])] + [norm(x) for x in organ_lexicon])
    
#     for sent in doc.sents:
#         s = norm(sent.text)
#         hit = False
        
#         # Verificar seeds com word boundaries
#         for t in organ_terms:
#             # Escapar caracteres especiais de regex
#             t_escaped = re.escape(t)
#             # Usar word boundary (\b) para match exato de palavras
#             pattern = r'\b' + t_escaped + r'\b'
#             if re.search(pattern, s):
#                 hit = True
#                 break
        
#         # Se não encontrou via seeds, tentar regex
#         if not hit and organ_cfg.get("regex"):
#             for rx in organ_cfg["regex"]:
#                 if re.search(rx, s):
#                     hit = True
#                     break
        
#         if hit:
#             spans.append((sent.start, sent.end))
    
#     return spans

# def segment_by_organs(doc: "spacy.tokens.Doc",
#                       organs_cfg: Dict[str, Any],
#                       organ_lexicons: Dict[str, List[str]]) -> List[Dict[str, Any]]:
#     blocks = []
#     for organ, cfg in organs_cfg.items():
#         anchors = organ_anchors(doc, organ, cfg, organ_lexicons.get(organ, []))
#         for (a,b) in anchors:
#             # bloco: da sentença-âncora até antes da próxima âncora de qualquer órgão (simples: apenas a sentença-âncora)
#             span = doc[a:b]
#             blocks.append({"organ": organ, "span": span})
#     return blocks

# # --- Não casar termos quando começar com negador ---
# NEG_SET = {norm(x) for x in CONFIG["negation"]["tokens"]}

# def _starts_with_negator(term: str) -> bool:
#     toks = norm(term).split()
#     return len(toks) > 0 and toks[0] in NEG_SET

# # --- Negação simples baseada em janela ---
# def is_negated_in_sentence(doc,
#                            start_char: int, end_char: int,
#                            neg_list: list[str],
#                            window_tokens: int = 12) -> bool:
#     """
#     True se houver negador NA MESMA SENTENÇA:
#       - à ESQUERDA do termo (ex.: "não se observa ...", "nega ...")
#       - à DIREITA do termo (ex.: "... sem alterações", "... ausência de ...")
#     Usa a MESMA lista `neg_list` (derivada do CONFIG), sem dicionários extras.
#     """
#     hit = doc.char_span(start_char, end_char, alignment_mode="expand")
#     if hit is None:
#         return False
#     sent = hit.sent

#     # normaliza negadores uma vez
#     neg_norm = [norm(x) for x in neg_list]

#     # 0) se o próprio hit já começar por negador (raro, mas válido)
#     hit_norm = norm(hit.text)
#     if any(hit_norm.startswith(n + " ") or hit_norm == n for n in neg_norm):
#         return True

#     # 1) janela à esquerda
#     left_tokens  = [t.text for t in sent if t.idx <  hit.start_char]
#     left_tokens  = left_tokens[-window_tokens:] if window_tokens > 0 else left_tokens
#     left_norm    = norm(" ".join(left_tokens))
#     if any(n in left_norm for n in neg_norm):
#         return True

#     # 2) janela à direita
#     right_tokens = [t.text for t in sent if t.idx >= hit.end_char]
#     right_tokens = right_tokens[:window_tokens] if window_tokens > 0 else right_tokens
#     right_norm   = norm(" ".join(right_tokens))
#     if any(n in right_norm for n in neg_norm):
#         return True

#     return False

# # --- Matching de achados (léxico expandido) dentro de cada bloco ---
# @dataclass
# class FindingHit:
#     organ: str
#     finding: str
#     surface: str
#     start_char: int
#     end_char: int
#     negated: bool
#     confidence: float

# def _sentence_mentions_organ(sent_text: str, organ_name: str, organ_cfg: Dict[str, Any], organ_lexicon: List[str]) -> bool:
#     """
#     Verifica se uma sentença menciona um órgão específico.
#     Usa os mesmos critérios de organ_anchors() mas para uma única sentença.
#     """
#     import re
#     s = norm(sent_text)
#     organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])] + [norm(x) for x in organ_lexicon])
    
#     # Verificar seeds com word boundaries
#     for t in organ_terms:
#         t_escaped = re.escape(t)
#         pattern = r'\b' + t_escaped + r'\b'
#         if re.search(pattern, s):
#             return True
    
#     # Verificar regex
#     if organ_cfg.get("regex"):
#         for rx in organ_cfg.get("regex"):
#             if re.search(rx, s):
#                 return True
    
#     return False


# def _find_organ_mentions_in_text(text: str, organ_name: str, organ_cfg: Dict[str, Any], organ_lexicon: List[str]) -> List[Tuple[int, int]]:
#     """
#     Encontra todas as posições onde o órgão é mencionado no texto.
#     Retorna lista de tuplas (start, end) com as posições das menções.
#     """
#     import re
#     mentions = []
#     text_norm = norm(text)
    
#     # Seeds
#     organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])] + [norm(x) for x in organ_lexicon])
    
#     for term in organ_terms:
#         term_escaped = re.escape(term)
#         pattern = r'\b' + term_escaped + r'\b'
#         for m in re.finditer(pattern, text_norm):
#             mentions.append((m.start(), m.end()))
    
#     # Regex
#     if organ_cfg.get("regex"):
#         for rx_pattern in organ_cfg.get("regex"):
#             for m in re.finditer(rx_pattern, text_norm):
#                 mentions.append((m.start(), m.end()))
    
#     return mentions


# def _finding_is_near_organ_mention(finding_pos: int, organ_mentions: List[Tuple[int, int]], max_distance: int = 200) -> bool:
#     """
#     Verifica se o finding está próximo (dentro de max_distance caracteres) de alguma menção ao órgão.
#     """
#     if not organ_mentions:
#         return False
    
#     for mention_start, mention_end in organ_mentions:
#         dist_to_start = abs(finding_pos - mention_start)
#         dist_to_end = abs(finding_pos - mention_end)
#         min_dist = min(dist_to_start, dist_to_end)
        
#         if min_dist <= max_distance:
#             return True
    
#     return False


# def _extract_snippet_around_finding(doc_text: str, start: int, end: int, window: int = 100) -> str:
#     """
#     Extrai trecho ao redor do finding
#     """
#     snippet_start = max(0, start - window)
#     snippet_end = min(len(doc_text), end + window)
#     return doc_text[snippet_start:snippet_end]


# def _snippet_mentions_other_organs(snippet: str, target_organ: str, all_organs_config: Dict) -> bool:
#     """
#     Verifica se o snippet menciona OUTROS órgãos (diferentes do target_organ)
#     Retorna True se menciona APENAS outros órgãos (não menciona o target)
#     """
#     import re
    
#     menciona_target = False
#     menciona_outros = False
    
#     # Verificar cada órgão configurado
#     for organ_name, organ_cfg in all_organs_config.items():
#         # Seeds do órgão
#         organ_terms = set([norm(organ_name)] + [norm(x) for x in organ_cfg.get("seeds", [])])
        
#         # Verificar se o snippet menciona este órgão
#         menciona_este_orgao = False
#         for term in organ_terms:
#             # Word boundary para evitar falsos positivos
#             pattern = r'\b' + re.escape(term) + r'\b'
#             if re.search(pattern, norm(snippet)):
#                 menciona_este_orgao = True
#                 break
        
#         # Classificar
#         if menciona_este_orgao:
#             if organ_name == target_organ:
#                 menciona_target = True
#             else:
#                 menciona_outros = True
    
#     # Retorna True se menciona APENAS outros (não menciona target)
#     return menciona_outros and not menciona_target



# def match_findings_in_block(block: Dict[str,Any],
#                             finding_cfg: Dict[str, List[str]],
#                             neg_tokens: List[str], neg_window: int,
#                             min_sim_seed_vocab: float,
#                             organ_cfg: Dict[str, Any] = None,
#                             organ_lexicon: List[str] = None) -> List[FindingHit]:
#     span = block["span"]
#     text_norm = norm(span.text)
#     hits: List[FindingHit] = []
#     organ_name = block["organ"]

#     # vocabulário local do bloco para expansão
#     local_vocab = build_doc_candidates(span, min_len=CONFIG["semantic"]["min_token_len"])

#     for f_can, seeds in finding_cfg.items():
#         # 0) seeds originais SEMPRE entram (normalizados e deduplicados)
#         base_seeds = unique([norm(s) for s in seeds if s and norm(s)])

#         # 1) expansão semântica (gera TERMOS, não inclui seeds)
#         sem = semantic_expand(seeds, local_vocab, min_sim_seed_vocab, CONFIG["semantic"]["top_k_per_seed"])

#         # antes: você simplesmente adicionava todos os termos de 'sem'
#         # agora: usa os MESMOS filtros já aplicados em fuzzy
#         sem_terms = []
#         for _s, pairs in sem.items():
#             for term, _sc in pairs:
#                 t = norm(term)
#                 if (t 
#                     and t not in sem_terms
#                     and t not in base_seeds
#                     and term_is_valid_for_seeds(t, base_seeds)   # exige interseção literal/raiz com as seeds
#                     and not _starts_with_negator(t)):
#                     sem_terms.append(t)

#         # 2) typos reais do texto via fuzzy (somente gerados)
#         fuzzy_terms = set()
#         for seed in seeds:
#             hit = best_fuzzy_in_vocab(seed, local_vocab, min_ratio=0.86)
#             if hit:
#                 fuzzy_terms.add(norm(hit))

#         # 3) filtrar APENAS os fuzzy por qualidade/negador; sem_terms entram como vieram
#         filtered_fuzzy = [
#             t for t in fuzzy_terms
#             if t not in base_seeds
#             and term_is_valid_for_seeds(t, base_seeds)
#             and not _starts_with_negator(t)
#         ]

#         # 4) léxico final = seeds + semânticos + fuzzy filtrados
#         lexicon = unique(base_seeds + sem_terms + list(filtered_fuzzy))

#         # 5) localizar termos no TEXTO (normalizado) e aplicar negação por sentença
#         neg_list_norm = [norm(x) for x in neg_tokens]
#         for term in lexicon:
#             pat = r"(?i)(?<!\w)" + _accent_rx(term) + r"(?!\w)"
#             rx = re.compile(pat)

#             for m in rx.finditer(text_norm):
#                 abs_start = int(span.start_char + m.start())
#                 abs_end   = int(span.start_char + m.end())

#                 # VALIDAÇÃO DE CONTEXTO: verificar se o finding está próximo de menção ao órgão
#                 if organ_cfg is not None and organ_lexicon is not None:
#                     organ_mentions = _find_organ_mentions_in_text(span.text, organ_name, organ_cfg, organ_lexicon)
#                     finding_pos_in_block = abs_start - span.start_char
#                     is_near = _finding_is_near_organ_mention(finding_pos_in_block, organ_mentions, max_distance=200)
                    
#                     if not is_near:
#                         continue
                    
#                     # NOVA VALIDAÇÃO GENERALISTA: Verificar se menciona APENAS outros órgãos
#                     snippet = _extract_snippet_around_finding(span.doc.text, abs_start, abs_end, window=100)
#                     if _snippet_mentions_other_organs(snippet, organ_name, CONFIG["organs"]):
#                         # Menciona apenas outros órgãos, não o target → REJEITAR
#                         continue

#                 # negação: mesma sentença, à esquerda, janela limitada
#                 neg = is_negated_in_sentence(
#                     span.doc,
#                     start_char=abs_start,
#                     end_char=abs_end,
#                     neg_list=neg_list_norm,
#                     window_tokens=neg_window
#                 )

#                 hits.append(FindingHit(
#                     organ=block["organ"],
#                     finding=f_can,
#                     surface=text_norm[m.start():m.end()],
#                     start_char=abs_start,
#                     end_char=abs_end,
#                     negated=neg,
#                     confidence=0.9 if not neg else 0.35
#                 ))
#     return hits



# def _force_sentence_breaks(text: str) -> str:
#     """
#     Força quebras de sentença em pontos finais seguidos de maiúscula.
#     """
#     import re
#     # Adiciona quebra de linha após ponto + espaço + maiúscula
#     text = re.sub(r'\.(\s+)([A-Z])', r'.\n\2', text)
#     return text


# # --- Pipeline principal ---
# def process_report(text: str, target_organ=None) -> Dict[str, Any]:
#     """
#     target_organ: None | 'figado' | ['figado','rim',...]
#     Executa a análise apenas para o(s) órgão(s) selecionado(s).
#     """
#     text = to_plain(text)  # já inclui cleanup + remove_final_laudo
#     text = _force_sentence_breaks(text)
#     doc = nlp(text)

#     # 1) resolve órgãos alvo
#     target_organs = _resolve_target_organs(
#         target_organ if target_organ is not None else TARGET_ORGAN,
#         CONFIG["organs"]
#     )

#     # 2) vocabulário global do documento (expansão semântica dos órgãos alvo)
#     doc_vocab = build_doc_candidates(doc, min_len=CONFIG["semantic"]["min_token_len"])

#     organ_lexicons = {}
#     for organ in target_organs:
#         ocfg = CONFIG["organs"][organ]
#         sem = semantic_expand(
#             ocfg.get("seeds", []),
#             doc_vocab,
#             CONFIG["semantic"]["min_sim_seed_vocab"],
#             CONFIG["semantic"]["top_k_per_seed"]
#         )
#         organ_lexicons[organ] = merge_semantic_sets(ocfg.get("seeds", []), sem)

#     # 3) segmentação por cabeçalho (preferencial). Se não achar, cai na sentença-âncora.
#     #organs_subset_cfg = {k: CONFIG["organs"][k] for k in target_organs}

#     #header_blocks = segment_by_headers(doc, target_organs)
#     #if header_blocks:
#     #    blocks = header_blocks
#     #else:
#         # fallback: sentença-âncora (sua função já existente)
#     #    blocks = segment_by_organs(doc, organs_subset_cfg, organ_lexicons)

#     # 3) segmentação
#     organs_subset_cfg = {k: CONFIG["organs"][k] for k in target_organs}

#     # NORMALIZAÇÃO do FORCE_FULL_DOC_FOR (aceita string ou set/list)
#     _force_set = set()
#     if isinstance(FORCE_FULL_DOC_FOR, str) and FORCE_FULL_DOC_FOR.strip():
#         _force_set = {norm(FORCE_FULL_DOC_FOR)}
#     else:
#         try:
#             _force_set = {norm(x) for x in (FORCE_FULL_DOC_FOR or set())}
#         except TypeError:
#             _force_set = set()

#     # Força somente se o órgão-alvo estiver no FORCE_FULL_DOC_FOR
#     force_full = any(norm(o) in _force_set for o in target_organs)

#     if force_full:
#         main_organ = next((o for o in target_organs if norm(o) in _force_set), target_organs[0])
#         blocks = [{"organ": main_organ, "span": doc[:]}]
#     else:
#         header_blocks = segment_by_headers(doc, target_organs)
#         if header_blocks:
#             blocks = header_blocks
#         else:
#             blocks = segment_by_organs(doc, organs_subset_cfg, organ_lexicons)

#     # 4) matching de achados apenas para os órgãos alvo
#     all_hits: List[FindingHit] = []
#     for b in blocks:
#         organ = b["organ"]
#         finding_cfg = CONFIG["findings"].get(organ, {})
#         if not finding_cfg:
#             continue
#         hits = match_findings_in_block(
#             b, finding_cfg,
#             neg_tokens=[norm(x) for x in CONFIG["negation"]["tokens"]],
#             neg_window=CONFIG["negation"]["window_tokens"],
#             min_sim_seed_vocab=CONFIG["semantic"]["min_sim_seed_vocab"],
#             organ_cfg=CONFIG["organs"].get(organ, {}),
#             organ_lexicon=organ_lexicons.get(organ, [])
#         )
#         all_hits.extend(hits)

#     # CORREÇÃO: Mover para FORA do loop for b in blocks
#     findings_dicts = [asdict(h) for h in all_hits]
#     result = {
#         "organs_requested": target_organs,
#         "organs_detected": unique([blk["organ"] for blk in blocks]),
#         "findings": findings_dicts,
#         "relevant": any(not h.negated for h in all_hits),
#     }

#     # resumo compacto
#     sc = compact_summary_from_hits(findings_dicts, doc, blocks)

#     # --- filtro: remove chaves com valores nulos/vazios e descarta dicts que ficarem vazios ---
#     def _is_useful(v):
#         if v is None:
#             return False
#         if isinstance(v, str):
#             x = v.strip().lower()
#             return x not in {"", "null", "none"}
#         return True

#     sc_clean = []
#     for d in sc:
#         clean = {k: v for k, v in d.items() if _is_useful(v)}
#         if clean:
#             sc_clean.append(clean)

#     result["summary_compact"] = sc_clean
#     return result

# def process_reports_batch(texts: list[str], target_organ=None, desc: str = "Processando laudos"):
#     try:
#         from tqdm import tqdm
#         it = tqdm(texts, desc=desc)
#     except Exception:
#         it = texts
#     return [process_report(txt, target_organ=target_organ) for txt in it]


# def print_result(res, mode="full"):
#     if mode == "summary":
#         import json
#         print(json.dumps(res.get("summary_compact", []), ensure_ascii=False, indent=2))
#         return

#     print(f"organs_requested: {res.get('organs_requested')}")
#     print(f"organs_detected:  {res.get('organs_detected')}")
#     print(f"relevant:         {res.get('relevant')}\n")
#     print("findings:")
#     for h in res.get("findings", []):
#         print(f"- organ: {h['organ']} | finding: {h['finding']} | surface: \"{h['surface']}\" "
#               f"| start: {h['start_char']:>3} | end: {h['end_char']:>3} | negated: {h['negated']} | conf: {h['confidence']}")
        
# def extrair_nao_nulos(resumo):
#     """
#     resumo: pode ser list[dict], dict, None ou lixo.
#     Retorna list[dict] só com {chave: texto} onde texto é não-nulo.
#     """
#     out = []

#     def _valido(v):
#         if v is None:
#             return False
#         if isinstance(v, float) and pd.isna(v):
#             return False
#         s = str(v).strip()
#         if s == "" or s.lower() == "null":
#             return False
#         return True

#     if isinstance(resumo, list):
#         for item in resumo:
#             if isinstance(item, dict):
#                 for k, v in item.items():
#                     if _valido(v):
#                         out.append({k: v})
#     elif isinstance(resumo, dict):
#         for k, v in resumo.items():
#             if _valido(v):
#                 out.append({k: v})
#     # qualquer outra coisa -> lista vazia
#     return out

####Dados de Entrada

In [0]:
# df_laudos_original_completo = spark.sql(f"""
#                                 select *
#                                 from {SCHEMA}.{INPUT_TABLENAME}
#                                 where cast(dataExecucaoModelo as date) = date('{data_execucao_modelo}')
#                                 -- and (
#                                 --     lower(Laudo) ILIKE '%adenocarcinoma%' OR
#                                 --     lower(Laudo) ILIKE '%carcinoma%' OR
#                                 --     lower(Laudo) ILIKE '%polipo%' OR
#                                 --     lower(Laudo) ILIKE '%colon%' OR
#                                 --     lower(Laudo) ILIKE '%polipo%' OR
#                                 --     lower(Laudo) ILIKE '%ulceracao%' OR
#                                 --     lower(Laudo) ILIKE '%espessamento%' OR
#                                 --     lower(Laudo) ILIKE '%sangramento%'
#                                 -- )
#                                 """).toPandas()
# df_laudos_original_completo.shape

In [0]:
df_laudos_original_completo = spark.sql(f"""
                                select *
                                from {SCHEMA}.{INPUT_TABLENAME}
                                where cast(dataExecucaoModelo as date) BETWEEN '2026-03-18' AND '2026-03-25'
                                --where cast(dataExecucaoModelo as date) = date('{data_execucao_modelo}')
                                --and cast(dataexame as date) >= date_sub(current_date(), 10)
                                """).toPandas()
df_laudos_original_completo.shape

# df_laudos_original_completo = spark.sql(f"""
#                                 select *
#                                 from {SCHEMA}.{INPUT_TABLENAME}
#                                 where cast(dataExecucaoModelo as date) = date('{data_execucao_modelo}')
#                                 -- and cast(dataexame as date) >= '2025-12-01'
#                                 -- and cast(dataexame as date) >= date_sub(current_date(), 30)
#                                 """).toPandas()
# df_laudos_original_completo.shape

# df_laudos_original_completo = spark.sql(f"""
#     SELECT *
#     FROM {SCHEMA}.{INPUT_TABLENAME}
#     WHERE CAST(dataExecucaoModelo AS DATE) = DATE('{data_execucao_modelo}')
#       AND CAST(dataExame AS DATE) BETWEEN
#             date_sub(DATE('{data_execucao_modelo}'), 7)
#         AND date_sub(DATE('{data_execucao_modelo}'), 1)
# """).toPandas()

# df_laudos_original_completo.shape

In [0]:
# df_laudos_original_completo = df_laudos_original_completo.head(4000)
# df_laudos_original_completo = df_laudos_original_completo.sample(n=500, random_state=42)

In [0]:
# laudo = """
# INFORMAÇÃO CLÍNICA: Retocolite ulcerativa. Diarreia e dor em FIE há 04 dias. TÉCNICA: Aquisição volumétrica sem a injeção do meio de contraste iodado. ACHADOS: Fígado: volume, contorno e atenuação normais. Vias biliares: calibre normal. Vesícula biliar: sem particularidades ao método. Baço: volume, contorno e atenuação normais. Pâncreas: espessura e atenuação normais. Glândulas adrenais: espessura normal. Rins: tópicos, com volume e atenuação normais. Microcálculo no grupamento calicinal inferior direito, mede 0,2 cm. Não háhidronefrose. Ureteres: sem cálculos ou estenose. Bexiga: vazia, de análise limitada. Trato gastrointestinal: distensão líquida em grau leve de alças colorretais, inespecífica. Ausência de diverticulose. Apêndice cecal: visualizado apenas o coto apendicular, de aspecto anatômico. Demaisórgãos pélvicos: dispositivo intra-uterino em topografia habitual. Formações císticas em regiões anexiais, de provável origem ovariana. Artérias e veias: trajetos e calibres normais. Linfonodos: morfologicamente normais. Mesentério e peritônio: ausência de espessamento, pneumoperitônio, coleções ou ascite. Conclusão: . IMPRESSÃO: Distensão líquida em grau leve de alças colorretais, inespecífica, Sinais de doença inflamatória intestinal (doença de Crohn) em atividade, no atual contexto clínico. Demais achados pormenorizados acima. Este Laudo pode não estar completo aqui em função do padrão de arquivo "RTF" utilizado. Recomendamos a sua visualização no sistema da Radiologia "WebRIS".
# """

# df_avalia = pd.DataFrame({
#     "id": [1],
#     "laudo": [laudo.strip()]
# })

# display(df_avalia)

In [0]:
df_laudos_original_completo['data_limpa'] = df_laudos_original_completo['dataexame'].astype(str).str.replace('-', '') ##<<lake
df_laudos_original_completo['record_id'] = df_laudos_original_completo['data_limpa'] + df_laudos_original_completo['id_pct'] ##<< lake

df_laudos_original_completo = df_laudos_original_completo.dropna(subset=['data_limpa'])

if df_laudos_original_completo.shape[0] == 0:
    dbutils.notebook.exit("OK")
else:
    display(df_laudos_original_completo)

In [0]:
df_laudos = df_laudos_original_completo
df_laudos.shape

####Executa o Modelo

In [0]:
df_laudos["laudo_tratado"] = df_laudos["Laudo"].map(to_plain)

In [0]:
laudos = df_laudos["laudo_tratado"].tolist()
resultados = df_laudos["laudo_tratado"].map(lambda s: (process_report(s, target_organ='doencas_biliares') or {}).get("summary_compact", []))

In [0]:
df_laudos["resultado"] = resultados
df_laudos["achados"] = df_laudos["resultado"].apply(lambda x: extract_keys_unique(x) if isinstance(x, list) else "")

In [0]:
df_laudos['fl_relevante'] = df_laudos['achados'].apply(lambda x: 1 if x != [] else 0)

In [0]:
# display(df_laudos[['Laudo', 'resultado', 'achados', 'fl_relevante']][df_laudos['fl_relevante'] == 1])
df_fl_relev = df_laudos[df_laudos['fl_relevante'] == 1]

if df_fl_relev.shape[0] > 0:
    display(df_fl_relev)

In [0]:
if df_laudos.shape[0] > 0:
    display(df_laudos)

####Tabela Saida

In [0]:
colunas = [
    'record_id',
    'id_pct',
    'idunidade',
    'unidade',
    'paciente',
    'cpf',
    'sexo',
    'nascimento',
    'idade_anos',
    'telefonePacienteDDD',
    'telefonePaciente',
    'dataexame',
    'modalidade',
    'exame',
    'tipoexame',
    'num_pedido_integracao',
    'nme_regional_hospital',
    'nme_convenio',
    'an',
    'idstatus',
    'DataLaudoLiberado',
    'Laudo',
    'laudo_tratado',
    'resultado',
    'achados',
    'fl_relevante',
    'dataExecucaoModelo'
]

# reorganizar dataframe
df_laudos = df_laudos.reindex(columns=colunas)

In [0]:
df_laudos = df_laudos.rename(columns={'paciente':'pct_nome',
                        'cpf':'pct_cpf',
                        'sexo':'pct_sexo',
                        'nascimento':'pct_datanasc',
                        'idade_anos':'idade_paciente',
                        'dataexame':'exm_data',
                        'modalidade':'exm_mod',
                        'exame':'exm_titulo',
                        'tipoexame':'exm_tipo',
                        'an':'exm_an',
                        'idstatus':'exm_status',
                        'DataLaudoLiberado':'exm_laudo_dataliber',
                        'Laudo':'exm_laudo_texto',
                        'laudo_tratado':'exm_laudo_texto_tratado',
                        'resultado':'exm_laudo_resultado',
                        'achados':'exm_laudo_achados'
                        })

In [0]:
# spark.sql(f"""
# ALTER TABLE {SCHEMA}.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.{OUTPUT_TABLENAME} (
  record_id STRING,
  id_pct STRING,
  idunidade STRING,
  unidade STRING,
  pct_nome STRING,
  pct_cpf STRING,
  pct_sexo STRING,
  pct_datanasc DATE,
  idade_paciente INT,
  telefonePacienteDDD STRING,
  telefonePaciente STRING,
  exm_data DATE,
  exm_mod STRING,
  exm_titulo STRING,
  exm_tipo STRING,
  num_pedido_integracao STRING,
  nme_regional_hospital STRING,
  nme_convenio STRING,
  exm_an STRING,
  exm_status STRING,
  exm_laudo_dataliber DATE,
  exm_laudo_texto STRING,
  exm_laudo_texto_tratado STRING,
  exm_laudo_resultado STRING,
  exm_laudo_achados STRING,
  fl_relevante INT,
  dataExecucaoModelo DATE
)
USING delta
COMMENT 'Laudos processados e enriquecidos com NLP para análise de doencas biliares';
""")

In [0]:
df_laudos["exm_data"] = pd.to_datetime(df_laudos["exm_data"], errors="coerce")
df_laudos["pct_datanasc"] = pd.to_datetime(df_laudos["pct_datanasc"], errors="coerce")
df_laudos["exm_laudo_dataliber"] = pd.to_datetime(df_laudos["exm_laudo_dataliber"], errors="coerce")
df_laudos["dataExecucaoModelo"] = pd.to_datetime(df_laudos["dataExecucaoModelo"], errors="coerce")
df_laudos["exm_status"] = (
    df_laudos["exm_status"]
    .astype("Int64")   # permite NA
    .astype(str)       # converte para string
    .replace("<NA>", None)
)

In [0]:
date_cols = [
    "exm_data",
    "pct_datanasc",
    "exm_laudo_dataliber",
    "dataExecucaoModelo"
]

for c in date_cols:
    df_laudos[c] = pd.to_datetime(df_laudos[c], errors="coerce").dt.date

In [0]:
int_cols = [
    "idade_paciente",
    "fl_relevante"
]

for c in int_cols:
    df_laudos[c] = (
        df_laudos[c]
        .astype("Int64")   # aceita NA
        .where(pd.notna(df_laudos[c]), None)
    )

In [0]:
import pandas as pd
import numpy as np

def safe_str(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, (list, tuple, set, dict, np.ndarray)):
        return str(x)
    return str(x)

In [0]:
import pandas as pd
import json
import math

def to_str(x):
    # trata NaN/NaT/None
    if x is None:
        return None
    # pandas escalar "missing"
    if isinstance(x, (float, int)) and isinstance(x, float) and math.isnan(x):
        return None
    if isinstance(x, pd._libs.missing.NAType):
        return None

    # se for lista ou dict (ex.: seus arrays de achados/resultado), serializa em JSON
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)

    # fallback: string normal
    return str(x)

string_cols = [
    "record_id",
    "id_pct",
    "idunidade",
    "unidade",
    "pct_nome",
    "pct_cpf",
    "pct_sexo",
    "telefonePacienteDDD",
    "telefonePaciente",
    "exm_mod",
    "exm_titulo",
    "exm_tipo",
    "num_pedido_integracao",
    "nme_regional_hospital",
    "nme_convenio",
    "exm_an",
    "exm_status",
    "exm_laudo_texto",
    "exm_laudo_texto_tratado",
    "exm_laudo_resultado",
    "exm_laudo_achados",
]

for c in string_cols:
    df_laudos[c] = df_laudos[c].apply(to_str)


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DateType, TimestampType
)

schema = StructType([
    StructField("record_id", StringType(), True),
    StructField("id_pct", StringType(), True),
    StructField("idunidade", StringType(), True),
    StructField("unidade", StringType(), True),
    StructField("pct_nome", StringType(), True),
    StructField("pct_cpf", StringType(), True),
    StructField("pct_sexo", StringType(), True),
    StructField("pct_datanasc", DateType(), True),
    StructField("idade_paciente", IntegerType(), True),
    StructField("telefonePacienteDDD", StringType(), True),
    StructField("telefonePaciente", StringType(), True),
    StructField("exm_data", DateType(), True),
    StructField("exm_mod", StringType(), True),
    StructField("exm_titulo", StringType(), True),
    StructField("exm_tipo", StringType(), True),
    StructField("num_pedido_integracao", StringType(), True),
    StructField("nme_regional_hospital", StringType(), True),
    StructField("nme_convenio", StringType(), True),
    StructField("exm_an", StringType(), True),
    StructField("exm_status", StringType(), True),
    StructField("exm_laudo_dataliber", DateType(), True),
    StructField("exm_laudo_texto", StringType(), True),
    StructField("exm_laudo_texto_tratado", StringType(), True),
    StructField("exm_laudo_resultado", StringType(), True),
    StructField("exm_laudo_achados", StringType(), True),
    StructField("fl_relevante", IntegerType(), True),
    StructField("dataExecucaoModelo", DateType(), True),
])

In [0]:
df_laudos_spark = spark.createDataFrame(df_laudos, schema=schema)

In [0]:
df_laudos_spark.createOrReplaceTempView("vw_doencas_biliares_tb")

In [0]:
spark.sql(f"""
INSERT INTO {SCHEMA}.{OUTPUT_TABLENAME} (
    record_id,
    id_pct,
    idunidade,
    unidade,
    pct_nome,
    pct_cpf,
    pct_sexo,
    pct_datanasc,
    idade_paciente,
    telefonePacienteDDD,
    telefonePaciente,
    exm_data,
    exm_mod,
    exm_titulo,
    exm_tipo,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    exm_an,
    exm_status,
    exm_laudo_dataliber,
    exm_laudo_texto,
    exm_laudo_texto_tratado,
    exm_laudo_resultado,
    exm_laudo_achados,
    fl_relevante,
    dataExecucaoModelo
)
SELECT
    s.record_id,
    s.id_pct,
    s.idunidade,
    s.unidade,
    s.pct_nome,
    s.pct_cpf,
    s.pct_sexo,
    s.pct_datanasc,            
    s.idade_paciente,   
    s.telefonePacienteDDD,
    s.telefonePaciente,
    s.exm_data,                 
    s.exm_mod,
    s.exm_titulo,
    s.exm_tipo,
    s.num_pedido_integracao,
    s.nme_regional_hospital,
    s.nme_convenio,
    s.exm_an,
    s.exm_status,
    s.exm_laudo_dataliber,
    s.exm_laudo_texto,
    s.exm_laudo_texto_tratado,
    s.exm_laudo_resultado,
    s.exm_laudo_achados,
    s.fl_relevante,
    s.dataExecucaoModelo

FROM vw_doencas_biliares_tb s
LEFT ANTI JOIN {SCHEMA}.{OUTPUT_TABLENAME} t
    ON t.exm_an    = s.exm_an
""")

####Analises

In [0]:
spark.sql(f"""
            select *
            from {SCHEMA}.{OUTPUT_TABLENAME}
            where fl_relevante = 1
        """).display()

In [0]:
spark.sql(f"""
            SELECT
                dataExecucaoModelo,
                COUNT(*) AS total_registros,
                SUM(CASE WHEN fl_relevante = 1 THEN 1 ELSE 0 END) AS total_relevantes
            FROM {SCHEMA}.{OUTPUT_TABLENAME}
            GROUP BY dataExecucaoModelo
            ORDER BY dataExecucaoModelo
        """).display()

In [0]:
dbutils.notebook.exit("Final Notebook")

####Analises com Sabia 3

In [0]:
# !pip install maritalk

In [0]:
# import maritalk
# import multiprocessing as mp
# from datetime import date, datetime, timedelta
# import time

In [0]:
# model = maritalk.MariTalk(
#     key='APIKEY',
#     model="sabia-3"
# )

In [0]:
# def separar_por_numero_de_palavras(df, coluna_palavras):
#     dfs = {}
    
#     # dfs['df_01'] = df[df[coluna_palavras] <= 20]
#     # dfs['df_02'] = df[(df[coluna_palavras] > 20) & (df[coluna_palavras] <= 50)]
#     # dfs['df_03'] = df[(df[coluna_palavras] > 50) & (df[coluna_palavras] <= 100)]
#     # dfs['df_04'] = df[df[coluna_palavras] > 100]
#     dfs['df_1'] = df[df[coluna_palavras] <= 70]
#     dfs['df_2'] = df[(df[coluna_palavras] > 70) & (df[coluna_palavras] <= 140)]
#     dfs['df_3'] = df[(df[coluna_palavras] > 140) & (df[coluna_palavras] <= 200)]
#     dfs['df_4'] = df[df[coluna_palavras] > 200]

#     dfs['df_01'] = df[df[coluna_palavras] <= 70]
#     dfs['df_02'] = df[(df[coluna_palavras] > 70) & (df[coluna_palavras] <= 140)]
#     dfs['df_03'] = df[(df[coluna_palavras] > 140) & (df[coluna_palavras] <= 200)]
#     dfs['df_04'] = df[df[coluna_palavras] > 200]

#     dfs['df_001'] = df[df[coluna_palavras] <= 70]
#     dfs['df_002'] = df[(df[coluna_palavras] > 70) & (df[coluna_palavras] <= 140)]
#     dfs['df_003'] = df[(df[coluna_palavras] > 140) & (df[coluna_palavras] <= 200)]
#     dfs['df_004'] = df[df[coluna_palavras] > 200]

#     dfs['df_0001'] = df[df[coluna_palavras] <= 70]
#     dfs['df_0002'] = df[(df[coluna_palavras] > 70) & (df[coluna_palavras] <= 140)]
#     dfs['df_0003'] = df[(df[coluna_palavras] > 140) & (df[coluna_palavras] <= 200)]
#     dfs['df_0004'] = df[df[coluna_palavras] > 200]
    
#     return dfs

# def contar_palavras(texto):
#     palavras = texto.split()
#     return len(palavras)

# def extrair_respostas(dado):
#     try:
#         # Converter a string em um dicionário
#         dado_dict = ast.literal_eval(dado) if isinstance(dado, str) else dado
#         # Verificar se é um dicionário e contém a chave 'answer'
#         if isinstance(dado_dict, dict) and 'answer' in dado_dict:
#             return dado_dict['answer']
#         else:
#             return None
#     except (ValueError, SyntaxError):
#         # Retornar None se a conversão falhar
#         return None

In [0]:
# dfdf = df.copy()

# dfdf = dfdf.head(10).copy()

# df.shape

In [0]:
# df['Num_Palavras'] = df['laudo_tratado'].apply(contar_palavras)
# dfs_separados_class = separar_por_numero_de_palavras(df, 'Num_Palavras')

# df_1 = dfs_separados_class['df_1']
# df_2 = dfs_separados_class['df_2']
# df_3 = dfs_separados_class['df_3']
# df_4 = dfs_separados_class['df_4']

In [0]:
# def processar_comentario(index, row, pergunta01):
#     # input_text = f"{pergunta01[0]} {row['Comentario_original']}" if 'vazio' in row['Comentario_original_tratado'].lower() else f"{pergunta01[0]} {row['Comentario_original_tratado']}"
#     input_text = f"{pergunta01[0]} {row['laudo_tratado']}"
    
#     sucesso = False
#     tentativas = 0
#     resposta = None
#     while not sucesso and tentativas < 5:
#         try:
#             response = model.generate(input_text,
#                                       max_tokens=1500,
#                                       num_tokens_per_message=550,
#                                       temperature=0.0, #0.0
#                                       top_p=0.99, #0.99
#                                       do_sample=False,
#                                       stream=False,
#                                       return_async_generator=False
#                                       )
#             resposta = response
#             sucesso = True
#         except Exception as e:
#             print(f"Erro na tentativa {tentativas+1}: {e}")
#             tentativas += 1
#             time.sleep(2 ** tentativas)
    
#     return index, resposta

# def processar_df_correcao_textual(df, pergunta01, tamanho_lote=8):
#     df['resposta_sabia'] = ''
    
#     with mp.Pool(mp.cpu_count()) as pool:
#         results = []
        
#         for index, row in df.iterrows():
#             results.append(pool.apply_async(processar_comentario, args=(index, row, pergunta01)))
        
#         for result in results:
#             index, resposta = result.get()
#             df.at[index, 'resposta_sabia'] = resposta

#         contador = len(df)
#         print(f"Processado: {contador} de {len(df)}")
    
#     return df

# pergunta01 = [
#     """
#         Você é um médico especialista em gastroenterologia.

#         Sua tarefa é analisar exclusivamente o texto do laudo médico abaixo e classificar se há diagnóstico explícito de Doença Inflamatória Intestinal (DII).

#         Responda apenas com uma letra, seguindo rigorosamente as regras abaixo:

#         S → somente se o laudo indicar explicitamente:

#         Doença de Crohn diagnosticada OU

#         Retocolite Ulcerativa diagnosticada

#         N → se o laudo não indicar diagnóstico dessas doenças, incluindo casos de:

#         achados inespecíficos

#         inflamação sem definição

#         suspeita não conclusiva

#         hipótese diagnóstica

#         necessidade de correlação clínica

#         Critérios obrigatórios

#         Considere apenas o conteúdo do laudo.

#         Não presuma diagnóstico.

#         Não utilize conhecimento externo.

#         Achados como espessamento, retite inespecífica, processo inflamatório, inflamação ativa, alterações inflamatórias NÃO são suficientes, isoladamente, para classificar como DII.

#         Expressões que AUTORIZAM resposta S

#         “doença de crohn”

#         “crohn”

#         “retocolite ulcerativa”

#         “colite ulcerativa”

#         “doença inflamatória intestinal”

#         “dii”

#         “compatível com doença de crohn”

#         “aspecto compatível com retocolite ulcerativa”

#         (Considere variações ortográficas, plural, acentuação ou ausência de acento.)

#         Regras finais da resposta

#         Responda somente com S ou N

#         Não explique

#         Não acrescente texto

#         Não inclua pontuação

#         Não acrescente nada
#     """
# ]

# dfs = [df_1, df_2, df_3, df_4]

# for df in dfs:
#     df = processar_df_correcao_textual(df, pergunta01)

In [0]:
# for df in dfs:
#     df['resposta_sabia'] = df['resposta_sabia'].apply(extrair_respostas)
#     df['resposta_sabia'] = df['resposta_sabia'].str.replace('[', '').str.replace(']', '')

# # df_final = pd.concat([df_1, df_2, df_3, df_4], ignore_index=True)
# df = pd.concat([df_1, df_2, df_3, df_4], ignore_index=True)

In [0]:
# pd.set_option("display.max_rows", None)       # mostra todas as linhas
# pd.set_option("display.max_columns", None)    # mostra todas as colunas
# pd.set_option("display.max_colwidth", None)  

In [0]:
# df_filtrado = df[
#     (df["resposta_sabia"] == "N") &
#     (df["achados"].apply(lambda x: isinstance(x, list) and len(x) > 0))
# ]

# df_filtrado.shape

In [0]:
# df_filtrado_positivo = df[
#     (df["resposta_sabia"] == "S") &
#     (df["achados"].apply(lambda x: isinstance(x, list) and len(x) > 0))
# ]

# df_filtrado_positivo.shape

In [0]:
# df_filtrado_nao_class = df[
#     df["resposta_sabia"].isin(["S", "N"]) &
#     df["achados"].apply(lambda x: isinstance(x, list) and len(x) == 0)
# ]

# df_filtrado_nao_class.shape

In [0]:
# df["resposta_sabia"] = df["resposta_sabia"].astype(str).str.strip()

In [0]:
# def is_lista_nao_vazia(x):
#     return isinstance(x, list) and len(x) > 0

# def is_lista_vazia(x):
#     return isinstance(x, list) and len(x) == 0

In [0]:
# mask_N_com_achado = (df["resposta_sabia"] == "N") & df["achados"].apply(is_lista_nao_vazia)
# mask_S_com_achado = (df["resposta_sabia"] == "S") & df["achados"].apply(is_lista_nao_vazia)

# mask_SN_sem_achado = df["resposta_sabia"].isin(["S", "N"]) & df["achados"].apply(is_lista_vazia)

In [0]:
# df_N_com_achado = df[mask_N_com_achado]
# df_S_com_achado = df[mask_S_com_achado]
# df_SN_sem_achado = df[mask_SN_sem_achado]

In [0]:
# total = len(df)
# n_N_com_achado = len(df_N_com_achado)
# n_S_com_achado = len(df_S_com_achado)
# n_SN_sem_achado = len(df_SN_sem_achado)

# print("total:", total)
# print("N com achado:", n_N_com_achado)
# print("S com achado:", n_S_com_achado)
# print("S ou N sem achado:", n_SN_sem_achado)
# print("sobra:", total - (n_N_com_achado + n_S_com_achado + n_SN_sem_achado))


In [0]:
# mask_restante = ~(mask_N_com_achado | mask_S_com_achado | mask_SN_sem_achado)
# df_restantes = df[mask_restante]
# display(df_restantes)

In [0]:
# teste = df_referencia[df_referencia['laudo'].str.startswith('<div')]
# teste.shape

In [0]:
# teste_id = teste[teste['id_exame'] == 'OBSTASYHBREXAIMA00004200010606280048052096010202072000007213319899']
# # display(teste)
# display(teste_id)

In [0]:
# display(df.iloc[50:70])

In [0]:
# df_new = spark.createDataFrame(df)
# print(type(df_new))

In [0]:
# for c in df_new.columns:
#     print(repr(c))

# df_new.printSchema()

In [0]:
# from pyspark.sql import functions as F

# df_fix = df_new.withColumn(
#     "resultado",
#     F.expr("""
#         transform(
#             resultado,
#             x -> named_struct(
#                 'doenca_de_crohn', x.`doenca de crohn`,
#                 'retocolite_ulcerativa', x.`retocolite ulcerativa`
#             )
#         )
#     """)
# )

In [0]:
# table_name_final = "dev_tbl_dii_analise"
# df_new.write.format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(f"diamond_dii.dii.{table_name_final}")

In [0]:
# teste = df[df['id_exame'] == 'OBSTASYVNSEXAIMA00007400018125080049893632010703370200001221646607']
# display(teste)

In [0]:
# display(df)

In [0]:
# display(df2[['descricao_laudo', 'laudo', 'laudo_tratado', 'resultado', 'achados']])